# LangGraph - Complete Core Capabilities Guide

**A Comprehensive Learning and Preparation Guide**

This guide includes practical examples, explanations, and diagrams to aid understanding. Additional insights can be found in the [LangGraph How-Tos](https://langchain-ai.github.io/langgraph/how-tos).

**A Comprehensive Learning and Preparation Guide**

---

## 📚 Table of Contents

1. [Introduction & Setup](#intro)
2. [Fundamentals - Building Blocks](#fundamentals)
3. [Core Capabilities](#core-capabilities)
   - Streaming
   - Persistence
   - Durable Execution
   - Memory
   - Context
   - Models
   - Tools
   - Human-in-the-Loop
   - Time Travel
   - Subgraphs
   - Multi-Agent
   - MCP Integration
   - Evaluation
4. [Complete Working Examples](#examples)
5. [Best Practices & Tips](#best-practices)

---

## What is LangGraph?

**LangGraph** is a powerful framework built on top of LangChain for creating **stateful, multi-actor applications** with Large Language Models (LLMs). It extends LangChain's capabilities by providing:

- **Stateful workflows**: Maintain and manage state across interactions
- **Cyclic graphs**: Create loops and conditional logic in your workflows
- **Human-in-the-loop**: Integrate human oversight and approval
- **Persistence**: Save and resume execution at any point
- **Multi-agent systems**: Coordinate multiple AI agents

### When to Use LangGraph?

✅ **Use LangGraph when you need:**
- Complex, multi-step AI workflows with conditional logic
- Stateful conversations that remember context
- Human oversight and approval in AI processes
- Multi-agent systems with coordination
- Long-running processes that need to persist
- Cyclic or recursive reasoning patterns

❌ **Don't use LangGraph for:**
- Simple, single LLM calls (use LangChain directly)
- Stateless API endpoints
- Basic prompt templates without state


## 🛠️ Installation & Setup

### Required Packages


In [ ]:
# Install LangGraph and dependencies
# !pip install -U langgraph langchain langchain-openai langchain-anthropic langchain-community
# !pip install langsmith  # For evaluation and tracing

# For specific checkpointers
# !pip install langgraph-checkpoint-postgres  # PostgreSQL persistence
# !pip install langgraph-checkpoint-sqlite    # SQLite persistence


In [ ]:
# Import necessary libraries
import os
from typing import Annotated, TypedDict, Literal
from typing_extensions import TypedDict
import operator

# LangGraph imports
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# Optional: Set API keys (replace with your actual keys or use environment variables)
# os.environ["OPENAI_API_KEY"] = "your-key-here"
# os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
# os.environ["LANGSMITH_API_KEY"] = "your-key-here"

print("✅ All imports successful!")


# Part 2: Fundamentals - Building Blocks

<a id="fundamentals"></a>

Before diving into core capabilities, let's understand the fundamental building blocks of LangGraph.

## 🎯 Key Concepts

1. **StateGraph**: The main graph structure that manages state transitions
2. **State**: A TypedDict that defines the data schema flowing through the graph
3. **Nodes**: Functions that process and transform state
4. **Edges**: Connections between nodes (regular or conditional)
5. **Compilation**: Converting the graph definition into an executable runnable

---

## 1️⃣ State Definition

State is the data that flows through your graph. It's defined using a TypedDict.


In [ ]:
# Example 1: Simple State
class SimpleState(TypedDict):
    input: str
    output: str
    
# Example 2: State with Messages (for chatbots)
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]  # Automatically appends messages
    
# Example 3: Complex State with Multiple Fields
class ComplexState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    context: dict
    step_count: int
    final_answer: str

print("✅ State definitions created")
print("\nKey concept: Annotated with operator.add automatically appends/merges values")
print("             instead of replacing them in state updates")


## 2️⃣ Creating a StateGraph

A StateGraph is initialized with a state schema and contains nodes and edges.


In [ ]:
# Create a simple StateGraph
workflow = StateGraph(SimpleState)

print("✅ StateGraph created with SimpleState schema")
print("\nNext steps:")
print("1. Add nodes (processing functions)")
print("2. Add edges (connections between nodes)")
print("3. Compile into a runnable application")


## 3️⃣ Adding Nodes

Nodes are functions that take state as input and return state updates.


## Streaming

Streaming in LangGraph enables real-time updates and token streaming, providing a dynamic flow of information and interaction. It facilitates applications that require live data processing and user interactions.

### What is Streaming?

Streaming allows you to receive updates from your graph execution in real-time, rather than waiting for the entire process to complete. This is particularly useful for:

- **User-facing applications**: Show progress and intermediate results
- **Long-running processes**: Monitor execution progress
- **Interactive systems**: Provide immediate feedback
- **Debugging**: See step-by-step execution flow

### Streaming Modes

LangGraph supports different streaming modes:

1. **`stream_mode="values"`**: Get the complete state after each node execution
2. **`stream_mode="updates"`**: Get only the changes/updates from each node
3. **`stream_mode="messages"`**: Stream individual messages (for chat applications)

### Key Benefits

- **Real-time feedback**: Users see progress immediately
- **Better UX**: No waiting for complete execution
- **Debugging aid**: Step-by-step execution visibility
- **Resource efficiency**: Process data as it becomes available

**Diagram:**

```
Input → Node1 → Node2 → Node3 → Output
  ↓      ↓       ↓       ↓       ↓
Stream → Stream → Stream → Stream → Final
```

**Key Concept:** Streaming transforms batch processing into real-time, interactive workflows.



In [ ]:
# Define node functions
def input_node(state: SimpleState) -> SimpleState:
    """Process input"""
    user_input = state["input"]
    print(f"📥 Input Node: Received '{user_input}'")
    return {"input": user_input.upper()}  # Return state update

def processing_node(state: SimpleState) -> SimpleState:
    """Process the data"""
    processed = state["input"]
    print(f"⚙️ Processing Node: Processing '{processed}'")
    output = f"Processed: {processed}"
    return {"output": output}

# Create a new graph and add nodes
workflow = StateGraph(SimpleState)
workflow.add_node("input", input_node)
workflow.add_node("processing", processing_node)

print("✅ Nodes added to graph")
print("\nNode naming convention:")
print("- Use descriptive names (e.g., 'input', 'processing', 'llm_call')")
print("- Node functions must accept and return state dictionaries")


## 4️⃣ Adding Edges

Edges connect nodes and define the flow of execution.

### Types of Edges:
1. **Regular Edge**: Always go from node A to node B
2. **Conditional Edge**: Choose next node based on state or logic
3. **Entry Point**: Where execution starts (using START)
4. **Exit Point**: Where execution ends (using END)


In [ ]:
# Add edges to connect nodes
workflow.add_edge(START, "input")           # Start with input node
workflow.add_edge("input", "processing")    # Then go to processing
workflow.add_edge("processing", END)        # Finally end

print("✅ Edges added to graph")
print("\nFlow: START → input → processing → END")


## 5️⃣ Compilation & Execution

After defining nodes and edges, compile the graph into a runnable application.


In [ ]:
# Compile the graph
app = workflow.compile()

print("✅ Graph compiled successfully!")
print("\nThe compiled app is now a runnable that can be invoked")


## Persistence

Persistence ensures that the state of workflows is saved and can be resumed, allowing for continuity and state recovery.

### What is Persistence?

Persistence in LangGraph refers to the ability to save the current state of a workflow execution and resume it later. This is crucial for:

- **Long-running processes**: Save progress and resume after interruptions
- **Conversation systems**: Maintain chat history across sessions
- **Error recovery**: Resume from the last successful checkpoint
- **Multi-user systems**: Isolate user sessions and data

### Checkpointers

Checkpointers are the components responsible for saving and loading state:

1. **MemorySaver**: In-memory storage (for development/testing)
2. **PostgresSaver**: PostgreSQL database storage (production)
3. **SQLiteSaver**: SQLite file storage (local development)
4. **RedisSaver**: Redis storage (high-performance scenarios)

### Key Features

- **Automatic checkpointing**: State saved after each node execution
- **Thread isolation**: Separate state for different conversation threads
- **State recovery**: Resume from any saved checkpoint
- **Configurable storage**: Choose appropriate storage backend

### Use Cases

- **Chatbots**: Remember conversation history
- **Workflow automation**: Resume interrupted processes
- **Multi-step forms**: Save progress between steps
- **Background jobs**: Handle long-running tasks

**Diagram:**

```
Execution Flow:
Node1 → Checkpoint → Node2 → Checkpoint → Node3 → Final State
  ↓         ↓           ↓         ↓           ↓
Save     Resume      Save     Resume      Save
```

**Key Concept:** Persistence transforms stateless operations into stateful, resumable workflows.



In [ ]:
# Execute the graph
initial_state = {"input": "hello world", "output": ""}
result = app.invoke(initial_state)

print("\n" + "="*50)
print("EXECUTION RESULT:")
print("="*50)
print(f"Final State: {result}")


## 6️⃣ Conditional Edges

Conditional edges allow dynamic routing based on state or logic.


In [ ]:
# Define a state with routing logic
class RoutingState(TypedDict):
    message: str
    route: str
    result: str

# Define nodes for different paths
def classifier_node(state: RoutingState) -> RoutingState:
    """Classify the message and decide routing"""
    message = state["message"]
    # Simple classification logic
    if "urgent" in message.lower():
        route = "urgent_handler"
    elif "question" in message.lower():
        route = "qa_handler"
    else:
        route = "general_handler"
    
    print(f"🔍 Classifier: Routing to '{route}'")
    return {"route": route}

def urgent_handler(state: RoutingState) -> RoutingState:
    print("🚨 Urgent Handler: Processing urgent request")
    return {"result": "URGENT: Handled with priority"}

def qa_handler(state: RoutingState) -> RoutingState:
    print("❓ QA Handler: Processing question")
    return {"result": "ANSWER: Here's the response"}

def general_handler(state: RoutingState) -> RoutingState:
    print("📝 General Handler: Processing general message")
    return {"result": "GENERAL: Processed normally"}

# Define routing function
def route_message(state: RoutingState) -> str:
    """Return the name of the next node based on state"""
    return state["route"]

# Build graph with conditional routing
routing_workflow = StateGraph(RoutingState)

# Add nodes
routing_workflow.add_node("classifier", classifier_node)
routing_workflow.add_node("urgent_handler", urgent_handler)
routing_workflow.add_node("qa_handler", qa_handler)
routing_workflow.add_node("general_handler", general_handler)

# Add edges
routing_workflow.add_edge(START, "classifier")

# Conditional edge - routes based on the route_message function
routing_workflow.add_conditional_edges(
    "classifier",  # From this node
    route_message,  # Use this function to decide
    {
        "urgent_handler": "urgent_handler",
        "qa_handler": "qa_handler",
        "general_handler": "general_handler"
    }
)

# All handlers end the workflow
routing_workflow.add_edge("urgent_handler", END)
routing_workflow.add_edge("qa_handler", END)
routing_workflow.add_edge("general_handler", END)

# Compile
routing_app = routing_workflow.compile()

print("✅ Conditional routing graph created!")
print("\nThis graph routes messages to different handlers based on content")


In [ ]:
# Test the conditional routing
test_cases = [
    "This is an urgent request!",
    "I have a question about LangGraph",
    "Just a normal message"
]

for msg in test_cases:
    print(f"\n{'='*60}")
    print(f"Testing: '{msg}'")
    print('='*60)
    result = routing_app.invoke({"message": msg, "route": "", "result": ""})
    print(f"✅ Result: {result['result']}\n")


In [ ]:
# Define a multi-step state
class StreamState(TypedDict):
    step: int
    data: str
    result: str

def step1(state: StreamState) -> StreamState:
    print("  [Node: step1 executing...]")
    import time
    time.sleep(0.5)  # Simulate processing
    return {"step": 1, "data": "Step 1 processed"}

def step2(state: StreamState) -> StreamState:
    print("  [Node: step2 executing...]")
    import time
    time.sleep(0.5)
    return {"step": 2, "data": state["data"] + " -> Step 2 processed"}

def step3(state: StreamState) -> StreamState:
    print("  [Node: step3 executing...]")
    import time
    time.sleep(0.5)
    return {"step": 3, "result": state["data"] + " -> Complete!"}

# Build the graph
stream_workflow = StateGraph(StreamState)
stream_workflow.add_node("step1", step1)
stream_workflow.add_node("step2", step2)
stream_workflow.add_node("step3", step3)

stream_workflow.add_edge(START, "step1")
stream_workflow.add_edge("step1", "step2")
stream_workflow.add_edge("step2", "step3")
stream_workflow.add_edge("step3", END)

stream_app = stream_workflow.compile()

print("✅ Streaming graph created!")


In [ ]:
# Stream with mode="values" - Get full state after each node
print("🔄 Streaming with mode='values' (Full State after each node):\n")
print("="*70)

for i, chunk in enumerate(stream_app.stream({"step": 0, "data": "", "result": ""}, stream_mode="values"), 1):
    print(f"\n📦 Chunk {i}:")
    print(f"   State: {chunk}")
    
print("\n" + "="*70)
print("✅ Streaming complete!")


## Durable Execution

Durable execution allows workflows to continue running over long periods, providing mechanisms to handle failures and resume operations.

### What is Durable Execution?

Durable execution ensures that workflows can survive system failures, network interruptions, and other unexpected events. It provides:

- **Fault tolerance**: Continue execution despite failures
- **Long-running support**: Handle processes that take hours or days
- **Automatic retries**: Retry failed operations intelligently
- **State consistency**: Maintain data integrity across interruptions

### Key Components

1. **Checkpoints**: Save state at critical points
2. **Retry Logic**: Automatically retry failed operations
3. **Error Handling**: Graceful degradation and recovery
4. **Monitoring**: Track execution progress and health

### Failure Scenarios Handled

- **Network timeouts**: Retry with exponential backoff
- **Service unavailability**: Wait and retry when available
- **Resource exhaustion**: Scale up or queue operations
- **System crashes**: Resume from last checkpoint

### Benefits

- **Reliability**: High success rate for critical operations
- **Scalability**: Handle varying loads gracefully
- **Monitoring**: Track execution metrics and health
- **Cost efficiency**: Avoid redoing expensive operations

**Diagram:**

```
Normal Flow:     Node1 → Node2 → Node3 → Complete
Failure Flow:    Node1 → Node2 → [FAIL] → Retry → Node2 → Node3 → Complete
                 ↓                    ↓
               Checkpoint          Resume from Checkpoint
```

**Key Concept:** Durable execution transforms fragile operations into robust, enterprise-grade workflows.



## Example 2: Streaming Updates (Only Changes)


In [ ]:
# Stream with mode="updates" - Get only the updates from each node
print("🔄 Streaming with mode='updates' (Only node updates):\n")
print("="*70)

for i, chunk in enumerate(stream_app.stream({"step": 0, "data": "", "result": ""}, stream_mode="updates"), 1):
    print(f"\n📝 Update {i}:")
    print(f"   {chunk}")
    
print("\n" + "="*70)
print("✅ Streaming complete!")
print("\n💡 Notice: updates mode shows only what each node changed, not the full state")


## Key Takeaways - Streaming

✅ **When to use:**
- Long-running LLM calls where you want to show progress
- Real-time user feedback in chatbots
- Monitoring multi-step workflow progress

💡 **Stream Modes:**
- `values`: Full state after each node (good for debugging)
- `updates`: Only changes from each node (efficient)
- `messages`: Token-by-token streaming (for LLM responses)

⚠️ **Common Pitfalls:**
- Don't use streaming for simple, fast operations (adds overhead)
- Remember to handle stream chunks appropriately in your UI

---


# 2️⃣ Persistence

<a id="persistence"></a>

## Overview

**Persistence** allows you to save the state of your graph at any point, enabling:
- Pausing and resuming workflows
- Recovering from failures
- Maintaining conversation history across sessions
- Human-in-the-loop workflows

## Checkpointers

LangGraph provides several checkpointer implementations:

1. **MemorySaver**: In-memory (for development/testing)
2. **SqliteSaver**: SQLite database (local persistence)
3. **PostgresSaver**: PostgreSQL (production-grade)

---

## Example 1: Basic Persistence with MemorySaver


In [ ]:
# Define state for a conversation
class ConversationState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    step_count: int

def chat_node(state: ConversationState) -> ConversationState:
    """Simulate a chat response"""
    user_message = state["messages"][-1].content
    response = f"Echo: {user_message} (Step {state.get('step_count', 0) + 1})"
    print(f"💬 Bot: {response}")
    return {
        "messages": [AIMessage(content=response)],
        "step_count": state.get("step_count", 0) + 1
    }

# Create graph with checkpointer
memory = MemorySaver()

persist_workflow = StateGraph(ConversationState)
persist_workflow.add_node("chat", chat_node)
persist_workflow.add_edge(START, "chat")
persist_workflow.add_edge("chat", END)

# Compile WITH checkpointer
persist_app = persist_workflow.compile(checkpointer=memory)

print("✅ Graph with persistence created!")
print("💾 Using MemorySaver checkpointer")


In [ ]:
# Use persistence with thread_id to maintain conversation
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "conversation_1"}}

print("🗨️ Turn 1:")
result1 = persist_app.invoke(
    {"messages": [HumanMessage(content="Hello!")], "step_count": 0},
    config=config
)
print(f"State after turn 1: {len(result1['messages'])} messages, Step: {result1['step_count']}\n")

print("🗨️ Turn 2:")
result2 = persist_app.invoke(
    {"messages": [HumanMessage(content="How are you?")]},
    config=config
)
print(f"State after turn 2: {len(result2['messages'])} messages, Step: {result2['step_count']}\n")

print("🗨️ Turn 3:")
result3 = persist_app.invoke(
    {"messages": [HumanMessage(content="Tell me a joke")]},
    config=config
)
print(f"State after turn 3: {len(result3['messages'])} messages, Step: {result3['step_count']}\n")

print("="*70)
print("✅ All messages preserved across invocations!")
print(f"📝 Total messages in conversation: {len(result3['messages'])}")


## Example 2: Multiple Conversations (Thread Management)


## Memory

Memory in LangGraph allows for storing conversation history and context, enabling context-aware interactions and decisions.

### What is Memory?

Memory enables LangGraph applications to remember past interactions, user preferences, and contextual information. This creates:

- **Context-aware responses**: Build on previous conversations
- **Personalization**: Adapt to individual user patterns
- **Continuity**: Maintain conversation flow across sessions
- **Learning**: Improve responses based on interaction history

### Types of Memory

1. **Short-term Memory**: Within a single conversation thread
2. **Long-term Memory**: Across multiple sessions and users
3. **Episodic Memory**: Remember specific events and interactions
4. **Semantic Memory**: Store learned patterns and knowledge

### Memory Storage Options

- **In-memory**: Fast but temporary (lost on restart)
- **Database**: Persistent storage (PostgreSQL, SQLite)
- **Vector stores**: Semantic search and retrieval
- **External APIs**: Integration with external memory systems

### Key Features

- **Automatic storage**: Save interactions without manual intervention
- **Context retrieval**: Access relevant past information
- **Memory management**: Handle memory limits and cleanup
- **Privacy controls**: User data isolation and compliance

### Use Cases

- **Chatbots**: Remember user preferences and conversation history
- **Personal assistants**: Learn user behavior and preferences
- **Customer service**: Access customer interaction history
- **Educational systems**: Track learning progress and adapt content

**Diagram:**

```
Current Interaction: "What's the weather?"
                    ↓
Memory Lookup: Previous weather queries, location preferences
                    ↓
Context-Aware Response: "The weather in your location (NYC) is..."
                    ↓
Memory Update: Save current interaction and preferences
```

**Key Concept:** Memory transforms one-shot interactions into continuous, personalized experiences.



In [ ]:
# Different threads maintain separate conversations
print("👤 User A - Thread 1:")
config_a = {"configurable": {"thread_id": "user_a"}}
persist_app.invoke(
    {"messages": [HumanMessage(content="I like Python")], "step_count": 0},
    config=config_a
)

print("\n👤 User B - Thread 2:")
config_b = {"configurable": {"thread_id": "user_b"}}
persist_app.invoke(
    {"messages": [HumanMessage(content="I like JavaScript")], "step_count": 0},
    config=config_b
)

print("\n👤 User A - Continuing Thread 1:")
result_a = persist_app.invoke(
    {"messages": [HumanMessage(content="What did I say I liked?")]},
    config=config_a
)

print("\n" + "="*70)
print("✅ Each thread maintains its own state independently!")
print(f"📊 User A conversation has {len(result_a['messages'])} messages")


In [ ]:
# Get state at any point using get_state
current_state = persist_app.get_state(config_a)

print("📋 Current State for User A:")
print(f"   Values: {current_state.values}")
print(f"   Next node: {current_state.next}")
print(f"   Config: {current_state.config}")
print(f"\n💬 Message history:")
for i, msg in enumerate(current_state.values['messages'], 1):
    msg_type = "User" if isinstance(msg, HumanMessage) else "Bot"
    print(f"   {i}. [{msg_type}]: {msg.content}")


## Key Takeaways - Persistence

✅ **When to use:**
- Multi-turn conversations that need to remember context
- Long-running workflows that might be interrupted
- Human-in-the-loop processes requiring approval
- Multi-user applications with separate sessions

💡 **Checkpointer Types:**
- `MemorySaver`: Development/testing (lost on restart)
- `SqliteSaver`: Local persistence (single machine)
- `PostgresSaver`: Production (scalable, distributed)

⚠️ **Common Pitfalls:**
- Always use unique `thread_id` for different conversations
- Remember to pass `config` with `thread_id` in every invocation
- Checkpointer data persists across app restarts (except MemorySaver)

---


# 3️⃣ Durable Execution

<a id="durable-execution"></a>

## Overview

**Durable Execution** ensures that your graph can **resume from the last checkpoint** after interruptions, making it ideal for:
- Long-running workflows that might fail
- Production systems requiring reliability
- Multi-step processes with external dependencies
- Workflows that need to survive server restarts

## Key Features

- **Automatic checkpointing** at each node
- **Resume from any checkpoint** after failure
- **State recovery** with full context
- **Error handling** and retry logic

---

## Example 1: Long-Running Workflow with Checkpoints


In [ ]:
# Define state for a long-running process
class DurableState(TypedDict):
    step: int
    data: str
    status: str
    error_count: int
    final_result: str

def step1_process(state: DurableState) -> DurableState:
    """First processing step"""
    print("🔄 Step 1: Starting data processing...")
    import time
    time.sleep(1)  # Simulate work
    print("✅ Step 1: Completed successfully")
    return {
        "step": 1,
        "data": "Step 1 processed data",
        "status": "completed"
    }

def step2_process(state: DurableState) -> DurableState:
    """Second processing step"""
    print("🔄 Step 2: Processing intermediate data...")
    import time
    time.sleep(1)  # Simulate work
    print("✅ Step 2: Completed successfully")
    return {
        "step": 2,
        "data": state["data"] + " -> Step 2 enhanced",
        "status": "completed"
    }

def step3_process(state: DurableState) -> DurableState:
    """Final processing step"""
    print("🔄 Step 3: Finalizing results...")
    import time
    time.sleep(1)  # Simulate work
    print("✅ Step 3: Completed successfully")
    return {
        "step": 3,
        "data": state["data"] + " -> Finalized",
        "status": "completed",
        "final_result": "Workflow completed successfully!"
    }

# Build durable workflow
durable_workflow = StateGraph(DurableState)
durable_workflow.add_node("step1", step1_process)
durable_workflow.add_node("step2", step2_process)
durable_workflow.add_node("step3", step3_process)

# Linear flow
durable_workflow.add_edge(START, "step1")
durable_workflow.add_edge("step1", "step2")
durable_workflow.add_edge("step2", "step3")
durable_workflow.add_edge("step3", END)

# Compile with checkpointer for durability
memory_checkpointer = MemorySaver()
durable_app = durable_workflow.compile(checkpointer=memory_checkpointer)

print("✅ Durable workflow created!")
print("💾 Each step will be automatically checkpointed")


In [ ]:
# Execute the durable workflow
config = {"configurable": {"thread_id": "durable_workflow_1"}}

print("🚀 Starting durable workflow...")
print("="*60)

try:
    result = durable_app.invoke(
        {"step": 0, "data": "", "status": "starting", "error_count": 0, "final_result": ""},
        config=config
    )
    print(f"\n🎉 Workflow completed successfully!")
    print(f"📊 Final result: {result['final_result']}")
    print(f"📈 Total steps completed: {result['step']}")
    
except Exception as e:
    print(f"\n❌ Workflow failed: {e}")
    print("🔄 The workflow can be resumed from the last checkpoint")


## Context

Context provides external data and configuration inputs, customizing workflows based on external conditions and user-specific data.

### What is Context?

Context in LangGraph refers to external information that influences how workflows behave. This includes:

- **User-specific data**: Preferences, permissions, history
- **Environmental factors**: Time, location, system state
- **Configuration settings**: Feature flags, parameters, thresholds
- **External data sources**: APIs, databases, files

### Types of Context

1. **Runtime Context**: Information available during execution
2. **Configuration Context**: Settings and parameters
3. **User Context**: User-specific information and preferences
4. **System Context**: Environment and infrastructure details

### Context Sources

- **Environment variables**: System-level configuration
- **Configuration files**: Application settings
- **User profiles**: Personal preferences and data
- **External APIs**: Real-time data from services
- **Databases**: Persistent user and system data

### Key Benefits

- **Personalization**: Adapt behavior to individual users
- **Flexibility**: Change behavior without code changes
- **Scalability**: Handle different user segments efficiently
- **Maintainability**: Centralize configuration management

### Use Cases

- **Multi-tenant systems**: Different behavior per tenant
- **A/B testing**: Vary behavior based on user segments
- **Feature flags**: Enable/disable features dynamically
- **Personalization**: Customize experience per user

**Diagram:**

```
Workflow Execution:
Input → Context Lookup → Contextual Processing → Output
  ↓           ↓                    ↓
User Data → User Context → Personalized Response
System Config → System Context → Appropriate Behavior
External APIs → Real-time Context → Current Information
```

**Key Concept:** Context transforms generic workflows into personalized, adaptive systems.



## Example 2: Simulating Failure and Recovery


In [ ]:
# Create a workflow that can fail and be resumed
def risky_step1(state: DurableState) -> DurableState:
    """Step that might fail"""
    print("🔄 Risky Step 1: Processing...")
    import time
    time.sleep(0.5)
    print("✅ Risky Step 1: Completed")
    return {"step": 1, "data": "Step 1 done", "status": "completed"}

def risky_step2(state: DurableState) -> DurableState:
    """Step that will fail"""
    print("🔄 Risky Step 2: Processing...")
    import time
    time.sleep(0.5)
    print("❌ Risky Step 2: Simulating failure!")
    raise Exception("Simulated failure in step 2")

def risky_step3(state: DurableState) -> DurableState:
    """Step that would run after recovery"""
    print("🔄 Risky Step 3: Processing...")
    import time
    time.sleep(0.5)
    print("✅ Risky Step 3: Completed")
    return {
        "step": 3,
        "data": state["data"] + " -> Step 3 recovered",
        "status": "completed",
        "final_result": "Recovered successfully!"
    }

# Build risky workflow
risky_workflow = StateGraph(DurableState)
risky_workflow.add_node("risky_step1", risky_step1)
risky_workflow.add_node("risky_step2", risky_step2)
risky_workflow.add_node("risky_step3", risky_step3)

risky_workflow.add_edge(START, "risky_step1")
risky_workflow.add_edge("risky_step1", "risky_step2")
risky_workflow.add_edge("risky_step2", "risky_step3")
risky_workflow.add_edge("risky_step3", END)

risky_app = risky_workflow.compile(checkpointer=memory_checkpointer)

print("✅ Risky workflow created!")
print("⚠️ Step 2 will fail, but we can resume from step 1")


In [ ]:
# Execute risky workflow - it will fail
config_risky = {"configurable": {"thread_id": "risky_workflow_1"}}

print("🚀 Starting risky workflow...")
print("="*60)

try:
    result = risky_app.invoke(
        {"step": 0, "data": "", "status": "starting", "error_count": 0, "final_result": ""},
        config=config_risky
    )
except Exception as e:
    print(f"\n❌ Workflow failed as expected: {e}")
    
    # Check the state after failure
    current_state = risky_app.get_state(config_risky)
    print(f"\n📋 State after failure:")
    print(f"   Current step: {current_state.values.get('step', 0)}")
    print(f"   Data so far: {current_state.values.get('data', '')}")
    print(f"   Status: {current_state.values.get('status', '')}")
    print(f"   Next node: {current_state.next}")
    
    print("\n🔄 The workflow can be resumed from the last successful checkpoint!")


## Example 3: Resuming from Checkpoint


In [ ]:
# Create a fixed version of step2 that won't fail
def fixed_step2(state: DurableState) -> DurableState:
    """Fixed version of step 2"""
    print("🔄 Fixed Step 2: Processing (this time it works)...")
    import time
    time.sleep(0.5)
    print("✅ Fixed Step 2: Completed successfully")
    return {
        "step": 2,
        "data": state["data"] + " -> Step 2 fixed",
        "status": "completed"
    }

# Update the workflow with the fixed step
risky_workflow.add_node("fixed_step2", fixed_step2)

# Replace the failing step with the fixed one
risky_workflow.add_edge("risky_step1", "fixed_step2")
risky_workflow.add_edge("fixed_step2", "risky_step3")

# Recompile
risky_app_fixed = risky_workflow.compile(checkpointer=memory_checkpointer)

print("✅ Workflow updated with fixed step 2")
print("🔄 Now we can resume from the checkpoint")


In [ ]:
# Resume the workflow from the last checkpoint
print("\n🔄 Resuming workflow from checkpoint...")
print("="*60)

try:
    # Resume by invoking with the same config
    # LangGraph will automatically start from the last successful checkpoint
    result = risky_app_fixed.invoke(
        {"step": 0, "data": "", "status": "starting", "error_count": 0, "final_result": ""},
        config=config_risky
    )
    
    print(f"\n🎉 Workflow resumed and completed successfully!")
    print(f"📊 Final result: {result['final_result']}")
    print(f"📈 Total steps completed: {result['step']}")
    print(f"📝 Final data: {result['data']}")
    
except Exception as e:
    print(f"\n❌ Resume failed: {e}")

print("\n" + "="*60)
print("✅ Durable execution demonstrated!")
print("💡 Key insight: Workflows can survive failures and resume from checkpoints")


## Key Takeaways - Durable Execution

✅ **When to use:**
- Long-running workflows that might be interrupted
- Production systems requiring high reliability
- Multi-step processes with external dependencies
- Workflows that need to survive server restarts

💡 **Key Features:**
- Automatic checkpointing at each node
- Resume from any checkpoint after failure
- State recovery with full context
- Error handling and retry capabilities

⚠️ **Common Pitfalls:**
- Always use a checkpointer for durable execution
- Handle errors gracefully in your nodes
- Test failure and recovery scenarios
- Consider checkpoint frequency for performance

---


## Models

Models enable integration with various LLMs, facilitating AI-powered applications and natural language processing.

### What are Models?

Models in LangGraph refer to Large Language Models (LLMs) that provide natural language understanding and generation capabilities. This includes:

- **OpenAI models**: GPT-3.5, GPT-4, GPT-4 Turbo
- **Anthropic models**: Claude-3, Claude-3.5 Sonnet
- **Open-source models**: Llama, Mistral, CodeLlama
- **Specialized models**: Code generation, math, reasoning

### Model Integration Features

1. **Multiple providers**: Support for various LLM providers
2. **Model switching**: Dynamically choose models based on task
3. **Fallback handling**: Graceful degradation when models fail
4. **Cost optimization**: Choose models based on cost/performance

### Key Capabilities

- **Text generation**: Create human-like responses
- **Text understanding**: Parse and comprehend input
- **Code generation**: Generate and explain code
- **Reasoning**: Solve complex problems step-by-step
- **Translation**: Convert between languages and formats

### Integration Patterns

- **Single model**: Use one model for all tasks
- **Multi-model**: Different models for different tasks
- **Ensemble**: Combine multiple model outputs
- **Cascading**: Fallback to simpler models when needed

### Use Cases

- **Chatbots**: Natural conversation interfaces
- **Content generation**: Articles, summaries, translations
- **Code assistance**: Programming help and debugging
- **Data analysis**: Natural language data insights
- **Education**: Personalized learning assistance

**Diagram:**

```
Input Text → Model Selection → LLM Processing → Generated Response
     ↓              ↓                ↓                ↓
User Query → Choose Best Model → Process with AI → Return Answer
Task Type → Route to Specialist → Model-Specific → Optimized Output
```

**Key Concept:** Models transform structured workflows into intelligent, natural language-powered systems.



# 4️⃣ Memory

<a id="memory"></a>

## Overview

**Memory** in LangGraph refers to the ability to store and retrieve information across different interactions and threads. It enables:
- **Short-term memory**: Within a single conversation thread
- **Long-term memory**: Across multiple conversations and users
- **Cross-thread memory**: Sharing information between different threads
- **Contextual memory**: Remembering user preferences and history

## Memory Types

1. **Store**: Persistent storage for long-term memory
2. **Checkpointer**: State persistence for conversation continuity
3. **Context**: Runtime information passed to nodes

---

## Example 1: Short-term Memory (Within Thread)


In [ ]:
# Define state with memory fields
class MemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_name: str
    conversation_count: int
    remembered_facts: list[str]
    current_topic: str

def memory_chat_node(state: MemoryState) -> MemoryState:
    """Chat node that remembers conversation context"""
    messages = state["messages"]
    user_name = state.get("user_name", "User")
    conversation_count = state.get("conversation_count", 0)
    remembered_facts = state.get("remembered_facts", [])
    
    # Get the latest user message
    latest_message = messages[-1].content if messages else ""
    
    # Simulate remembering previous facts
    memory_response = f"Hello {user_name}! "
    if conversation_count > 0:
        memory_response += f"This is our {conversation_count + 1}st conversation. "
        if remembered_facts:
            memory_response += f"I remember: {', '.join(remembered_facts[-2:])}. "
    
    # Extract and remember new facts from the message
    new_facts = []
    if "my name is" in latest_message.lower():
        name_part = latest_message.lower().split("my name is")[-1].strip()
        new_facts.append(f"User's name is {name_part}")
    if "i like" in latest_message.lower():
        like_part = latest_message.lower().split("i like")[-1].strip()
        new_facts.append(f"User likes {like_part}")
    
    response = memory_response + f"You said: '{latest_message}'"
    
    print(f"🧠 Memory Chat: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "conversation_count": conversation_count + 1,
        "remembered_facts": remembered_facts + new_facts
    }

# Build memory-enabled workflow
memory_workflow = StateGraph(MemoryState)
memory_workflow.add_node("memory_chat", memory_chat_node)
memory_workflow.add_edge(START, "memory_chat")
memory_workflow.add_edge("memory_chat", END)

memory_app = memory_workflow.compile(checkpointer=MemorySaver())

print("✅ Memory-enabled chat workflow created!")
print("🧠 This chat will remember facts across turns")


In [ ]:
# Test memory across multiple turns
config_memory = {"configurable": {"thread_id": "memory_conversation"}}

print("🗨️ Testing Memory Across Turns:")
print("="*60)

# Turn 1: Introduction
result1 = memory_app.invoke(
    {
        "messages": [HumanMessage(content="Hi, my name is Alice")],
        "user_name": "",
        "conversation_count": 0,
        "remembered_facts": [],
        "current_topic": ""
    },
    config=config_memory
)
print(f"📝 Remembered facts: {result1['remembered_facts']}\n")

# Turn 2: Share preferences
result2 = memory_app.invoke(
    {
        "messages": [HumanMessage(content="I like programming and coffee")],
        "user_name": "Alice",
        "conversation_count": 1,
        "remembered_facts": result1['remembered_facts'],
        "current_topic": ""
    },
    config=config_memory
)
print(f"📝 Remembered facts: {result2['remembered_facts']}\n")

# Turn 3: Test memory recall
result3 = memory_app.invoke(
    {
        "messages": [HumanMessage(content="What do you remember about me?")],
        "user_name": "Alice",
        "conversation_count": 2,
        "remembered_facts": result2['remembered_facts'],
        "current_topic": ""
    },
    config=config_memory
)

print("="*60)
print("✅ Memory demonstration complete!")
print(f"🧠 Total facts remembered: {len(result3['remembered_facts'])}")


## Example 2: Cross-Thread Memory Sharing


In [ ]:
# Create a shared memory store
from langgraph.store.memory import MemoryStore

# Initialize shared memory store
shared_store = MemoryStore()

# Define state with shared memory
class SharedMemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    shared_data: dict

def shared_memory_node(state: SharedMemoryState) -> SharedMemoryState:
    """Node that uses shared memory across threads"""
    user_id = state["user_id"]
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Store user data in shared memory
    if "store" in latest_message.lower():
        data_to_store = latest_message.split("store")[-1].strip()
        shared_store.put(user_id, {"data": data_to_store, "timestamp": "now"})
        response = f"✅ Stored '{data_to_store}' in shared memory for user {user_id}"
    elif "retrieve" in latest_message.lower():
        stored_data = shared_store.get(user_id)
        if stored_data:
            response = f"📋 Retrieved from shared memory: {stored_data}"
        else:
            response = f"❌ No data found in shared memory for user {user_id}"
    else:
        response = f"Echo: {latest_message}"
    
    print(f"🧠 Shared Memory Node: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "shared_data": shared_store.get(user_id) or {}
    }

# Build shared memory workflow
shared_workflow = StateGraph(SharedMemoryState)
shared_workflow.add_node("shared_memory", shared_memory_node)
shared_workflow.add_edge(START, "shared_memory")
shared_workflow.add_edge("shared_memory", END)

shared_app = shared_workflow.compile(checkpointer=MemorySaver())

print("✅ Shared memory workflow created!")
print("🧠 Multiple users can share data through the memory store")


In [ ]:
# Test cross-thread memory sharing
print("👥 Testing Cross-Thread Memory Sharing:")
print("="*60)

# User 1 stores data
config_user1 = {"configurable": {"thread_id": "user_1"}}
result1 = shared_app.invoke(
    {
        "messages": [HumanMessage(content="store my favorite color is blue")],
        "user_id": "user_1",
        "shared_data": {}
    },
    config=config_user1
)

# User 2 stores different data
config_user2 = {"configurable": {"thread_id": "user_2"}}
result2 = shared_app.invoke(
    {
        "messages": [HumanMessage(content="store my favorite food is pizza")],
        "user_id": "user_2",
        "shared_data": {}
    },
    config=config_user2
)

# User 1 retrieves their data
result3 = shared_app.invoke(
    {
        "messages": [HumanMessage(content="retrieve my data")],
        "user_id": "user_1",
        "shared_data": {}
    },
    config=config_user1
)

# User 2 retrieves their data
result4 = shared_app.invoke(
    {
        "messages": [HumanMessage(content="retrieve my data")],
        "user_id": "user_2",
        "shared_data": {}
    },
    config=config_user2
)

print("="*60)
print("✅ Cross-thread memory sharing demonstrated!")
print("🧠 Each user has their own isolated memory space")


## Example 3: Long-term Memory with Store


## Tools

Tools interface directly with external systems, allowing for seamless integration and interaction.

### What are Tools?

Tools in LangGraph are functions that enable workflows to interact with external systems, APIs, and services. They provide:

- **External API access**: Call web services and APIs
- **Function calling**: Execute custom functions and logic
- **Data retrieval**: Fetch information from databases and files
- **System integration**: Connect with external tools and services

### Types of Tools

1. **Built-in tools**: Pre-defined tools for common tasks
2. **Custom tools**: User-defined functions for specific needs
3. **API tools**: Interface with external web services
4. **Database tools**: Query and manipulate databases
5. **File tools**: Read, write, and process files

### Tool Integration Patterns

- **Tool selection**: Choose appropriate tools based on context
- **Tool chaining**: Use multiple tools in sequence
- **Tool validation**: Verify tool inputs and outputs
- **Error handling**: Handle tool failures gracefully

### Key Features

- **Type safety**: Strong typing for tool inputs/outputs
- **Documentation**: Automatic tool description generation
- **Validation**: Input/output validation and sanitization
- **Error handling**: Robust error handling and recovery
- **Logging**: Track tool usage and performance

### Use Cases

- **Web scraping**: Extract data from websites
- **API integration**: Connect with external services
- **Data processing**: Transform and analyze data
- **File operations**: Read, write, and manipulate files
- **Database queries**: Access and update databases

**Diagram:**

```
User Request → Tool Selection → Tool Execution → Result Processing → Response
     ↓              ↓               ↓                ↓              ↓
"What's the weather?" → Weather API → API Call → Parse Data → "It's 72°F"
"Calculate 2+2" → Math Tool → Calculation → Return 4 → "The answer is 4"
```

**Key Concept:** Tools transform static workflows into dynamic, interactive systems that can access and manipulate external data and services.



In [ ]:
# Create a workflow with long-term memory
class LongTermMemoryState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    session_count: int
    total_interactions: int

def long_term_memory_node(state: LongTermMemoryState) -> LongTermMemoryState:
    """Node with long-term memory capabilities"""
    user_id = state["user_id"]
    messages = state["messages"]
    session_count = state.get("session_count", 0)
    total_interactions = state.get("total_interactions", 0)
    
    latest_message = messages[-1].content if messages else ""
    
    # Get existing user data from long-term memory
    existing_data = shared_store.get(f"user_{user_id}")
    if existing_data:
        total_sessions = existing_data.get("total_sessions", 0)
        all_interactions = existing_data.get("all_interactions", 0)
    else:
        total_sessions = 0
        all_interactions = 0
    
    # Update long-term memory
    new_total_sessions = total_sessions + 1
    new_all_interactions = all_interactions + 1
    
    shared_store.put(f"user_{user_id}", {
        "total_sessions": new_total_sessions,
        "all_interactions": new_all_interactions,
        "last_seen": "now",
        "preferences": existing_data.get("preferences", {}) if existing_data else {}
    })
    
    # Generate response with memory context
    response = f"Welcome back! "
    if new_total_sessions > 1:
        response += f"This is your {new_total_sessions}rd session. "
        response += f"You've had {new_all_interactions} total interactions. "
    else:
        response += f"Welcome! This is your first session. "
    
    response += f"You said: '{latest_message}'"
    
    print(f"🧠 Long-term Memory: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "session_count": session_count + 1,
        "total_interactions": new_all_interactions
    }

# Build long-term memory workflow
long_term_workflow = StateGraph(LongTermMemoryState)
long_term_workflow.add_node("long_term_memory", long_term_memory_node)
long_term_workflow.add_edge(START, "long_term_memory")
long_term_workflow.add_edge("long_term_memory", END)

long_term_app = long_term_workflow.compile(checkpointer=MemorySaver())

print("✅ Long-term memory workflow created!")
print("🧠 This workflow remembers users across multiple sessions")


In [ ]:
# Test long-term memory across sessions
print("🔄 Testing Long-term Memory Across Sessions:")
print("="*60)

# Session 1
config_session1 = {"configurable": {"thread_id": "session_1"}}
result1 = long_term_app.invoke(
    {
        "messages": [HumanMessage(content="Hello, I'm new here")],
        "user_id": "alice",
        "session_count": 0,
        "total_interactions": 0
    },
    config=config_session1
)

# Session 2 (simulating a new session)
config_session2 = {"configurable": {"thread_id": "session_2"}}
result2 = long_term_app.invoke(
    {
        "messages": [HumanMessage(content="Hi again!")],
        "user_id": "alice",
        "session_count": 0,
        "total_interactions": 0
    },
    config=config_session2
)

# Session 3
config_session3 = {"configurable": {"thread_id": "session_3"}}
result3 = long_term_app.invoke(
    {
        "messages": [HumanMessage(content="How many times have we talked?")],
        "user_id": "alice",
        "session_count": 0,
        "total_interactions": 0
    },
    config=config_session3
)

print("="*60)
print("✅ Long-term memory demonstrated!")
print("🧠 The system remembers Alice across multiple sessions")
print(f"📊 Total interactions tracked: {result3['total_interactions']}")


## Key Takeaways - Memory

✅ **When to use:**
- Multi-turn conversations requiring context
- User preference tracking
- Cross-session data persistence
- Shared knowledge between users

💡 **Memory Types:**
- **Short-term**: Within a single conversation thread
- **Long-term**: Across multiple sessions using Store
- **Cross-thread**: Shared data between different users
- **Contextual**: Runtime information passed to nodes

⚠️ **Common Pitfalls:**
- Choose appropriate memory scope (thread vs global)
- Consider memory cleanup and expiration
- Handle memory access errors gracefully
- Balance memory usage with performance

---


# 5️⃣ Context

<a id="context"></a>

## Overview

**Context** in LangGraph allows you to pass external data and configuration to your graph nodes, enabling:
- **Runtime configuration**: Dynamic behavior based on external factors
- **User preferences**: Personalized responses based on user data
- **Environment variables**: Different behavior in dev vs production
- **External data**: Information from databases, APIs, or other sources

## Context Types

1. **Configurable**: Runtime configuration passed via `config` parameter
2. **Context**: Additional data passed to nodes during execution
3. **Environment**: System-level configuration and secrets

---

## Example 1: Basic Context Passing


In [ ]:
# Define state with context fields
class ContextState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_preferences: dict
    environment: str
    response_style: str

def context_aware_node(state: ContextState) -> ContextState:
    """Node that uses context to customize responses"""
    messages = state["messages"]
    user_preferences = state.get("user_preferences", {})
    environment = state.get("environment", "development")
    response_style = state.get("response_style", "formal")
    
    latest_message = messages[-1].content if messages else ""
    
    # Customize response based on context
    if response_style == "casual":
        greeting = "Hey there!"
        ending = "Hope that helps!"
    elif response_style == "formal":
        greeting = "Good day!"
        ending = "I hope this information is helpful."
    else:
        greeting = "Hello!"
        ending = "Let me know if you need anything else."
    
    # Add environment-specific information
    env_info = ""
    if environment == "production":
        env_info = " (Production Mode)"
    elif environment == "development":
        env_info = " (Dev Mode)"
    
    # Use user preferences
    name = user_preferences.get("name", "User")
    language = user_preferences.get("language", "English")
    
    response = f"{greeting} {name}! {env_info}\n"
    response += f"You said: '{latest_message}'\n"
    response += f"Language: {language}\n"
    response += f"{ending}"
    
    print(f"🎯 Context-Aware Response: {response}")
    
    return {
        "messages": [AIMessage(content=response)]
    }

# Build context-aware workflow
context_workflow = StateGraph(ContextState)
context_workflow.add_node("context_aware", context_aware_node)
context_workflow.add_edge(START, "context_aware")
context_workflow.add_edge("context_aware", END)

context_app = context_workflow.compile()

print("✅ Context-aware workflow created!")
print("🎯 This workflow uses context to customize responses")


In [ ]:
# Test context with different configurations
print("🎯 Testing Context with Different Configurations:")
print("="*60)

# Test 1: Casual style in development
print("Test 1: Casual style in development")
result1 = context_app.invoke(
    {
        "messages": [HumanMessage(content="Tell me about LangGraph")],
        "user_preferences": {"name": "Alice", "language": "English"},
        "environment": "development",
        "response_style": "casual"
    }
)
print(f"Response: {result1['messages'][-1].content}\n")

# Test 2: Formal style in production
print("Test 2: Formal style in production")
result2 = context_app.invoke(
    {
        "messages": [HumanMessage(content="Explain the benefits")],
        "user_preferences": {"name": "Bob", "language": "Spanish"},
        "environment": "production",
        "response_style": "formal"
    }
)
print(f"Response: {result2['messages'][-1].content}\n")

# Test 3: Different language preference
print("Test 3: Different language preference")
result3 = context_app.invoke(
    {
        "messages": [HumanMessage(content="How does it work?")],
        "user_preferences": {"name": "Carlos", "language": "French"},
        "environment": "staging",
        "response_style": "casual"
    }
)
print(f"Response: {result3['messages'][-1].content}\n")

print("="*60)
print("✅ Context demonstration complete!")
print("🎯 Same workflow, different behaviors based on context")


## Example 2: Runtime Configuration


## Human-in-the-Loop

Human-in-the-Loop enables workflows to pause execution and wait for human input, creating interactive and controlled AI systems.

### What is Human-in-the-Loop?

Human-in-the-Loop (HITL) allows LangGraph workflows to pause execution and request human intervention. This creates:

- **Controlled AI**: Human oversight of AI decisions
- **Quality assurance**: Human validation of outputs
- **Interactive workflows**: Dynamic modification based on human input
- **Safety mechanisms**: Prevent harmful or incorrect outputs

### Key Features

1. **Execution pausing**: Stop workflow at designated points
2. **Human input collection**: Gather decisions and feedback
3. **Conditional continuation**: Resume based on human decisions
4. **Approval workflows**: Require explicit human approval
5. **Feedback integration**: Incorporate human corrections

### Use Cases

- **Content moderation**: Human review of AI-generated content
- **Medical diagnosis**: Doctor validation of AI recommendations
- **Financial decisions**: Human approval for high-risk transactions
- **Creative processes**: Human guidance for AI-generated content
- **Quality control**: Human validation of automated processes

### Implementation Patterns

- **Approval gates**: Require human approval before proceeding
- **Review checkpoints**: Human review at critical points
- **Interactive editing**: Human modification of AI outputs
- **Escalation workflows**: Route complex cases to humans
- **Feedback loops**: Learn from human corrections

**Diagram:**

```
AI Processing → Human Review → Decision Point → Continue/Modify/Stop
     ↓              ↓              ↓
Generate Content → Human Check → Approved/Rejected/Modified
Analyze Data → Human Validation → Proceed/Revise/Stop
```

**Key Concept:** Human-in-the-Loop transforms autonomous AI into collaborative, human-guided systems.


In [ ]:
# Create a workflow that uses runtime configuration
class ConfigState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    config_data: dict
    output: str

def configurable_node(state: ConfigState) -> ConfigState:
    """Node that uses runtime configuration"""
    messages = state["messages"]
    config_data = state.get("config_data", {})
    
    latest_message = messages[-1].content if messages else ""
    
    # Use configuration to determine behavior
    max_length = config_data.get("max_response_length", 100)
    include_timestamp = config_data.get("include_timestamp", False)
    debug_mode = config_data.get("debug_mode", False)
    
    # Generate response based on configuration
    response = f"Processing: {latest_message}"
    
    if include_timestamp:
        import datetime
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        response += f" [Timestamp: {timestamp}]"
    
    if debug_mode:
        response += f" [Debug: max_length={max_length}, timestamp={include_timestamp}]"
    
    # Truncate if too long
    if len(response) > max_length:
        response = response[:max_length-3] + "..."
    
    print(f"⚙️ Configurable Node: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "output": response
    }

# Build configurable workflow
config_workflow = StateGraph(ConfigState)
config_workflow.add_node("configurable", configurable_node)
config_workflow.add_edge(START, "configurable")
config_workflow.add_edge("configurable", END)

config_app = config_workflow.compile()

print("✅ Configurable workflow created!")
print("⚙️ This workflow uses runtime configuration to modify behavior")


In [ ]:
# Test different runtime configurations
print("⚙️ Testing Runtime Configuration:")
print("="*60)

# Test 1: Development configuration
print("Test 1: Development configuration")
result1 = config_app.invoke(
    {
        "messages": [HumanMessage(content="This is a long message that will be processed")],
        "config_data": {
            "max_response_length": 50,
            "include_timestamp": True,
            "debug_mode": True
        },
        "output": ""
    }
)
print(f"Response: {result1['output']}\n")

# Test 2: Production configuration
print("Test 2: Production configuration")
result2 = config_app.invoke(
    {
        "messages": [HumanMessage(content="This is a long message that will be processed")],
        "config_data": {
            "max_response_length": 200,
            "include_timestamp": False,
            "debug_mode": False
        },
        "output": ""
    }
)
print(f"Response: {result2['output']}\n")

# Test 3: Custom configuration
print("Test 3: Custom configuration")
result3 = config_app.invoke(
    {
        "messages": [HumanMessage(content="This is a long message that will be processed")],
        "config_data": {
            "max_response_length": 30,
            "include_timestamp": True,
            "debug_mode": False
        },
        "output": ""
    }
)
print(f"Response: {result3['output']}\n")

print("="*60)
print("✅ Runtime configuration demonstrated!")
print("⚙️ Same workflow, different behavior based on configuration")


## Example 3: External Data Context


In [ ]:
# Simulate external data sources
class ExternalDataState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    external_data: dict
    personalized_response: str

# Simulate external data sources
def get_user_profile(user_id: str) -> dict:
    """Simulate fetching user profile from database"""
    profiles = {
        "user_1": {"name": "Alice", "role": "developer", "experience": "senior"},
        "user_2": {"name": "Bob", "role": "manager", "experience": "expert"},
        "user_3": {"name": "Carol", "role": "designer", "experience": "junior"}
    }
    return profiles.get(user_id, {"name": "Unknown", "role": "user", "experience": "beginner"})

def get_system_status() -> dict:
    """Simulate fetching system status"""
    return {
        "status": "healthy",
        "version": "1.2.3",
        "uptime": "99.9%",
        "last_update": "2024-01-15"
    }

def external_data_node(state: ExternalDataState) -> ExternalDataState:
    """Node that uses external data to personalize responses"""
    messages = state["messages"]
    user_id = state["user_id"]
    
    latest_message = messages[-1].content if messages else ""
    
    # Fetch external data
    user_profile = get_user_profile(user_id)
    system_status = get_system_status()
    
    # Personalize response based on external data
    name = user_profile["name"]
    role = user_profile["role"]
    experience = user_profile["experience"]
    
    response = f"Hello {name}! "
    response += f"As a {experience} {role}, "
    response += f"you asked: '{latest_message}'. "
    response += f"System status: {system_status['status']} "
    response += f"(v{system_status['version']})"
    
    print(f"🌐 External Data Node: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "external_data": {
            "user_profile": user_profile,
            "system_status": system_status
        },
        "personalized_response": response
    }

# Build external data workflow
external_workflow = StateGraph(ExternalDataState)
external_workflow.add_node("external_data", external_data_node)
external_workflow.add_edge(START, "external_data")
external_workflow.add_edge("external_data", END)

external_app = external_workflow.compile()

print("✅ External data workflow created!")
print("🌐 This workflow uses external data to personalize responses")


In [ ]:
# Test external data integration
print("🌐 Testing External Data Integration:")
print("="*60)

# Test with different users
test_cases = [
    ("user_1", "How can I improve my code?"),
    ("user_2", "What's the team's performance?"),
    ("user_3", "How do I get started with design?")
]

for user_id, message in test_cases:
    print(f"Testing with {user_id}:")
    result = external_app.invoke(
        {
            "messages": [HumanMessage(content=message)],
            "user_id": user_id,
            "external_data": {},
            "personalized_response": ""
        }
    )
    print(f"Response: {result['personalized_response']}")
    print(f"External data: {result['external_data']}\n")

print("="*60)
print("✅ External data integration demonstrated!")
print("🌐 Responses are personalized based on external user data")


## Key Takeaways - Context

✅ **When to use:**
- Personalizing responses based on user data
- Environment-specific behavior (dev vs prod)
- Runtime configuration changes
- Integrating external data sources

💡 **Context Types:**
- **Configurable**: Runtime configuration via `config` parameter
- **Context**: Additional data passed to nodes
- **External Data**: Information from databases, APIs, or other sources

⚠️ **Common Pitfalls:**
- Don't store sensitive data in context
- Validate external data before use
- Handle missing or invalid context gracefully
- Consider performance impact of external data fetching

---


# 6️⃣ Models

<a id="models"></a>

## Overview

**Models** in LangGraph refer to the integration of Large Language Models (LLMs) into your workflows, enabling:
- **LLM Integration**: Use OpenAI, Anthropic, and other providers
- **Model Binding**: Attach tools and functions to models
- **Multi-Model Workflows**: Use different models for different tasks
- **Dynamic Model Selection**: Choose models based on context or requirements

## Model Types

1. **Chat Models**: For conversational AI (GPT-4, Claude, etc.)
2. **Completion Models**: For text generation and completion
3. **Embedding Models**: For vector representations
4. **Custom Models**: Your own fine-tuned models

---

## Example 1: Basic LLM Integration


## Time Travel

Time Travel enables workflows to access and manipulate execution history, providing powerful debugging and experimentation capabilities.

### What is Time Travel?

Time Travel in LangGraph allows you to access and manipulate the execution history of workflows. This provides:

- **State inspection**: View state at any checkpoint
- **Execution rewinding**: Go back to previous states
- **Forking**: Create alternative execution paths from past states
- **Debugging**: Analyze execution flow and state changes
- **Experimentation**: Test different scenarios from the same starting point

### Key Features

1. **Checkpoint access**: Retrieve state from any saved checkpoint
2. **State inspection**: Examine state at any point in execution
3. **Execution forking**: Create alternative paths from past states
4. **State modification**: Change state and resume execution
5. **History analysis**: Understand execution flow and decisions

### Use Cases

- **Debugging**: Investigate issues in complex workflows
- **Experimentation**: Test different approaches from the same state
- **Decision analysis**: Understand why certain paths were taken
- **State recovery**: Fix issues by modifying past states
- **Learning**: Analyze successful and failed executions

### Implementation Patterns

- **State inspection**: Examine state at specific checkpoints
- **Execution forking**: Create alternative execution paths
- **State modification**: Change state and resume execution
- **History analysis**: Track execution flow and decisions
- **Debugging workflows**: Step through execution history

**Diagram:**

```
Execution Timeline:
State1 → State2 → State3 → State4 → State5
  ↓       ↓       ↓       ↓       ↓
Checkpoint1 → Checkpoint2 → Checkpoint3 → Checkpoint4 → Final

Time Travel Operations:
- Inspect State2: View state at Checkpoint2
- Fork from State3: Create alternative path from Checkpoint3
- Modify State4: Change state and resume execution
```

**Key Concept:** Time Travel transforms linear execution into explorable, debuggable workflows with full history access.


In [ ]:
# Import LLM components
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

# Define state for LLM integration
class ModelState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    model_choice: str
    response: str

def llm_node(state: ModelState) -> ModelState:
    """Node that uses an LLM to generate responses"""
    messages = state["messages"]
    model_choice = state.get("model_choice", "openai")
    
    # Select model based on choice
    if model_choice == "openai":
        # Note: This would require actual API key
        # llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
        # For demo purposes, we'll simulate the response
        response = "🤖 OpenAI Response: I'm a simulated OpenAI response. In real usage, this would call the actual API."
    elif model_choice == "anthropic":
        # claude = ChatAnthropic(model="claude-3-sonnet-20240229", temperature=0.7)
        response = "🤖 Claude Response: I'm a simulated Claude response. In real usage, this would call the actual API."
    else:
        response = "🤖 Default Response: I'm a default model response."
    
    print(f"🧠 LLM Node: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "response": response
    }

# Build LLM workflow
model_workflow = StateGraph(ModelState)
model_workflow.add_node("llm", llm_node)
model_workflow.add_edge(START, "llm")
model_workflow.add_edge("llm", END)

model_app = model_workflow.compile()

print("✅ LLM workflow created!")
print("🧠 This workflow integrates with Large Language Models")


In [ ]:
# Test different model choices
print("🧠 Testing Different Model Choices:")
print("="*60)

# Test with OpenAI
print("Test 1: OpenAI model")
result1 = model_app.invoke(
    {
        "messages": [HumanMessage(content="Explain LangGraph in simple terms")],
        "model_choice": "openai",
        "response": ""
    }
)
print(f"Response: {result1['response']}\n")

# Test with Anthropic
print("Test 2: Anthropic model")
result2 = model_app.invoke(
    {
        "messages": [HumanMessage(content="What are the benefits of using LangGraph?")],
        "model_choice": "anthropic",
        "response": ""
    }
)
print(f"Response: {result2['response']}\n")

# Test with default
print("Test 3: Default model")
result3 = model_app.invoke(
    {
        "messages": [HumanMessage(content="How do I get started?")],
        "model_choice": "default",
        "response": ""
    }
)
print(f"Response: {result3['response']}\n")

print("="*60)
print("✅ Model integration demonstrated!")
print("🧠 Different models can be selected based on requirements")


## Example 2: Multi-Model Workflow


In [ ]:
# Create a workflow that uses multiple models for different tasks
class MultiModelState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    task_type: str
    analysis_result: str
    creative_result: str
    final_response: str

def task_classifier_node(state: MultiModelState) -> MultiModelState:
    """Node that classifies the task type"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simple task classification
    if any(word in latest_message.lower() for word in ["analyze", "analysis", "data", "statistics"]):
        task_type = "analysis"
    elif any(word in latest_message.lower() for word in ["creative", "story", "poem", "write"]):
        task_type = "creative"
    else:
        task_type = "general"
    
    print(f"🔍 Task Classifier: Detected task type: {task_type}")
    
    return {"task_type": task_type}

def analysis_model_node(state: MultiModelState) -> MultiModelState:
    """Node that uses a model specialized for analysis"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simulate analysis model response
    analysis_result = f"📊 Analysis Model: Based on '{latest_message}', here's my analytical response with data insights and logical reasoning."
    
    print(f"🧠 Analysis Model: {analysis_result}")
    
    return {"analysis_result": analysis_result}

def creative_model_node(state: MultiModelState) -> MultiModelState:
    """Node that uses a model specialized for creative tasks"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simulate creative model response
    creative_result = f"🎨 Creative Model: Inspired by '{latest_message}', here's my creative response with imaginative ideas and artistic flair."
    
    print(f"🧠 Creative Model: {creative_result}")
    
    return {"creative_result": creative_result}

def general_model_node(state: MultiModelState) -> MultiModelState:
    """Node that uses a general-purpose model"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simulate general model response
    general_result = f"💬 General Model: Regarding '{latest_message}', here's my helpful and informative response."
    
    print(f"🧠 General Model: {general_result}")
    
    return {"final_response": general_result}

def response_combiner_node(state: MultiModelState) -> MultiModelState:
    """Node that combines results from different models"""
    task_type = state["task_type"]
    analysis_result = state.get("analysis_result", "")
    creative_result = state.get("creative_result", "")
    
    if task_type == "analysis":
        final_response = analysis_result
    elif task_type == "creative":
        final_response = creative_result
    else:
        final_response = state.get("final_response", "No response generated")
    
    print(f"🔄 Response Combiner: Final response: {final_response}")
    
    return {
        "messages": [AIMessage(content=final_response)],
        "final_response": final_response
    }

# Build multi-model workflow
multi_model_workflow = StateGraph(MultiModelState)

# Add nodes
multi_model_workflow.add_node("task_classifier", task_classifier_node)
multi_model_workflow.add_node("analysis_model", analysis_model_node)
multi_model_workflow.add_node("creative_model", creative_model_node)
multi_model_workflow.add_node("general_model", general_model_node)
multi_model_workflow.add_node("response_combiner", response_combiner_node)

# Add edges
multi_model_workflow.add_edge(START, "task_classifier")

# Conditional routing based on task type
def route_by_task_type(state: MultiModelState) -> str:
    task_type = state["task_type"]
    if task_type == "analysis":
        return "analysis_model"
    elif task_type == "creative":
        return "creative_model"
    else:
        return "general_model"

multi_model_workflow.add_conditional_edges(
    "task_classifier",
    route_by_task_type,
    {
        "analysis_model": "analysis_model",
        "creative_model": "creative_model",
        "general_model": "general_model"
    }
)

# All model nodes go to response combiner
multi_model_workflow.add_edge("analysis_model", "response_combiner")
multi_model_workflow.add_edge("creative_model", "response_combiner")
multi_model_workflow.add_edge("general_model", "response_combiner")
multi_model_workflow.add_edge("response_combiner", END)

multi_model_app = multi_model_workflow.compile()

print("✅ Multi-model workflow created!")
print("🧠 This workflow uses different models for different task types")


In [ ]:
# Test multi-model workflow with different task types
print("🧠 Testing Multi-Model Workflow:")
print("="*60)

# Test 1: Analysis task
print("Test 1: Analysis task")
result1 = multi_model_app.invoke(
    {
        "messages": [HumanMessage(content="Analyze the sales data for Q4")],
        "task_type": "",
        "analysis_result": "",
        "creative_result": "",
        "final_response": ""
    }
)
print(f"Final response: {result1['final_response']}\n")

# Test 2: Creative task
print("Test 2: Creative task")
result2 = multi_model_app.invoke(
    {
        "messages": [HumanMessage(content="Write a creative story about a robot")],
        "task_type": "",
        "analysis_result": "",
        "creative_result": "",
        "final_response": ""
    }
)
print(f"Final response: {result2['final_response']}\n")

# Test 3: General task
print("Test 3: General task")
result3 = multi_model_app.invoke(
    {
        "messages": [HumanMessage(content="What is LangGraph?")],
        "task_type": "",
        "analysis_result": "",
        "creative_result": "",
        "final_response": ""
    }
)
print(f"Final response: {result3['final_response']}\n")

print("="*60)
print("✅ Multi-model workflow demonstrated!")
print("🧠 Different models are selected based on task type")


## Example 3: Model with Tools Binding


In [ ]:
# Create tools that can be bound to models
@tool
def calculate_tool(expression: str) -> str:
    """Calculate a mathematical expression safely."""
    try:
        # Simple safe evaluation for demo
        allowed_chars = set('0123456789+-*/.() ')
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        else:
            return "Error: Invalid characters in expression"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def weather_tool(city: str) -> str:
    """Get weather information for a city."""
    # Simulate weather data
    weather_data = {
        "new york": "Sunny, 72°F",
        "london": "Cloudy, 65°F",
        "tokyo": "Rainy, 68°F",
        "paris": "Partly cloudy, 70°F"
    }
    city_lower = city.lower()
    return weather_data.get(city_lower, f"Weather data not available for {city}")

@tool
def search_tool(query: str) -> str:
    """Search for information on a topic."""
    return f"Search results for '{query}': Here are some relevant findings about the topic."

# Define state for tool-enabled model
class ToolModelState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    available_tools: list
    tool_results: list
    final_response: str

def tool_enabled_model_node(state: ToolModelState) -> ToolModelState:
    """Node that uses a model with bound tools"""
    messages = state["messages"]
    available_tools = state.get("available_tools", [])
    
    latest_message = messages[-1].content if messages else ""
    
    # Simulate tool selection and execution
    tool_results = []
    
    # Check if any tools should be used
    if "calculate" in latest_message.lower() or any(op in latest_message for op in ["+", "-", "*", "/"]):
        # Use calculate tool
        import re
        numbers = re.findall(r'\d+', latest_message)
        if len(numbers) >= 2:
            expression = f"{numbers[0]} + {numbers[1]}"
            result = calculate_tool.invoke({"expression": expression})
            tool_results.append(f"Calculator: {result}")
    
    if "weather" in latest_message.lower():
        # Use weather tool
        cities = ["new york", "london", "tokyo", "paris"]
        for city in cities:
            if city in latest_message.lower():
                result = weather_tool.invoke({"city": city})
                tool_results.append(f"Weather: {result}")
                break
    
    if "search" in latest_message.lower() or "find" in latest_message.lower():
        # Use search tool
        result = search_tool.invoke({"query": latest_message})
        tool_results.append(f"Search: {result}")
    
    # Generate response with tool results
    if tool_results:
        response = f"🤖 Tool-Enabled Model: I used tools to help with your request.\n"
        response += "\n".join(tool_results)
        response += f"\n\nRegarding your message: '{latest_message}'"
    else:
        response = f"🤖 Tool-Enabled Model: I can help with calculations, weather, and search. You said: '{latest_message}'"
    
    print(f"🧠 Tool-Enabled Model: {response}")
    
    return {
        "messages": [AIMessage(content=response)],
        "tool_results": tool_results,
        "final_response": response
    }

# Build tool-enabled workflow
tool_model_workflow = StateGraph(ToolModelState)
tool_model_workflow.add_node("tool_model", tool_enabled_model_node)
tool_model_workflow.add_edge(START, "tool_model")
tool_model_workflow.add_edge("tool_model", END)

tool_model_app = tool_model_workflow.compile()

print("✅ Tool-enabled model workflow created!")
print("🧠 This workflow uses models with bound tools for enhanced capabilities")


## Subgraphs

Subgraphs enable modular, reusable graph components that can be composed into larger workflows, promoting code organization and reusability.

### What are Subgraphs?

Subgraphs in LangGraph are self-contained graph components that can be used as building blocks for larger workflows. They provide:

- **Modularity**: Break complex workflows into smaller, manageable pieces
- **Reusability**: Use the same subgraph in multiple workflows
- **Composition**: Combine subgraphs to create complex systems
- **Maintainability**: Update subgraphs independently
- **Testing**: Test individual components in isolation

### Key Features

1. **Self-contained**: Complete workflows within subgraphs
2. **State management**: Handle their own state transitions
3. **Input/output**: Clear interfaces for data flow
4. **Composition**: Can be nested within other graphs
5. **Isolation**: Independent execution and state

### Types of Subgraphs

- **Functional subgraphs**: Perform specific tasks
- **Conditional subgraphs**: Handle different scenarios
- **Sequential subgraphs**: Execute steps in order
- **Parallel subgraphs**: Execute multiple paths simultaneously
- **Recursive subgraphs**: Call themselves for iterative processes

### Use Cases

- **Data processing pipelines**: Modular data transformation steps
- **API workflows**: Reusable API interaction patterns
- **Validation workflows**: Common validation logic
- **Notification systems**: Reusable notification patterns
- **Error handling**: Standardized error recovery

### Composition Patterns

- **Sequential composition**: Chain subgraphs in order
- **Conditional composition**: Choose subgraphs based on conditions
- **Parallel composition**: Execute multiple subgraphs simultaneously
- **Nested composition**: Subgraphs within subgraphs
- **Dynamic composition**: Select subgraphs at runtime

**Diagram:**

```
Main Workflow:
Input → Subgraph A → Subgraph B → Subgraph C → Output
         ↓           ↓           ↓
    [Validation] [Processing] [Formatting]
         ↓           ↓           ↓
    Subgraph D → Subgraph E → Subgraph F
    [Error Handling] [Retry Logic] [Logging]
```

**Key Concept:** Subgraphs transform monolithic workflows into modular, maintainable, and reusable systems.


In [ ]:
# Test tool-enabled model workflow
print("🧠 Testing Tool-Enabled Model Workflow:")
print("="*60)

# Test 1: Calculation request
print("Test 1: Calculation request")
result1 = tool_model_app.invoke(
    {
        "messages": [HumanMessage(content="Calculate 25 + 17")],
        "available_tools": ["calculate", "weather", "search"],
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result1['final_response']}\n")

# Test 2: Weather request
print("Test 2: Weather request")
result2 = tool_model_app.invoke(
    {
        "messages": [HumanMessage(content="What's the weather in New York?")],
        "available_tools": ["calculate", "weather", "search"],
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result2['final_response']}\n")

# Test 3: Search request
print("Test 3: Search request")
result3 = tool_model_app.invoke(
    {
        "messages": [HumanMessage(content="Search for information about LangGraph")],
        "available_tools": ["calculate", "weather", "search"],
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result3['final_response']}\n")

print("="*60)
print("✅ Tool-enabled model workflow demonstrated!")
print("🧠 Models can use tools to enhance their capabilities")


## Key Takeaways - Models

✅ **When to use:**
- Integrating LLMs into your workflows
- Using different models for different tasks
- Enhancing models with tools and functions
- Dynamic model selection based on context

💡 **Model Types:**
- **Chat Models**: For conversational AI (GPT-4, Claude, etc.)
- **Completion Models**: For text generation
- **Embedding Models**: For vector representations
- **Custom Models**: Your own fine-tuned models

⚠️ **Common Pitfalls:**
- Always handle API errors and rate limits
- Consider token costs and usage limits
- Validate model outputs before using them
- Test with different model configurations

---


# 7️⃣ Tools

<a id="tools"></a>

## Overview

**Tools** in LangGraph allow your workflows to interact with external systems and perform specific functions, enabling:
- **Function Calling**: Execute specific functions within your graph
- **External API Integration**: Connect to web services, databases, and other systems
- **Tool Binding**: Attach tools to models for enhanced capabilities
- **Tool Selection**: Dynamically choose which tools to use based on context

## Tool Types

1. **Custom Tools**: Functions you define for specific tasks
2. **Prebuilt Tools**: Ready-to-use tools from LangChain
3. **Tool Nodes**: Special nodes that handle tool execution
4. **Tool Conditionals**: Logic for choosing between tools

---

## Example 1: Basic Tool Definition and Usage


In [ ]:
# Define custom tools
@tool
def get_current_time() -> str:
    """Get the current time."""
    import datetime
    return f"Current time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

@tool
def calculate_area(length: float, width: float) -> str:
    """Calculate the area of a rectangle."""
    area = length * width
    return f"Area of rectangle ({length} x {width}) = {area}"

@tool
def get_weather_info(city: str) -> str:
    """Get weather information for a city."""
    # Simulate weather data
    weather_data = {
        "new york": "Sunny, 72°F, Light winds",
        "london": "Cloudy, 65°F, Moderate winds", 
        "tokyo": "Rainy, 68°F, Strong winds",
        "paris": "Partly cloudy, 70°F, Light winds"
    }
    city_lower = city.lower()
    return weather_data.get(city_lower, f"Weather data not available for {city}")

# Define state for tool usage
class ToolState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    tool_results: list[str]
    selected_tool: str

def tool_selector_node(state: ToolState) -> ToolState:
    """Node that selects which tool to use based on the message"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simple tool selection logic
    if "time" in latest_message.lower():
        selected_tool = "get_current_time"
    elif "area" in latest_message.lower() or "calculate" in latest_message.lower():
        selected_tool = "calculate_area"
    elif "weather" in latest_message.lower():
        selected_tool = "get_weather_info"
    else:
        selected_tool = "none"
    
    print(f"🔧 Tool Selector: Selected tool: {selected_tool}")
    
    return {"selected_tool": selected_tool}

def tool_executor_node(state: ToolState) -> ToolState:
    """Node that executes the selected tool"""
    selected_tool = state["selected_tool"]
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    tool_results = []
    
    if selected_tool == "get_current_time":
        result = get_current_time.invoke({})
        tool_results.append(result)
        
    elif selected_tool == "calculate_area":
        # Extract numbers from message
        import re
        numbers = re.findall(r'\d+\.?\d*', latest_message)
        if len(numbers) >= 2:
            length = float(numbers[0])
            width = float(numbers[1])
            result = calculate_area.invoke({"length": length, "width": width})
            tool_results.append(result)
        else:
            tool_results.append("Error: Need two numbers for area calculation")
            
    elif selected_tool == "get_weather_info":
        # Extract city name
        cities = ["new york", "london", "tokyo", "paris"]
        city = None
        for c in cities:
            if c in latest_message.lower():
                city = c
                break
        
        if city:
            result = get_weather_info.invoke({"city": city})
            tool_results.append(result)
        else:
            tool_results.append("Error: Please specify a city (New York, London, Tokyo, Paris)")
    
    else:
        tool_results.append("No suitable tool found for this request")
    
    print(f"🔧 Tool Executor: Results: {tool_results}")
    
    return {
        "messages": [AIMessage(content=f"Tool Result: {tool_results[0]}")],
        "tool_results": tool_results
    }

# Build tool workflow
tool_workflow = StateGraph(ToolState)
tool_workflow.add_node("tool_selector", tool_selector_node)
tool_workflow.add_node("tool_executor", tool_executor_node)

tool_workflow.add_edge(START, "tool_selector")
tool_workflow.add_edge("tool_selector", "tool_executor")
tool_workflow.add_edge("tool_executor", END)

tool_app = tool_workflow.compile()

print("✅ Tool workflow created!")
print("🔧 This workflow can select and execute different tools based on user input")


In [ ]:
# Test tool workflow with different requests
print("🔧 Testing Tool Workflow:")
print("="*60)

# Test 1: Time request
print("Test 1: Time request")
result1 = tool_app.invoke(
    {
        "messages": [HumanMessage(content="What time is it?")],
        "tool_results": [],
        "selected_tool": ""
    }
)
print(f"Response: {result1['messages'][-1].content}\n")

# Test 2: Area calculation
print("Test 2: Area calculation")
result2 = tool_app.invoke(
    {
        "messages": [HumanMessage(content="Calculate the area of a rectangle with length 5 and width 3")],
        "tool_results": [],
        "selected_tool": ""
    }
)
print(f"Response: {result2['messages'][-1].content}\n")

# Test 3: Weather request
print("Test 3: Weather request")
result3 = tool_app.invoke(
    {
        "messages": [HumanMessage(content="What's the weather in Tokyo?")],
        "tool_results": [],
        "selected_tool": ""
    }
)
print(f"Response: {result3['messages'][-1].content}\n")

# Test 4: No matching tool
print("Test 4: No matching tool")
result4 = tool_app.invoke(
    {
        "messages": [HumanMessage(content="Tell me a joke")],
        "tool_results": [],
        "selected_tool": ""
    }
)
print(f"Response: {result4['messages'][-1].content}\n")

print("="*60)
print("✅ Tool workflow demonstrated!")
print("🔧 Different tools are selected and executed based on user input")


## Example 2: Tool Node with Prebuilt Tools


In [ ]:
# Create a workflow using ToolNode for automatic tool execution
class ToolNodeState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def tool_router_node(state: ToolNodeState) -> ToolNodeState:
    """Node that decides whether to use tools or respond directly"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Check if the message requires tool usage
    tool_keywords = ["calculate", "time", "weather", "area", "search"]
    needs_tools = any(keyword in latest_message.lower() for keyword in tool_keywords)
    
    if needs_tools:
        # Add a message indicating tools should be used
        tool_message = AIMessage(content="I'll use tools to help with your request.")
        return {"messages": [tool_message]}
    else:
        # Respond directly without tools
        response = AIMessage(content=f"I understand you said: '{latest_message}'. How can I help you further?")
        return {"messages": [response]}

# Create a ToolNode that can execute multiple tools
tool_node = ToolNode([get_current_time, calculate_area, get_weather_info])

# Build workflow with ToolNode
tool_node_workflow = StateGraph(ToolNodeState)
tool_node_workflow.add_node("tool_router", tool_router_node)
tool_node_workflow.add_node("tools", tool_node)

tool_node_workflow.add_edge(START, "tool_router")

# Conditional edge to decide whether to use tools
def should_use_tools(state: ToolNodeState) -> str:
    """Decide whether to use tools or end"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    tool_keywords = ["calculate", "time", "weather", "area", "search"]
    needs_tools = any(keyword in latest_message.lower() for keyword in tool_keywords)
    
    return "tools" if needs_tools else "end"

tool_node_workflow.add_conditional_edges(
    "tool_router",
    should_use_tools,
    {
        "tools": "tools",
        "end": END
    }
)

tool_node_workflow.add_edge("tools", END)

tool_node_app = tool_node_workflow.compile()

print("✅ ToolNode workflow created!")
print("🔧 This workflow uses ToolNode for automatic tool execution")


## Multi-Agent

Multi-Agent systems enable multiple specialized agents to work together, coordinate, and collaborate on complex tasks.

### What are Multi-Agent Systems?

Multi-Agent systems in LangGraph enable multiple specialized agents to work together, coordinate, and collaborate on complex tasks. This provides:

- **Agent specialization**: Different agents with specific expertise
- **Supervisor pattern**: A coordinator agent that delegates tasks
- **Agent collaboration**: Agents working together on shared goals
- **Hierarchical organization**: Structured agent relationships
- **Task distribution**: Efficient workload management

### Key Features

1. **Agent specialization**: Each agent has specific capabilities
2. **Coordination**: Agents communicate and coordinate actions
3. **Task delegation**: Supervisor agents assign tasks to specialists
4. **Collaboration**: Agents work together on complex problems
5. **Hierarchical structure**: Organized agent relationships

### Agent Patterns

- **Supervisor pattern**: One agent coordinates others
- **Collaborative pattern**: Agents work together equally
- **Hierarchical pattern**: Multi-level agent organization
- **Peer-to-peer pattern**: Agents communicate directly
- **Pipeline pattern**: Agents process tasks sequentially

### Use Cases

- **Research systems**: Multiple experts collaborating on research
- **Customer service**: Specialized agents for different issues
- **Content creation**: Writers, editors, and reviewers working together
- **Data analysis**: Statisticians, domain experts, and visualization specialists
- **Software development**: Developers, testers, and reviewers

### Coordination Mechanisms

- **Task assignment**: Supervisor assigns tasks to appropriate agents
- **Information sharing**: Agents share knowledge and results
- **Conflict resolution**: Handle disagreements between agents
- **Progress tracking**: Monitor agent performance and progress
- **Quality control**: Ensure output quality across agents

**Diagram:**

```
Supervisor Agent
    ↓
Task Assignment
    ↓
┌─────────────┬─────────────┬─────────────┐
│   Agent A   │   Agent B   │   Agent C   │
│ (Research)  │ (Analysis)  │ (Writing)   │
└─────────────┴─────────────┴─────────────┘
    ↓           ↓           ↓
    Results → Coordination → Final Output
```

**Key Concept:** Multi-Agent systems transform single-agent workflows into collaborative, specialized, and efficient team-based processes.


In [ ]:
# Test ToolNode workflow
print("🔧 Testing ToolNode Workflow:")
print("="*60)

# Test 1: Request that needs tools
print("Test 1: Request that needs tools")
result1 = tool_node_app.invoke(
    {
        "messages": [HumanMessage(content="What time is it now?")]
    }
)
print(f"Response: {result1['messages'][-1].content}\n")

# Test 2: Request that doesn't need tools
print("Test 2: Request that doesn't need tools")
result2 = tool_node_app.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")]
    }
)
print(f"Response: {result2['messages'][-1].content}\n")

# Test 3: Another tool request
print("Test 3: Another tool request")
result3 = tool_node_app.invoke(
    {
        "messages": [HumanMessage(content="Calculate the area of a 10x5 rectangle")]
    }
)
print(f"Response: {result3['messages'][-1].content}\n")

print("="*60)
print("✅ ToolNode workflow demonstrated!")
print("🔧 ToolNode automatically handles tool execution when needed")


## Example 3: Advanced Tool Integration with Conditional Logic


In [ ]:
# Create advanced tools for different domains
@tool
def database_query(query: str) -> str:
    """Query a simulated database."""
    # Simulate database results
    if "users" in query.lower():
        return "Database Result: Found 150 users in the system"
    elif "orders" in query.lower():
        return "Database Result: Found 89 orders from last month"
    else:
        return "Database Result: Query executed successfully"

@tool
def api_call(endpoint: str) -> str:
    """Make an API call to a simulated service."""
    # Simulate API responses
    if "users" in endpoint.lower():
        return "API Response: User data retrieved successfully"
    elif "analytics" in endpoint.lower():
        return "API Response: Analytics data updated"
    else:
        return "API Response: Request processed"

@tool
def file_operation(operation: str, filename: str) -> str:
    """Perform file operations."""
    if operation.lower() == "read":
        return f"File Operation: Read {filename} - Content: Sample file content"
    elif operation.lower() == "write":
        return f"File Operation: Write {filename} - Success: File created"
    else:
        return f"File Operation: {operation} on {filename} - Status: Completed"

# Define state for advanced tool workflow
class AdvancedToolState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    tool_category: str
    tool_results: list[str]
    final_response: str

def tool_classifier_node(state: AdvancedToolState) -> AdvancedToolState:
    """Node that classifies the type of tool needed"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Classify tool category
    if any(word in latest_message.lower() for word in ["database", "query", "sql", "data"]):
        tool_category = "database"
    elif any(word in latest_message.lower() for word in ["api", "service", "endpoint", "request"]):
        tool_category = "api"
    elif any(word in latest_message.lower() for word in ["file", "read", "write", "save"]):
        tool_category = "file"
    else:
        tool_category = "none"
    
    print(f"🔧 Tool Classifier: Category: {tool_category}")
    
    return {"tool_category": tool_category}

def database_tool_node(state: AdvancedToolState) -> AdvancedToolState:
    """Node that executes database tools"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    result = database_query.invoke({"query": latest_message})
    
    print(f"🔧 Database Tool: {result}")
    
    return {
        "tool_results": [result],
        "final_response": f"Database operation completed: {result}"
    }

def api_tool_node(state: AdvancedToolState) -> AdvancedToolState:
    """Node that executes API tools"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    result = api_call.invoke({"endpoint": latest_message})
    
    print(f"🔧 API Tool: {result}")
    
    return {
        "tool_results": [result],
        "final_response": f"API call completed: {result}"
    }

def file_tool_node(state: AdvancedToolState) -> AdvancedToolState:
    """Node that executes file tools"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Extract operation and filename
    operation = "read"  # Default operation
    filename = "sample.txt"  # Default filename
    
    if "write" in latest_message.lower():
        operation = "write"
    if "save" in latest_message.lower():
        operation = "write"
    
    result = file_operation.invoke({"operation": operation, "filename": filename})
    
    print(f"🔧 File Tool: {result}")
    
    return {
        "tool_results": [result],
        "final_response": f"File operation completed: {result}"
    }

def no_tool_node(state: AdvancedToolState) -> AdvancedToolState:
    """Node for requests that don't need tools"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    response = f"I understand your request: '{latest_message}'. However, I don't have a specific tool for this task."
    
    print(f"🔧 No Tool: {response}")
    
    return {
        "final_response": response
    }

# Build advanced tool workflow
advanced_tool_workflow = StateGraph(AdvancedToolState)

# Add nodes
advanced_tool_workflow.add_node("tool_classifier", tool_classifier_node)
advanced_tool_workflow.add_node("database_tool", database_tool_node)
advanced_tool_workflow.add_node("api_tool", api_tool_node)
advanced_tool_workflow.add_node("file_tool", file_tool_node)
advanced_tool_workflow.add_node("no_tool", no_tool_node)

# Add edges
advanced_tool_workflow.add_edge(START, "tool_classifier")

# Conditional routing based on tool category
def route_by_tool_category(state: AdvancedToolState) -> str:
    tool_category = state["tool_category"]
    return tool_category

advanced_tool_workflow.add_conditional_edges(
    "tool_classifier",
    route_by_tool_category,
    {
        "database": "database_tool",
        "api": "api_tool",
        "file": "file_tool",
        "none": "no_tool"
    }
)

# All tool nodes end the workflow
advanced_tool_workflow.add_edge("database_tool", END)
advanced_tool_workflow.add_edge("api_tool", END)
advanced_tool_workflow.add_edge("file_tool", END)
advanced_tool_workflow.add_edge("no_tool", END)

advanced_tool_app = advanced_tool_workflow.compile()

print("✅ Advanced tool workflow created!")
print("🔧 This workflow classifies and routes to different tool categories")


In [ ]:
# Test advanced tool workflow
print("🔧 Testing Advanced Tool Workflow:")
print("="*60)

# Test 1: Database tool
print("Test 1: Database tool")
result1 = advanced_tool_app.invoke(
    {
        "messages": [HumanMessage(content="Query the database for all users")],
        "tool_category": "",
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result1['final_response']}\n")

# Test 2: API tool
print("Test 2: API tool")
result2 = advanced_tool_app.invoke(
    {
        "messages": [HumanMessage(content="Call the analytics API endpoint")],
        "tool_category": "",
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result2['final_response']}\n")

# Test 3: File tool
print("Test 3: File tool")
result3 = advanced_tool_app.invoke(
    {
        "messages": [HumanMessage(content="Write data to a file")],
        "tool_category": "",
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result3['final_response']}\n")

# Test 4: No tool needed
print("Test 4: No tool needed")
result4 = advanced_tool_app.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "tool_category": "",
        "tool_results": [],
        "final_response": ""
    }
)
print(f"Response: {result4['final_response']}\n")

print("="*60)
print("✅ Advanced tool workflow demonstrated!")
print("🔧 Different tool categories are selected and executed based on user input")


## Key Takeaways - Tools

✅ **When to use:**
- Integrating external systems and APIs
- Performing specific calculations or operations
- Accessing databases or file systems
- Enhancing model capabilities with function calling

💡 **Tool Types:**
- **Custom Tools**: Functions you define for specific tasks
- **Prebuilt Tools**: Ready-to-use tools from LangChain
- **Tool Nodes**: Special nodes that handle tool execution
- **Tool Conditionals**: Logic for choosing between tools

⚠️ **Common Pitfalls:**
- Always validate tool inputs and handle errors
- Consider tool execution time and rate limits
- Test tools thoroughly before production use
- Use appropriate tool selection logic

---


# 8️⃣ Human-in-the-Loop

<a id="human-in-the-loop"></a>

## Overview

**Human-in-the-Loop** in LangGraph allows you to pause workflow execution and wait for human input, enabling:
- **Approval Workflows**: Require human approval before proceeding
- **Content Moderation**: Human review of AI-generated content
- **Quality Control**: Human validation of results
- **Interactive Processes**: Dynamic workflow modification based on human input

## Key Features

- **Interrupts**: Pause execution at specific nodes
- **Human Input**: Collect input from users during execution
- **State Updates**: Modify workflow state based on human input
- **Resume**: Continue execution after human interaction

---

## Example 1: Basic Human Approval Workflow


In [ ]:
# Define state for human-in-the-loop workflow
class HumanLoopState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    content: str
    human_approval: str
    status: str
    final_result: str

def content_generator_node(state: HumanLoopState) -> HumanLoopState:
    """Node that generates content requiring human approval"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Generate content based on user request
    content = f"Generated content for: '{latest_message}'\n\n"
    content += "This is AI-generated content that requires human approval before proceeding.\n"
    content += "Please review and approve or reject this content."
    
    print(f"🤖 Content Generator: {content}")
    
    return {
        "content": content,
        "status": "pending_approval"
    }

def human_approval_node(state: HumanLoopState) -> HumanLoopState:
    """Node that waits for human approval"""
    content = state["content"]
    
    # In a real implementation, this would pause and wait for human input
    # For demo purposes, we'll simulate the approval process
    print(f"👤 Human Approval Required:")
    print(f"Content to review: {content}")
    print("Waiting for human approval...")
    
    # Simulate human approval (in real usage, this would be interactive)
    human_approval = "approved"  # Could be "approved" or "rejected"
    
    print(f"👤 Human Decision: {human_approval}")
    
    return {
        "human_approval": human_approval,
        "status": "approved" if human_approval == "approved" else "rejected"
    }

def content_processor_node(state: HumanLoopState) -> HumanLoopState:
    """Node that processes approved content"""
    content = state["content"]
    
    # Process the approved content
    processed_content = f"✅ PROCESSED: {content}\n\n"
    processed_content += "Content has been successfully processed and is ready for use."
    
    print(f"⚙️ Content Processor: {processed_content}")
    
    return {
        "final_result": processed_content,
        "status": "completed"
    }

def rejection_handler_node(state: HumanLoopState) -> HumanLoopState:
    """Node that handles rejected content"""
    content = state["content"]
    
    rejection_message = f"❌ REJECTED: {content}\n\n"
    rejection_message += "Content was rejected by human reviewer. Please modify your request."
    
    print(f"🚫 Rejection Handler: {rejection_message}")
    
    return {
        "final_result": rejection_message,
        "status": "rejected"
    }

# Build human-in-the-loop workflow
human_loop_workflow = StateGraph(HumanLoopState)

# Add nodes
human_loop_workflow.add_node("content_generator", content_generator_node)
human_loop_workflow.add_node("human_approval", human_approval_node)
human_loop_workflow.add_node("content_processor", content_processor_node)
human_loop_workflow.add_node("rejection_handler", rejection_handler_node)

# Add edges
human_loop_workflow.add_edge(START, "content_generator")
human_loop_workflow.add_edge("content_generator", "human_approval")

# Conditional routing based on human approval
def route_by_approval(state: HumanLoopState) -> str:
    human_approval = state.get("human_approval", "")
    if human_approval == "approved":
        return "content_processor"
    else:
        return "rejection_handler"

human_loop_workflow.add_conditional_edges(
    "human_approval",
    route_by_approval,
    {
        "content_processor": "content_processor",
        "rejection_handler": "rejection_handler"
    }
)

# Both end nodes complete the workflow
human_loop_workflow.add_edge("content_processor", END)
human_loop_workflow.add_edge("rejection_handler", END)

human_loop_app = human_loop_workflow.compile()

print("✅ Human-in-the-loop workflow created!")
print("👤 This workflow requires human approval before processing content")


## MCP Integration

MCP Integration enables connecting to external systems and resources through Model Context Protocol (MCP) servers.

### What is MCP Integration?

MCP Integration in LangGraph enables connecting to external systems and resources through Model Context Protocol (MCP) servers. This provides:

- **External resource access**: Connect to databases, APIs, and external services
- **Tool integration**: Use MCP tools within graph workflows
- **Dynamic resource discovery**: Discover and use available MCP resources
- **Protocol compliance**: Standardized communication with external systems
- **Interoperability**: Connect with various external systems

### Key Features

1. **Server connection**: Connect to MCP servers
2. **Resource discovery**: Find available tools and resources
3. **Tool execution**: Execute MCP tools within workflows
4. **Error handling**: Robust error handling and recovery
5. **Configuration management**: Manage MCP server configurations

### MCP Components

- **MCP servers**: External systems that provide tools and resources
- **MCP tools**: Functions available through MCP servers
- **MCP resources**: Data and capabilities exposed by servers
- **MCP clients**: LangGraph workflows that connect to servers
- **MCP protocol**: Standardized communication protocol

### Use Cases

- **Database integration**: Connect to various database systems
- **API integration**: Access external web services
- **File system access**: Read and write files
- **Cloud services**: Connect to cloud platforms
- **Legacy systems**: Integrate with existing systems

### Integration Patterns

- **Direct connection**: Connect directly to MCP servers
- **Resource discovery**: Dynamically discover available resources
- **Tool chaining**: Chain multiple MCP tools together
- **Error recovery**: Handle failures and retry operations
- **Configuration management**: Manage server configurations

**Diagram:**

```
LangGraph Workflow
    ↓
MCP Client
    ↓
┌─────────────┬─────────────┬─────────────┐
│ MCP Server A│ MCP Server B│ MCP Server C│
│ (Database)  │ (File System)│ (API)      │
└─────────────┴─────────────┴─────────────┘
    ↓           ↓           ↓
    Tools → Resource Discovery → Execution
```

**Key Concept:** MCP Integration transforms isolated workflows into connected systems that can access and utilize external resources and capabilities.


In [ ]:
# Test human-in-the-loop workflow
print("👤 Testing Human-in-the-Loop Workflow:")
print("="*60)

# Test 1: Approved content
print("Test 1: Approved content")
result1 = human_loop_app.invoke(
    {
        "messages": [HumanMessage(content="Generate a marketing email")],
        "content": "",
        "human_approval": "",
        "status": "",
        "final_result": ""
    }
)
print(f"Final result: {result1['final_result']}\n")

# Test 2: Rejected content (simulate by changing the approval logic)
print("Test 2: Rejected content")
# For demo, we'll modify the approval node to simulate rejection
def human_approval_node_reject(state: HumanLoopState) -> HumanLoopState:
    """Node that simulates human rejection"""
    content = state["content"]
    
    print(f"👤 Human Approval Required:")
    print(f"Content to review: {content}")
    print("Waiting for human approval...")
    
    # Simulate human rejection
    human_approval = "rejected"
    
    print(f"👤 Human Decision: {human_approval}")
    
    return {
        "human_approval": human_approval,
        "status": "rejected"
    }

# Update the workflow with the rejection node
human_loop_workflow.add_node("human_approval_reject", human_approval_node_reject)
human_loop_workflow.add_edge("content_generator", "human_approval_reject")

human_loop_workflow.add_conditional_edges(
    "human_approval_reject",
    route_by_approval,
    {
        "content_processor": "content_processor",
        "rejection_handler": "rejection_handler"
    }
)

human_loop_app_reject = human_loop_workflow.compile()

result2 = human_loop_app_reject.invoke(
    {
        "messages": [HumanMessage(content="Generate a controversial statement")],
        "content": "",
        "human_approval": "",
        "status": "",
        "final_result": ""
    }
)
print(f"Final result: {result2['final_result']}\n")

print("="*60)
print("✅ Human-in-the-loop workflow demonstrated!")
print("👤 Workflow pauses for human approval and routes based on decision")


## Example 2: Content Moderation with Human Review


In [ ]:
# Define state for content moderation workflow
class ModerationState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    generated_content: str
    moderation_score: float
    human_review: str
    moderation_decision: str
    final_content: str

def content_generator_node(state: ModerationState) -> ModerationState:
    """Node that generates content"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Generate content based on user request
    generated_content = f"Generated response to: '{latest_message}'\n\n"
    generated_content += "This is AI-generated content that will be moderated before delivery."
    
    print(f"🤖 Content Generator: {generated_content}")
    
    return {"generated_content": generated_content}

def auto_moderation_node(state: ModerationState) -> ModerationState:
    """Node that performs automatic content moderation"""
    generated_content = state["generated_content"]
    
    # Simple automatic moderation scoring
    risky_keywords = ["violence", "hate", "inappropriate", "dangerous"]
    content_lower = generated_content.lower()
    
    risk_score = 0.0
    for keyword in risky_keywords:
        if keyword in content_lower:
            risk_score += 0.3
    
    # Cap the score at 1.0
    moderation_score = min(risk_score, 1.0)
    
    print(f"🤖 Auto Moderation: Risk score: {moderation_score}")
    
    return {"moderation_score": moderation_score}

def human_review_node(state: ModerationState) -> ModerationState:
    """Node that requires human review for high-risk content"""
    generated_content = state["generated_content"]
    moderation_score = state["moderation_score"]
    
    if moderation_score > 0.5:
        print(f"👤 Human Review Required:")
        print(f"Content: {generated_content}")
        print(f"Risk Score: {moderation_score}")
        print("Waiting for human moderation decision...")
        
        # Simulate human review decision
        human_review = "approved"  # Could be "approved", "rejected", or "modified"
        
        print(f"👤 Human Decision: {human_review}")
        
        return {"human_review": human_review}
    else:
        print(f"✅ Auto-approved: Risk score {moderation_score} is below threshold")
        return {"human_review": "auto_approved"}

def content_delivery_node(state: ModerationState) -> ModerationState:
    """Node that delivers approved content"""
    generated_content = state["generated_content"]
    human_review = state.get("human_review", "")
    
    if human_review in ["approved", "auto_approved"]:
        final_content = f"✅ DELIVERED: {generated_content}"
        moderation_decision = "approved"
    else:
        final_content = "❌ Content was not approved for delivery"
        moderation_decision = "rejected"
    
    print(f"📤 Content Delivery: {final_content}")
    
    return {
        "final_content": final_content,
        "moderation_decision": moderation_decision
    }

# Build content moderation workflow
moderation_workflow = StateGraph(ModerationState)

# Add nodes
moderation_workflow.add_node("content_generator", content_generator_node)
moderation_workflow.add_node("auto_moderation", auto_moderation_node)
moderation_workflow.add_node("human_review", human_review_node)
moderation_workflow.add_node("content_delivery", content_delivery_node)

# Add edges
moderation_workflow.add_edge(START, "content_generator")
moderation_workflow.add_edge("content_generator", "auto_moderation")
moderation_workflow.add_edge("auto_moderation", "human_review")
moderation_workflow.add_edge("human_review", "content_delivery")
moderation_workflow.add_edge("content_delivery", END)

moderation_app = moderation_workflow.compile()

print("✅ Content moderation workflow created!")
print("👤 This workflow automatically moderates content and requires human review for high-risk content")


In [ ]:
# Test content moderation workflow
print("👤 Testing Content Moderation Workflow:")
print("="*60)

# Test 1: Low-risk content (auto-approved)
print("Test 1: Low-risk content")
result1 = moderation_app.invoke(
    {
        "messages": [HumanMessage(content="Write a friendly greeting")],
        "generated_content": "",
        "moderation_score": 0.0,
        "human_review": "",
        "moderation_decision": "",
        "final_content": ""
    }
)
print(f"Final content: {result1['final_content']}\n")

# Test 2: High-risk content (requires human review)
print("Test 2: High-risk content")
result2 = moderation_app.invoke(
    {
        "messages": [HumanMessage(content="Write about violence and hate")],
        "generated_content": "",
        "moderation_score": 0.0,
        "human_review": "",
        "moderation_decision": "",
        "final_content": ""
    }
)
print(f"Final content: {result2['final_content']}\n")

# Test 3: Medium-risk content
print("Test 3: Medium-risk content")
result3 = moderation_app.invoke(
    {
        "messages": [HumanMessage(content="Write about inappropriate behavior")],
        "generated_content": "",
        "moderation_score": 0.0,
        "human_review": "",
        "moderation_decision": "",
        "final_content": ""
    }
)
print(f"Final content: {result3['final_content']}\n")

print("="*60)
print("✅ Content moderation workflow demonstrated!")
print("👤 Content is automatically moderated with human review for high-risk items")


## Example 3: Interactive Workflow with Human Input


In [ ]:
# Define state for interactive workflow
class InteractiveState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    current_step: int
    human_input: str
    workflow_data: dict
    next_action: str

def workflow_initiator_node(state: InteractiveState) -> InteractiveState:
    """Node that initiates the interactive workflow"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    print(f"🚀 Workflow Initiator: Starting interactive workflow for: '{latest_message}'")
    
    return {
        "current_step": 1,
        "workflow_data": {"original_request": latest_message},
        "next_action": "collect_input"
    }

def input_collector_node(state: InteractiveState) -> InteractiveState:
    """Node that collects human input"""
    current_step = state["current_step"]
    workflow_data = state["workflow_data"]
    
    print(f"👤 Input Collector (Step {current_step}):")
    print("Please provide additional information to continue the workflow.")
    
    # Simulate human input collection
    if current_step == 1:
        human_input = "I need detailed analysis with charts"
    elif current_step == 2:
        human_input = "Focus on Q4 performance metrics"
    else:
        human_input = "Generate final report"
    
    print(f"👤 Human Input: {human_input}")
    
    # Update workflow data with human input
    workflow_data[f"step_{current_step}_input"] = human_input
    
    return {
        "human_input": human_input,
        "workflow_data": workflow_data,
        "next_action": "process_input"
    }

def input_processor_node(state: InteractiveState) -> InteractiveState:
    """Node that processes human input"""
    human_input = state["human_input"]
    current_step = state["current_step"]
    workflow_data = state["workflow_data"]
    
    print(f"⚙️ Input Processor: Processing input from step {current_step}")
    
    # Process the input and determine next action
    if "detailed analysis" in human_input.lower():
        next_action = "detailed_analysis"
    elif "performance metrics" in human_input.lower():
        next_action = "performance_analysis"
    elif "final report" in human_input.lower():
        next_action = "generate_report"
    else:
        next_action = "collect_more_input"
    
    print(f"⚙️ Next Action: {next_action}")
    
    return {
        "next_action": next_action,
        "workflow_data": workflow_data
    }

def detailed_analysis_node(state: InteractiveState) -> InteractiveState:
    """Node that performs detailed analysis"""
    workflow_data = state["workflow_data"]
    
    analysis_result = "📊 Detailed Analysis Complete:\n"
    analysis_result += "- Data processed and analyzed\n"
    analysis_result += "- Charts and visualizations created\n"
    analysis_result += "- Insights extracted\n"
    
    workflow_data["analysis_result"] = analysis_result
    
    print(f"📊 Detailed Analysis: {analysis_result}")
    
    return {
        "workflow_data": workflow_data,
        "next_action": "collect_input"
    }

def performance_analysis_node(state: InteractiveState) -> InteractiveState:
    """Node that performs performance analysis"""
    workflow_data = state["workflow_data"]
    
    performance_result = "📈 Performance Analysis Complete:\n"
    performance_result += "- Q4 metrics analyzed\n"
    performance_result += "- Performance trends identified\n"
    performance_result += "- Recommendations generated\n"
    
    workflow_data["performance_result"] = performance_result
    
    print(f"📈 Performance Analysis: {performance_result}")
    
    return {
        "workflow_data": workflow_data,
        "next_action": "collect_input"
    }

def report_generator_node(state: InteractiveState) -> InteractiveState:
    """Node that generates the final report"""
    workflow_data = state["workflow_data"]
    
    final_report = "📋 Final Report Generated:\n"
    final_report += f"Original Request: {workflow_data.get('original_request', 'N/A')}\n"
    final_report += f"Step 1 Input: {workflow_data.get('step_1_input', 'N/A')}\n"
    final_report += f"Step 2 Input: {workflow_data.get('step_2_input', 'N/A')}\n"
    
    if "analysis_result" in workflow_data:
        final_report += f"Analysis: {workflow_data['analysis_result']}\n"
    
    if "performance_result" in workflow_data:
        final_report += f"Performance: {workflow_data['performance_result']}\n"
    
    final_report += "\n✅ Report completed successfully!"
    
    print(f"📋 Report Generator: {final_report}")
    
    return {
        "workflow_data": workflow_data,
        "next_action": "complete"
    }

def step_incrementer_node(state: InteractiveState) -> InteractiveState:
    """Node that increments the workflow step"""
    current_step = state["current_step"]
    next_action = state["next_action"]
    
    if next_action == "collect_more_input":
        new_step = current_step + 1
        print(f"🔄 Step Incrementer: Moving to step {new_step}")
        return {
            "current_step": new_step,
            "next_action": "collect_input"
        }
    else:
        return {"next_action": next_action}

# Build interactive workflow
interactive_workflow = StateGraph(InteractiveState)

# Add nodes
interactive_workflow.add_node("workflow_initiator", workflow_initiator_node)
interactive_workflow.add_node("input_collector", input_collector_node)
interactive_workflow.add_node("input_processor", input_processor_node)
interactive_workflow.add_node("detailed_analysis", detailed_analysis_node)
interactive_workflow.add_node("performance_analysis", performance_analysis_node)
interactive_workflow.add_node("report_generator", report_generator_node)
interactive_workflow.add_node("step_incrementer", step_incrementer_node)

# Add edges
interactive_workflow.add_edge(START, "workflow_initiator")
interactive_workflow.add_edge("workflow_initiator", "input_collector")
interactive_workflow.add_edge("input_collector", "input_processor")

# Conditional routing based on next action
def route_by_next_action(state: InteractiveState) -> str:
    next_action = state["next_action"]
    return next_action

interactive_workflow.add_conditional_edges(
    "input_processor",
    route_by_next_action,
    {
        "detailed_analysis": "detailed_analysis",
        "performance_analysis": "performance_analysis",
        "generate_report": "report_generator",
        "collect_more_input": "step_incrementer"
    }
)

# Step incrementer routes back to input collector
interactive_workflow.add_edge("step_incrementer", "input_collector")

# Analysis nodes route back to input collector
interactive_workflow.add_edge("detailed_analysis", "input_collector")
interactive_workflow.add_edge("performance_analysis", "input_collector")

# Report generator ends the workflow
interactive_workflow.add_edge("report_generator", END)

interactive_app = interactive_workflow.compile()

print("✅ Interactive workflow created!")
print("👤 This workflow collects human input at multiple steps and adapts accordingly")


In [ ]:
# Test interactive workflow
print("👤 Testing Interactive Workflow:")
print("="*60)

# Test the interactive workflow
result = interactive_app.invoke(
    {
        "messages": [HumanMessage(content="I need a business analysis report")],
        "current_step": 0,
        "human_input": "",
        "workflow_data": {},
        "next_action": ""
    }
)

print("="*60)
print("✅ Interactive workflow demonstrated!")
print("👤 Workflow adapts based on human input at each step")
print(f"📋 Final workflow data: {result['workflow_data']}")


## Evaluation

Evaluation enables comprehensive assessment of graph performance, quality, and behavior through integration with LangSmith and custom metrics.

### What is Evaluation?

Evaluation in LangGraph enables comprehensive assessment of graph performance, quality, and behavior through integration with LangSmith and custom metrics. This provides:

- **Performance monitoring**: Track execution time, resource usage, and throughput
- **Quality assessment**: Evaluate response quality, accuracy, and relevance
- **Debugging support**: Trace execution paths and identify bottlenecks
- **Continuous improvement**: Monitor system performance over time
- **Compliance tracking**: Ensure system meets quality standards

### Key Features

1. **LangSmith integration**: Comprehensive tracing and monitoring
2. **Custom metrics**: Define and track specific performance indicators
3. **Quality assessment**: Evaluate response quality and accuracy
4. **Performance analysis**: Monitor execution time and resource usage
5. **Alerting**: Notify when performance thresholds are exceeded

### Evaluation Types

- **Performance evaluation**: Execution time, resource usage, throughput
- **Quality evaluation**: Response accuracy, relevance, completeness
- **Behavioral evaluation**: System behavior and decision patterns
- **Comparative evaluation**: Compare different system versions
- **Continuous evaluation**: Ongoing monitoring and assessment

### Metrics and KPIs

- **Execution metrics**: Time, memory, CPU usage
- **Quality metrics**: Accuracy, relevance, completeness scores
- **User metrics**: Satisfaction, engagement, conversion rates
- **System metrics**: Availability, reliability, error rates
- **Business metrics**: Cost, efficiency, ROI

### Use Cases

- **System monitoring**: Track system health and performance
- **Quality assurance**: Ensure output quality meets standards
- **Performance optimization**: Identify and fix bottlenecks
- **A/B testing**: Compare different system configurations
- **Compliance monitoring**: Ensure regulatory compliance

**Diagram:**

```
Workflow Execution
    ↓
Evaluation Framework
    ↓
┌─────────────┬─────────────┬─────────────┐
│ Performance │   Quality  │  Behavioral │
│ Monitoring  │ Assessment │  Analysis   │
└─────────────┴─────────────┴─────────────┘
    ↓           ↓           ↓
    Metrics → Analysis → Recommendations
```

**Key Concept:** Evaluation transforms ad-hoc systems into monitored, measurable, and continuously improving workflows.


## Key Takeaways - Human-in-the-Loop

✅ **When to use:**
- Content that requires human approval or review
- Quality control and validation processes
- Interactive workflows that need human input
- Compliance and safety-critical applications

💡 **Key Features:**
- **Interrupts**: Pause execution at specific nodes
- **Human Input**: Collect input from users during execution
- **State Updates**: Modify workflow state based on human input
- **Resume**: Continue execution after human interaction

⚠️ **Common Pitfalls:**
- Design clear interfaces for human input
- Provide context for human decision-making
- Handle cases where humans don't respond
- Consider workflow timeout and fallback strategies

---


# 9️⃣ Time Travel

<a id="time-travel"></a>

## Overview

**Time Travel** in LangGraph allows you to access and manipulate the execution history of your workflows, enabling:
- **State Inspection**: View state at any checkpoint
- **Execution Rewinding**: Go back to previous states
- **Forking**: Create alternative execution paths from past states
- **Debugging**: Analyze workflow execution history

## Key Features

- **Checkpoint Access**: Get state at any previous checkpoint
- **State Rewinding**: Revert to previous states
- **Execution Forking**: Branch from past states
- **History Navigation**: Traverse execution timeline

---

## Example 1: Basic State Inspection


In [ ]:
# Define state for time travel workflow
class TimeTravelState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    step: int
    data: str
    checkpoint_id: str

def step1_node(state: TimeTravelState) -> TimeTravelState:
    """First step in the workflow"""
    step = state.get("step", 0) + 1
    data = f"Step {step} processed"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "checkpoint_id": f"checkpoint_{step}"
    }

def step2_node(state: TimeTravelState) -> TimeTravelState:
    """Second step in the workflow"""
    step = state.get("step", 0) + 1
    data = state.get("data", "") + f" -> Step {step} processed"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "checkpoint_id": f"checkpoint_{step}"
    }

def step3_node(state: TimeTravelState) -> TimeTravelState:
    """Third step in the workflow"""
    step = state.get("step", 0) + 1
    data = state.get("data", "") + f" -> Step {step} processed"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "checkpoint_id": f"checkpoint_{step}"
    }

# Build time travel workflow
time_travel_workflow = StateGraph(TimeTravelState)
time_travel_workflow.add_node("step1", step1_node)
time_travel_workflow.add_node("step2", step2_node)
time_travel_workflow.add_node("step3", step3_node)

time_travel_workflow.add_edge(START, "step1")
time_travel_workflow.add_edge("step1", "step2")
time_travel_workflow.add_edge("step2", "step3")
time_travel_workflow.add_edge("step3", END)

# Compile with checkpointer for time travel capabilities
time_travel_app = time_travel_workflow.compile(checkpointer=MemorySaver())

print("✅ Time travel workflow created!")
print("⏰ This workflow creates checkpoints at each step for time travel")


In [ ]:
# Execute the workflow and inspect states
config = {"configurable": {"thread_id": "time_travel_demo"}}

print("⏰ Executing Time Travel Workflow:")
print("="*60)

# Execute the workflow
result = time_travel_app.invoke(
    {
        "messages": [HumanMessage(content="Start time travel demo")],
        "step": 0,
        "data": "",
        "checkpoint_id": ""
    },
    config=config
)

print(f"\n🎯 Final Result: {result['data']}")
print(f"📊 Final Step: {result['step']}")

# Now inspect the state at different checkpoints
print("\n🔍 Inspecting States at Different Checkpoints:")
print("="*60)

# Get current state
current_state = time_travel_app.get_state(config)
print(f"Current state: Step {current_state.values.get('step', 0)}, Data: {current_state.values.get('data', '')}")

# Get state at specific checkpoint (if available)
# Note: In a real implementation, you would use checkpoint IDs
# For demo purposes, we'll simulate checkpoint inspection
print("\n📋 Checkpoint History:")
for i in range(1, result['step'] + 1):
    checkpoint_data = f"Step {i} processed"
    if i > 1:
        checkpoint_data = " -> ".join([f"Step {j} processed" for j in range(1, i + 1)])
    print(f"  Checkpoint {i}: {checkpoint_data}")

print("="*60)
print("✅ Time travel state inspection demonstrated!")
print("⏰ You can access state at any checkpoint in the execution history")


## Example 2: Execution Rewinding


In [ ]:
# Create a workflow that can be rewound
class RewindState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    step: int
    data: str
    rewind_point: int

def rewindable_step1(state: RewindState) -> RewindState:
    """Step 1 that can be rewound"""
    step = 1
    data = "Step 1: Initial processing"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "rewind_point": step
    }

def rewindable_step2(state: RewindState) -> RewindState:
    """Step 2 that can be rewound"""
    step = 2
    data = state.get("data", "") + " -> Step 2: Data enhancement"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "rewind_point": step
    }

def rewindable_step3(state: RewindState) -> RewindState:
    """Step 3 that can be rewound"""
    step = 3
    data = state.get("data", "") + " -> Step 3: Final processing"
    
    print(f"⏰ Step {step}: {data}")
    
    return {
        "step": step,
        "data": data,
        "rewind_point": step
    }

def rewind_handler_node(state: RewindState) -> RewindState:
    """Node that handles rewinding"""
    rewind_point = state.get("rewind_point", 0)
    current_step = state.get("step", 0)
    
    if rewind_point < current_step:
        print(f"🔄 Rewinding from step {current_step} to step {rewind_point}")
        
        # Simulate rewinding by truncating data
        if rewind_point == 1:
            data = "Step 1: Initial processing"
        elif rewind_point == 2:
            data = "Step 1: Initial processing -> Step 2: Data enhancement"
        else:
            data = state.get("data", "")
        
        return {
            "step": rewind_point,
            "data": data,
            "rewind_point": rewind_point
        }
    else:
        return state

# Build rewindable workflow
rewind_workflow = StateGraph(RewindState)
rewind_workflow.add_node("step1", rewindable_step1)
rewind_workflow.add_node("step2", rewindable_step2)
rewind_workflow.add_node("step3", rewindable_step3)
rewind_workflow.add_node("rewind_handler", rewind_handler_node)

rewind_workflow.add_edge(START, "step1")
rewind_workflow.add_edge("step1", "step2")
rewind_workflow.add_edge("step2", "step3")
rewind_workflow.add_edge("step3", "rewind_handler")
rewind_workflow.add_edge("rewind_handler", END)

rewind_app = rewind_workflow.compile(checkpointer=MemorySaver())

print("✅ Rewindable workflow created!")
print("🔄 This workflow can be rewound to previous states")


In [ ]:
# Test execution rewinding
config_rewind = {"configurable": {"thread_id": "rewind_demo"}}

print("🔄 Testing Execution Rewinding:")
print("="*60)

# Execute the workflow normally
result1 = rewind_app.invoke(
    {
        "messages": [HumanMessage(content="Execute normal workflow")],
        "step": 0,
        "data": "",
        "rewind_point": 0
    },
    config=config_rewind
)

print(f"\n🎯 Normal Execution Result: {result1['data']}")

# Now simulate rewinding by modifying the state
print("\n🔄 Simulating Rewind to Step 2:")
rewind_result = rewind_app.invoke(
    {
        "messages": [HumanMessage(content="Rewind to step 2")],
        "step": 3,  # Current step
        "data": "Step 1: Initial processing -> Step 2: Data enhancement -> Step 3: Final processing",
        "rewind_point": 2  # Rewind to step 2
    },
    config=config_rewind
)

print(f"🔄 Rewind Result: {rewind_result['data']}")
print(f"📊 Rewound to Step: {rewind_result['step']}")

# Simulate rewinding to step 1
print("\n🔄 Simulating Rewind to Step 1:")
rewind_result2 = rewind_app.invoke(
    {
        "messages": [HumanMessage(content="Rewind to step 1")],
        "step": 3,
        "data": "Step 1: Initial processing -> Step 2: Data enhancement -> Step 3: Final processing",
        "rewind_point": 1
    },
    config=config_rewind
)

print(f"🔄 Rewind Result: {rewind_result2['data']}")
print(f"📊 Rewound to Step: {rewind_result2['step']}")

print("="*60)
print("✅ Execution rewinding demonstrated!")
print("🔄 Workflow can be rewound to any previous state")


## Example 3: Execution Forking


In [ ]:
# Create a workflow that supports forking from past states
class ForkState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    step: int
    data: str
    fork_point: int
    execution_path: str

def forkable_step1(state: ForkState) -> ForkState:
    """Step 1 that can be forked from"""
    step = 1
    data = "Step 1: Base processing"
    execution_path = "original"
    
    print(f"⏰ Step {step} ({execution_path}): {data}")
    
    return {
        "step": step,
        "data": data,
        "fork_point": step,
        "execution_path": execution_path
    }

def forkable_step2(state: ForkState) -> ForkState:
    """Step 2 that can be forked from"""
    step = 2
    execution_path = state.get("execution_path", "original")
    data = state.get("data", "") + f" -> Step 2: {execution_path} processing"
    
    print(f"⏰ Step {step} ({execution_path}): {data}")
    
    return {
        "step": step,
        "data": data,
        "fork_point": step,
        "execution_path": execution_path
    }

def forkable_step3(state: ForkState) -> ForkState:
    """Step 3 that can be forked from"""
    step = 3
    execution_path = state.get("execution_path", "original")
    data = state.get("data", "") + f" -> Step 3: {execution_path} completion"
    
    print(f"⏰ Step {step} ({execution_path}): {data}")
    
    return {
        "step": step,
        "data": data,
        "fork_point": step,
        "execution_path": execution_path
    }

def fork_handler_node(state: ForkState) -> ForkState:
    """Node that handles forking from past states"""
    fork_point = state.get("fork_point", 0)
    execution_path = state.get("execution_path", "original")
    
    if fork_point > 0 and execution_path != "original":
        print(f"🌿 Forking from step {fork_point} with path: {execution_path}")
        
        # Simulate forking by modifying the data
        if execution_path == "alternative":
            data = state.get("data", "").replace("original", "alternative")
        elif execution_path == "experimental":
            data = state.get("data", "").replace("original", "experimental")
        else:
            data = state.get("data", "")
        
        return {
            "data": data,
            "execution_path": execution_path
        }
    else:
        return state

# Build forkable workflow
fork_workflow = StateGraph(ForkState)
fork_workflow.add_node("step1", forkable_step1)
fork_workflow.add_node("step2", forkable_step2)
fork_workflow.add_node("step3", forkable_step3)
fork_workflow.add_node("fork_handler", fork_handler_node)

fork_workflow.add_edge(START, "step1")
fork_workflow.add_edge("step1", "step2")
fork_workflow.add_edge("step2", "step3")
fork_workflow.add_edge("step3", "fork_handler")
fork_workflow.add_edge("fork_handler", END)

fork_app = fork_workflow.compile(checkpointer=MemorySaver())

print("✅ Forkable workflow created!")
print("🌿 This workflow can fork from past states to explore alternative paths")


In [ ]:
# Test execution forking
config_fork = {"configurable": {"thread_id": "fork_demo"}}

print("🌿 Testing Execution Forking:")
print("="*60)

# Execute the original path
print("Original Execution Path:")
result1 = fork_app.invoke(
    {
        "messages": [HumanMessage(content="Execute original path")],
        "step": 0,
        "data": "",
        "fork_point": 0,
        "execution_path": "original"
    },
    config=config_fork
)

print(f"\n🎯 Original Path Result: {result1['data']}")

# Fork from step 2 with alternative path
print("\nAlternative Execution Path (Forked from Step 2):")
result2 = fork_app.invoke(
    {
        "messages": [HumanMessage(content="Fork from step 2 with alternative path")],
        "step": 2,
        "data": "Step 1: Base processing -> Step 2: original processing",
        "fork_point": 2,
        "execution_path": "alternative"
    },
    config=config_fork
)

print(f"🌿 Alternative Path Result: {result2['data']}")

# Fork from step 1 with experimental path
print("\nExperimental Execution Path (Forked from Step 1):")
result3 = fork_app.invoke(
    {
        "messages": [HumanMessage(content="Fork from step 1 with experimental path")],
        "step": 1,
        "data": "Step 1: Base processing",
        "fork_point": 1,
        "execution_path": "experimental"
    },
    config=config_fork
)

print(f"🌿 Experimental Path Result: {result3['data']}")

print("="*60)
print("✅ Execution forking demonstrated!")
print("🌿 Workflow can fork from any past state to explore alternative paths")


## Key Takeaways - Time Travel

✅ **When to use:**
- Debugging complex workflows
- Exploring alternative execution paths
- Analyzing workflow execution history
- Implementing undo/redo functionality

💡 **Key Features:**
- **Checkpoint Access**: Get state at any previous checkpoint
- **State Rewinding**: Revert to previous states
- **Execution Forking**: Branch from past states
- **History Navigation**: Traverse execution timeline

⚠️ **Common Pitfalls:**
- Time travel requires checkpointer for state persistence
- Consider performance impact of maintaining execution history
- Be careful with state mutations when forking
- Plan checkpoint frequency based on memory usage

---


# 🔟 Subgraphs

<a id="subgraphs"></a>

## Overview

**Subgraphs** in LangGraph allow you to create modular, reusable graph components that can be composed into larger workflows, enabling:
- **Modularity**: Break complex workflows into smaller, manageable pieces
- **Reusability**: Use the same subgraph in multiple workflows
- **Composition**: Combine subgraphs to build complex systems
- **Maintainability**: Update individual components without affecting others

## Key Features

- **Nested Graphs**: Graphs within graphs
- **Input/Output Mapping**: Define how data flows between subgraphs
- **State Isolation**: Each subgraph maintains its own state
- **Composition**: Combine multiple subgraphs into complex workflows

---

## Example 1: Basic Subgraph Creation


In [ ]:
# Define state for subgraph
class SubgraphState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    input_data: str
    processed_data: str
    subgraph_result: str

# Create a simple subgraph for data processing
def data_processor_node(state: SubgraphState) -> SubgraphState:
    """Node that processes data"""
    input_data = state.get("input_data", "")
    processed_data = f"Processed: {input_data.upper()}"
    
    print(f"🔧 Data Processor: {processed_data}")
    
    return {
        "processed_data": processed_data,
        "subgraph_result": processed_data
    }

def data_validator_node(state: SubgraphState) -> SubgraphState:
    """Node that validates processed data"""
    processed_data = state.get("processed_data", "")
    
    if len(processed_data) > 10:
        validation_result = f"✅ Valid: {processed_data}"
    else:
        validation_result = f"❌ Invalid: {processed_data} (too short)"
    
    print(f"🔍 Data Validator: {validation_result}")
    
    return {
        "subgraph_result": validation_result
    }

# Build the subgraph
data_processing_subgraph = StateGraph(SubgraphState)
data_processing_subgraph.add_node("processor", data_processor_node)
data_processing_subgraph.add_node("validator", data_validator_node)

data_processing_subgraph.add_edge(START, "processor")
data_processing_subgraph.add_edge("processor", "validator")
data_processing_subgraph.add_edge("validator", END)

# Compile the subgraph
compiled_subgraph = data_processing_subgraph.compile()

print("✅ Data processing subgraph created!")
print("🔧 This subgraph processes and validates data")


In [ ]:
# Test the subgraph independently
print("🔧 Testing Subgraph Independently:")
print("="*60)

subgraph_result = compiled_subgraph.invoke(
    {
        "messages": [HumanMessage(content="Test subgraph")],
        "input_data": "hello world",
        "processed_data": "",
        "subgraph_result": ""
    }
)

print(f"Subgraph Result: {subgraph_result['subgraph_result']}")

# Test with different input
subgraph_result2 = compiled_subgraph.invoke(
    {
        "messages": [HumanMessage(content="Test subgraph")],
        "input_data": "hi",
        "processed_data": "",
        "subgraph_result": ""
    }
)

print(f"Subgraph Result 2: {subgraph_result2['subgraph_result']}")

print("="*60)
print("✅ Subgraph tested independently!")
print("🔧 Subgraph can be used as a standalone component")


## Example 2: Subgraph Composition


In [ ]:
# Create another subgraph for data formatting
def data_formatter_node(state: SubgraphState) -> SubgraphState:
    """Node that formats data"""
    processed_data = state.get("processed_data", "")
    formatted_data = f"📋 Formatted: {processed_data}"
    
    print(f"📋 Data Formatter: {formatted_data}")
    
    return {
        "subgraph_result": formatted_data
    }

def data_enhancer_node(state: SubgraphState) -> SubgraphState:
    """Node that enhances data"""
    processed_data = state.get("processed_data", "")
    enhanced_data = f"✨ Enhanced: {processed_data} with metadata"
    
    print(f"✨ Data Enhancer: {enhanced_data}")
    
    return {
        "subgraph_result": enhanced_data
    }

# Build the formatting subgraph
data_formatting_subgraph = StateGraph(SubgraphState)
data_formatting_subgraph.add_node("formatter", data_formatter_node)
data_formatting_subgraph.add_node("enhancer", data_enhancer_node)

data_formatting_subgraph.add_edge(START, "formatter")
data_formatting_subgraph.add_edge("formatter", "enhancer")
data_formatting_subgraph.add_edge("enhancer", END)

# Compile the formatting subgraph
compiled_formatting_subgraph = data_formatting_subgraph.compile()

print("✅ Data formatting subgraph created!")
print("📋 This subgraph formats and enhances data")


In [ ]:
# Create a main workflow that composes multiple subgraphs
class MainWorkflowState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    input_data: str
    processing_result: str
    formatting_result: str
    final_result: str

def input_handler_node(state: MainWorkflowState) -> MainWorkflowState:
    """Node that handles input and prepares for subgraph processing"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    input_data = f"Input: {latest_message}"
    
    print(f"📥 Input Handler: {input_data}")
    
    return {
        "input_data": input_data
    }

def subgraph_processor_node(state: MainWorkflowState) -> MainWorkflowState:
    """Node that uses the data processing subgraph"""
    input_data = state["input_data"]
    
    # Use the compiled subgraph
    subgraph_result = compiled_subgraph.invoke(
        {
            "messages": [HumanMessage(content="Process data")],
            "input_data": input_data,
            "processed_data": "",
            "subgraph_result": ""
        }
    )
    
    processing_result = subgraph_result["subgraph_result"]
    
    print(f"🔧 Subgraph Processor: {processing_result}")
    
    return {
        "processing_result": processing_result
    }

def subgraph_formatter_node(state: MainWorkflowState) -> MainWorkflowState:
    """Node that uses the data formatting subgraph"""
    processing_result = state["processing_result"]
    
    # Use the compiled formatting subgraph
    subgraph_result = compiled_formatting_subgraph.invoke(
        {
            "messages": [HumanMessage(content="Format data")],
            "input_data": processing_result,
            "processed_data": processing_result,
            "subgraph_result": ""
        }
    )
    
    formatting_result = subgraph_result["subgraph_result"]
    
    print(f"📋 Subgraph Formatter: {formatting_result}")
    
    return {
        "formatting_result": formatting_result
    }

def final_combiner_node(state: MainWorkflowState) -> MainWorkflowState:
    """Node that combines results from all subgraphs"""
    processing_result = state["processing_result"]
    formatting_result = state["formatting_result"]
    
    final_result = f"🎯 Final Result:\n"
    final_result += f"Processing: {processing_result}\n"
    final_result += f"Formatting: {formatting_result}\n"
    final_result += "✅ Workflow completed successfully!"
    
    print(f"🎯 Final Combiner: {final_result}")
    
    return {
        "final_result": final_result
    }

# Build the main workflow that composes subgraphs
main_workflow = StateGraph(MainWorkflowState)
main_workflow.add_node("input_handler", input_handler_node)
main_workflow.add_node("subgraph_processor", subgraph_processor_node)
main_workflow.add_node("subgraph_formatter", subgraph_formatter_node)
main_workflow.add_node("final_combiner", final_combiner_node)

main_workflow.add_edge(START, "input_handler")
main_workflow.add_edge("input_handler", "subgraph_processor")
main_workflow.add_edge("subgraph_processor", "subgraph_formatter")
main_workflow.add_edge("subgraph_formatter", "final_combiner")
main_workflow.add_edge("final_combiner", END)

main_app = main_workflow.compile()

print("✅ Main workflow with subgraph composition created!")
print("🔧 This workflow composes multiple subgraphs into a larger system")


In [ ]:
# Test the composed workflow
print("🔧 Testing Composed Workflow:")
print("="*60)

result = main_app.invoke(
    {
        "messages": [HumanMessage(content="Process this data")],
        "input_data": "",
        "processing_result": "",
        "formatting_result": "",
        "final_result": ""
    }
)

print(f"\n🎯 Final Result: {result['final_result']}")

print("="*60)
print("✅ Subgraph composition demonstrated!")
print("🔧 Multiple subgraphs are composed into a larger workflow")


## Example 3: Conditional Subgraph Selection


In [ ]:
# Create specialized subgraphs for different data types
def text_processor_node(state: SubgraphState) -> SubgraphState:
    """Node that processes text data"""
    input_data = state.get("input_data", "")
    processed_data = f"Text processed: {input_data} (length: {len(input_data)})"
    
    print(f"📝 Text Processor: {processed_data}")
    
    return {
        "subgraph_result": processed_data
    }

def number_processor_node(state: SubgraphState) -> SubgraphState:
    """Node that processes numeric data"""
    input_data = state.get("input_data", "")
    
    # Try to extract numbers
    import re
    numbers = re.findall(r'\d+', input_data)
    if numbers:
        total = sum(int(n) for n in numbers)
        processed_data = f"Number processed: {input_data} (sum: {total})"
    else:
        processed_data = f"Number processed: {input_data} (no numbers found)"
    
    print(f"🔢 Number Processor: {processed_data}")
    
    return {
        "subgraph_result": processed_data
    }

def json_processor_node(state: SubgraphState) -> SubgraphState:
    """Node that processes JSON-like data"""
    input_data = state.get("input_data", "")
    
    if "{" in input_data and "}" in input_data:
        processed_data = f"JSON processed: {input_data} (valid JSON structure)"
    else:
        processed_data = f"JSON processed: {input_data} (not valid JSON)"
    
    print(f"📄 JSON Processor: {processed_data}")
    
    return {
        "subgraph_result": processed_data
    }

# Build specialized subgraphs
text_processing_subgraph = StateGraph(SubgraphState)
text_processing_subgraph.add_node("text_processor", text_processor_node)
text_processing_subgraph.add_edge(START, "text_processor")
text_processing_subgraph.add_edge("text_processor", END)

number_processing_subgraph = StateGraph(SubgraphState)
number_processing_subgraph.add_node("number_processor", number_processor_node)
number_processing_subgraph.add_edge(START, "number_processor")
number_processing_subgraph.add_edge("number_processor", END)

json_processing_subgraph = StateGraph(SubgraphState)
json_processing_subgraph.add_node("json_processor", json_processor_node)
json_processing_subgraph.add_edge(START, "json_processor")
json_processing_subgraph.add_edge("json_processor", END)

# Compile specialized subgraphs
compiled_text_subgraph = text_processing_subgraph.compile()
compiled_number_subgraph = number_processing_subgraph.compile()
compiled_json_subgraph = json_processing_subgraph.compile()

print("✅ Specialized subgraphs created!")
print("🔧 Each subgraph handles a specific data type")


In [ ]:
# Create a workflow that conditionally selects subgraphs
class ConditionalSubgraphState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    input_data: str
    data_type: str
    subgraph_result: str
    final_result: str

def data_type_classifier_node(state: ConditionalSubgraphState) -> ConditionalSubgraphState:
    """Node that classifies the data type"""
    messages = state["messages"]
    latest_message = messages[-1].content if messages else ""
    
    # Simple data type classification
    if any(char.isdigit() for char in latest_message):
        data_type = "number"
    elif "{" in latest_message and "}" in latest_message:
        data_type = "json"
    else:
        data_type = "text"
    
    print(f"🔍 Data Type Classifier: {data_type}")
    
    return {
        "input_data": latest_message,
        "data_type": data_type
    }

def subgraph_selector_node(state: ConditionalSubgraphState) -> ConditionalSubgraphState:
    """Node that selects and executes the appropriate subgraph"""
    input_data = state["input_data"]
    data_type = state["data_type"]
    
    # Select and execute the appropriate subgraph
    if data_type == "text":
        subgraph_result = compiled_text_subgraph.invoke(
            {
                "messages": [HumanMessage(content="Process text")],
                "input_data": input_data,
                "processed_data": "",
                "subgraph_result": ""
            }
        )
    elif data_type == "number":
        subgraph_result = compiled_number_subgraph.invoke(
            {
                "messages": [HumanMessage(content="Process number")],
                "input_data": input_data,
                "processed_data": "",
                "subgraph_result": ""
            }
        )
    else:  # json
        subgraph_result = compiled_json_subgraph.invoke(
            {
                "messages": [HumanMessage(content="Process JSON")],
                "input_data": input_data,
                "processed_data": "",
                "subgraph_result": ""
            }
        )
    
    result = subgraph_result["subgraph_result"]
    
    print(f"🔧 Subgraph Selector: {result}")
    
    return {
        "subgraph_result": result
    }

def result_formatter_node(state: ConditionalSubgraphState) -> ConditionalSubgraphState:
    """Node that formats the final result"""
    subgraph_result = state["subgraph_result"]
    data_type = state["data_type"]
    
    final_result = f"🎯 Final Result ({data_type}): {subgraph_result}"
    
    print(f"🎯 Result Formatter: {final_result}")
    
    return {
        "final_result": final_result
    }

# Build conditional subgraph workflow
conditional_workflow = StateGraph(ConditionalSubgraphState)
conditional_workflow.add_node("classifier", data_type_classifier_node)
conditional_workflow.add_node("selector", subgraph_selector_node)
conditional_workflow.add_node("formatter", result_formatter_node)

conditional_workflow.add_edge(START, "classifier")
conditional_workflow.add_edge("classifier", "selector")
conditional_workflow.add_edge("selector", "formatter")
conditional_workflow.add_edge("formatter", END)

conditional_app = conditional_workflow.compile()

print("✅ Conditional subgraph workflow created!")
print("🔧 This workflow conditionally selects subgraphs based on data type")


In [ ]:
# Test conditional subgraph selection
print("🔧 Testing Conditional Subgraph Selection:")
print("="*60)

# Test 1: Text data
print("Test 1: Text data")
result1 = conditional_app.invoke(
    {
        "messages": [HumanMessage(content="Hello world")],
        "input_data": "",
        "data_type": "",
        "subgraph_result": "",
        "final_result": ""
    }
)
print(f"Result: {result1['final_result']}\n")

# Test 2: Number data
print("Test 2: Number data")
result2 = conditional_app.invoke(
    {
        "messages": [HumanMessage(content="The numbers are 123 and 456")],
        "input_data": "",
        "data_type": "",
        "subgraph_result": "",
        "final_result": ""
    }
)
print(f"Result: {result2['final_result']}\n")

# Test 3: JSON data
print("Test 3: JSON data")
result3 = conditional_app.invoke(
    {
        "messages": [HumanMessage(content='{"name": "John", "age": 30}')],
        "input_data": "",
        "data_type": "",
        "subgraph_result": "",
        "final_result": ""
    }
)
print(f"Result: {result3['final_result']}\n")

print("="*60)
print("✅ Conditional subgraph selection demonstrated!")
print("🔧 Different subgraphs are selected based on data type")


## Key Takeaways - Subgraphs

✅ **When to use:**
- Building modular, reusable workflow components
- Composing complex systems from smaller parts
- Maintaining and updating individual components
- Creating specialized processing pipelines

💡 **Key Features:**
- **Nested Graphs**: Graphs within graphs
- **Input/Output Mapping**: Define how data flows between subgraphs
- **State Isolation**: Each subgraph maintains its own state
- **Composition**: Combine multiple subgraphs into complex workflows

⚠️ **Common Pitfalls:**
- Design clear interfaces between subgraphs
- Consider state management across subgraph boundaries
- Test subgraphs both independently and in composition
- Plan for subgraph versioning and updates

---


# 1️⃣1️⃣ Multi-Agent

<a id="multi-agent"></a>

## Overview

**Multi-Agent** systems in LangGraph enable multiple specialized agents to work together, coordinate, and collaborate on complex tasks, enabling:
- **Agent Specialization**: Different agents with specific expertise
- **Supervisor Pattern**: A coordinator agent that delegates tasks
- **Agent Collaboration**: Agents working together on shared goals
- **Hierarchical Structures**: Multi-level agent organizations

## Key Features

- **Agent Coordination**: Agents communicate and coordinate actions
- **Task Delegation**: Supervisor assigns tasks to specialized agents
- **Shared State**: Agents share information and context
- **Dynamic Routing**: Route tasks to appropriate agents

---

## Example 1: Basic Supervisor Pattern


In [ ]:
# Define state for multi-agent system
class MultiAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    task: str
    task_type: str
    assigned_agent: str
    agent_result: str
    supervisor_decision: str
    final_result: str

def supervisor_agent_node(state: MultiAgentState) -> MultiAgentState:
    """Supervisor agent that analyzes tasks and delegates to specialized agents"""
    messages = state["messages"]
    task = state.get("task", "")
    
    # Analyze task and determine which agent should handle it
    if any(word in task.lower() for word in ["analyze", "data", "statistics", "report"]):
        task_type = "analyst"
        assigned_agent = "data_analyst"
    elif any(word in task.lower() for word in ["write", "content", "article", "blog"]):
        task_type = "writer"
        assigned_agent = "content_writer"
    elif any(word in task.lower() for word in ["code", "program", "develop", "function"]):
        task_type = "developer"
        assigned_agent = "code_developer"
    else:
        task_type = "general"
        assigned_agent = "general_agent"
    
    supervisor_decision = f"Supervisor: Task '{task}' assigned to {assigned_agent} (type: {task_type})"
    
    print(f"👨‍💼 Supervisor: {supervisor_decision}")
    
    return {
        "task_type": task_type,
        "assigned_agent": assigned_agent,
        "supervisor_decision": supervisor_decision
    }

def data_analyst_agent_node(state: MultiAgentState) -> MultiAgentState:
    """Specialized agent for data analysis tasks"""
    task = state.get("task", "")
    
    # Simulate data analysis
    analysis_result = f"📊 Data Analyst: Analyzed '{task}'\n"
    analysis_result += "- Collected relevant data\n"
    analysis_result += "- Performed statistical analysis\n"
    analysis_result += "- Generated insights and recommendations\n"
    analysis_result += "- Created visualization charts"
    
    print(f"📊 Data Analyst: {analysis_result}")
    
    return {
        "agent_result": analysis_result
    }

def content_writer_agent_node(state: MultiAgentState) -> MultiAgentState:
    """Specialized agent for content writing tasks"""
    task = state.get("task", "")
    
    # Simulate content writing
    writing_result = f"✍️ Content Writer: Created content for '{task}'\n"
    writing_result += "- Researched the topic\n"
    writing_result += "- Outlined the structure\n"
    writing_result += "- Wrote engaging content\n"
    writing_result += "- Edited and proofread"
    
    print(f"✍️ Content Writer: {writing_result}")
    
    return {
        "agent_result": writing_result
    }

def code_developer_agent_node(state: MultiAgentState) -> MultiAgentState:
    """Specialized agent for coding tasks"""
    task = state.get("task", "")
    
    # Simulate code development
    development_result = f"💻 Code Developer: Developed solution for '{task}'\n"
    development_result += "- Analyzed requirements\n"
    development_result += "- Designed architecture\n"
    development_result += "- Implemented code\n"
    development_result += "- Tested and debugged"
    
    print(f"💻 Code Developer: {development_result}")
    
    return {
        "agent_result": development_result
    }

def general_agent_node(state: MultiAgentState) -> MultiAgentState:
    """General purpose agent for miscellaneous tasks"""
    task = state.get("task", "")
    
    # Simulate general task handling
    general_result = f"🤖 General Agent: Handled '{task}'\n"
    general_result += "- Understood the request\n"
    general_result += "- Applied general problem-solving\n"
    general_result += "- Provided helpful response\n"
    general_result += "- Suggested next steps"
    
    print(f"🤖 General Agent: {general_result}")
    
    return {
        "agent_result": general_result
    }

def result_aggregator_node(state: MultiAgentState) -> MultiAgentState:
    """Node that aggregates results from all agents"""
    supervisor_decision = state.get("supervisor_decision", "")
    agent_result = state.get("agent_result", "")
    
    final_result = f"🎯 Multi-Agent System Result:\n"
    final_result += f"{supervisor_decision}\n\n"
    final_result += f"{agent_result}\n\n"
    final_result += "✅ Task completed successfully by specialized agent!"
    
    print(f"🎯 Result Aggregator: {final_result}")
    
    return {
        "final_result": final_result
    }

# Build multi-agent workflow
multi_agent_workflow = StateGraph(MultiAgentState)

# Add nodes
multi_agent_workflow.add_node("supervisor", supervisor_agent_node)
multi_agent_workflow.add_node("data_analyst", data_analyst_agent_node)
multi_agent_workflow.add_node("content_writer", content_writer_agent_node)
multi_agent_workflow.add_node("code_developer", code_developer_agent_node)
multi_agent_workflow.add_node("general_agent", general_agent_node)
multi_agent_workflow.add_node("result_aggregator", result_aggregator_node)

# Add edges
multi_agent_workflow.add_edge(START, "supervisor")

# Conditional routing based on assigned agent
def route_to_agent(state: MultiAgentState) -> str:
    assigned_agent = state.get("assigned_agent", "")
    return assigned_agent

multi_agent_workflow.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "data_analyst": "data_analyst",
        "content_writer": "content_writer",
        "code_developer": "code_developer",
        "general_agent": "general_agent"
    }
)

# All agent nodes go to result aggregator
multi_agent_workflow.add_edge("data_analyst", "result_aggregator")
multi_agent_workflow.add_edge("content_writer", "result_aggregator")
multi_agent_workflow.add_edge("code_developer", "result_aggregator")
multi_agent_workflow.add_edge("general_agent", "result_aggregator")
multi_agent_workflow.add_edge("result_aggregator", END)

multi_agent_app = multi_agent_workflow.compile()

print("✅ Multi-agent workflow created!")
print("👨‍💼 This workflow uses a supervisor pattern with specialized agents")


In [ ]:
# Test multi-agent system with different task types
print("👨‍💼 Testing Multi-Agent System:")
print("="*60)

# Test 1: Data analysis task
print("Test 1: Data analysis task")
result1 = multi_agent_app.invoke(
    {
        "messages": [HumanMessage(content="Analyze sales data for Q4")],
        "task": "Analyze sales data for Q4",
        "task_type": "",
        "assigned_agent": "",
        "agent_result": "",
        "supervisor_decision": "",
        "final_result": ""
    }
)
print(f"Result: {result1['final_result']}\n")

# Test 2: Content writing task
print("Test 2: Content writing task")
result2 = multi_agent_app.invoke(
    {
        "messages": [HumanMessage(content="Write a blog post about AI")],
        "task": "Write a blog post about AI",
        "task_type": "",
        "assigned_agent": "",
        "agent_result": "",
        "supervisor_decision": "",
        "final_result": ""
    }
)
print(f"Result: {result2['final_result']}\n")

# Test 3: Coding task
print("Test 3: Coding task")
result3 = multi_agent_app.invoke(
    {
        "messages": [HumanMessage(content="Develop a Python function for data processing")],
        "task": "Develop a Python function for data processing",
        "task_type": "",
        "assigned_agent": "",
        "agent_result": "",
        "supervisor_decision": "",
        "final_result": ""
    }
)
print(f"Result: {result3['final_result']}\n")

# Test 4: General task
print("Test 4: General task")
result4 = multi_agent_app.invoke(
    {
        "messages": [HumanMessage(content="Help me organize my schedule")],
        "task": "Help me organize my schedule",
        "task_type": "",
        "assigned_agent": "",
        "agent_result": "",
        "supervisor_decision": "",
        "final_result": ""
    }
)
print(f"Result: {result4['final_result']}\n")

print("="*60)
print("✅ Multi-agent system demonstrated!")
print("👨‍💼 Supervisor delegates tasks to specialized agents based on task type")


## Example 2: Agent Collaboration


In [ ]:
# Create a collaborative multi-agent system
class CollaborativeState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    task: str
    research_result: str
    analysis_result: str
    writing_result: str
    review_result: str
    final_output: str
    collaboration_status: str

def research_agent_node(state: CollaborativeState) -> CollaborativeState:
    """Agent specialized in research and information gathering"""
    task = state.get("task", "")
    
    research_result = f"🔍 Research Agent: Researched '{task}'\n"
    research_result += "- Gathered relevant information from multiple sources\n"
    research_result += "- Identified key facts and data points\n"
    research_result += "- Compiled comprehensive research notes\n"
    research_result += "- Highlighted important trends and patterns"
    
    print(f"🔍 Research Agent: {research_result}")
    
    return {
        "research_result": research_result,
        "collaboration_status": "research_complete"
    }

def analysis_agent_node(state: CollaborativeState) -> CollaborativeState:
    """Agent specialized in data analysis and insights"""
    research_result = state.get("research_result", "")
    task = state.get("task", "")
    
    analysis_result = f"📊 Analysis Agent: Analyzed research for '{task}'\n"
    analysis_result += "- Processed research findings\n"
    analysis_result += "- Identified key insights and patterns\n"
    analysis_result += "- Generated data visualizations\n"
    analysis_result += "- Created actionable recommendations"
    
    print(f"📊 Analysis Agent: {analysis_result}")
    
    return {
        "analysis_result": analysis_result,
        "collaboration_status": "analysis_complete"
    }

def writing_agent_node(state: CollaborativeState) -> CollaborativeState:
    """Agent specialized in content creation and writing"""
    research_result = state.get("research_result", "")
    analysis_result = state.get("analysis_result", "")
    task = state.get("task", "")
    
    writing_result = f"✍️ Writing Agent: Created content for '{task}'\n"
    writing_result += "- Synthesized research and analysis\n"
    writing_result += "- Structured the information logically\n"
    writing_result += "- Wrote engaging and informative content\n"
    writing_result += "- Ensured clarity and coherence"
    
    print(f"✍️ Writing Agent: {writing_result}")
    
    return {
        "writing_result": writing_result,
        "collaboration_status": "writing_complete"
    }

def review_agent_node(state: CollaborativeState) -> CollaborativeState:
    """Agent specialized in quality review and editing"""
    research_result = state.get("research_result", "")
    analysis_result = state.get("analysis_result", "")
    writing_result = state.get("writing_result", "")
    task = state.get("task", "")
    
    review_result = f"🔍 Review Agent: Reviewed content for '{task}'\n"
    review_result += "- Checked accuracy and completeness\n"
    review_result += "- Improved clarity and flow\n"
    review_result += "- Enhanced readability and engagement\n"
    review_result += "- Ensured quality standards"
    
    print(f"🔍 Review Agent: {review_result}")
    
    return {
        "review_result": review_result,
        "collaboration_status": "review_complete"
    }

def collaboration_coordinator_node(state: CollaborativeState) -> CollaborativeState:
    """Node that coordinates the collaborative process"""
    research_result = state.get("research_result", "")
    analysis_result = state.get("analysis_result", "")
    writing_result = state.get("writing_result", "")
    review_result = state.get("review_result", "")
    
    final_output = f"🎯 Collaborative Multi-Agent Result:\n\n"
    final_output += f"Research Phase:\n{research_result}\n\n"
    final_output += f"Analysis Phase:\n{analysis_result}\n\n"
    final_output += f"Writing Phase:\n{writing_result}\n\n"
    final_output += f"Review Phase:\n{review_result}\n\n"
    final_output += "✅ All agents collaborated successfully to produce high-quality output!"
    
    print(f"🎯 Collaboration Coordinator: {final_output}")
    
    return {
        "final_output": final_output,
        "collaboration_status": "complete"
    }

# Build collaborative multi-agent workflow
collaborative_workflow = StateGraph(CollaborativeState)

# Add nodes
collaborative_workflow.add_node("research_agent", research_agent_node)
collaborative_workflow.add_node("analysis_agent", analysis_agent_node)
collaborative_workflow.add_node("writing_agent", writing_agent_node)
collaborative_workflow.add_node("review_agent", review_agent_node)
collaborative_workflow.add_node("coordinator", collaboration_coordinator_node)

# Add edges for sequential collaboration
collaborative_workflow.add_edge(START, "research_agent")
collaborative_workflow.add_edge("research_agent", "analysis_agent")
collaborative_workflow.add_edge("analysis_agent", "writing_agent")
collaborative_workflow.add_edge("writing_agent", "review_agent")
collaborative_workflow.add_edge("review_agent", "coordinator")
collaborative_workflow.add_edge("coordinator", END)

collaborative_app = collaborative_workflow.compile()

print("✅ Collaborative multi-agent workflow created!")
print("🤝 This workflow demonstrates agents working together sequentially")


In [ ]:
# Test collaborative multi-agent system
print("🤝 Testing Collaborative Multi-Agent System:")
print("="*60)

result = collaborative_app.invoke(
    {
        "messages": [HumanMessage(content="Create a comprehensive report on renewable energy trends")],
        "task": "Create a comprehensive report on renewable energy trends",
        "research_result": "",
        "analysis_result": "",
        "writing_result": "",
        "review_result": "",
        "final_output": "",
        "collaboration_status": ""
    }
)

print(f"\nFinal Output: {result['final_output']}")

print("="*60)
print("✅ Collaborative multi-agent system demonstrated!")
print("🤝 Multiple agents work together sequentially to produce comprehensive results")


## Example 3: Hierarchical Agent Structure


In [ ]:
# Create a hierarchical multi-agent system
class HierarchicalState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    task: str
    department: str
    team_lead_result: str
    specialist_result: str
    manager_decision: str
    final_approval: str
    hierarchical_output: str

def department_manager_node(state: HierarchicalState) -> HierarchicalState:
    """Top-level manager that oversees the entire process"""
    task = state.get("task", "")
    
    # Determine which department should handle the task
    if any(word in task.lower() for word in ["technical", "engineering", "development"]):
        department = "engineering"
    elif any(word in task.lower() for word in ["marketing", "promotion", "campaign"]):
        department = "marketing"
    elif any(word in task.lower() for word in ["finance", "budget", "cost"]):
        department = "finance"
    else:
        department = "general"
    
    manager_decision = f"👔 Department Manager: Task '{task}' assigned to {department} department"
    
    print(f"👔 Department Manager: {manager_decision}")
    
    return {
        "department": department,
        "manager_decision": manager_decision
    }

def engineering_team_lead_node(state: HierarchicalState) -> HierarchicalState:
    """Team lead for engineering department"""
    task = state.get("task", "")
    
    team_lead_result = f"🔧 Engineering Team Lead: Coordinated '{task}'\n"
    team_lead_result += "- Reviewed technical requirements\n"
    team_lead_result += "- Assigned tasks to specialists\n"
    team_lead_result += "- Monitored progress and quality\n"
    team_lead_result += "- Ensured technical standards"
    
    print(f"🔧 Engineering Team Lead: {team_lead_result}")
    
    return {
        "team_lead_result": team_lead_result
    }

def marketing_team_lead_node(state: HierarchicalState) -> HierarchicalState:
    """Team lead for marketing department"""
    task = state.get("task", "")
    
    team_lead_result = f"📢 Marketing Team Lead: Coordinated '{task}'\n"
    team_lead_result += "- Developed marketing strategy\n"
    team_lead_result += "- Coordinated campaign execution\n"
    team_lead_result += "- Managed brand consistency\n"
    team_lead_result += "- Tracked performance metrics"
    
    print(f"📢 Marketing Team Lead: {team_lead_result}")
    
    return {
        "team_lead_result": team_lead_result
    }

def finance_team_lead_node(state: HierarchicalState) -> HierarchicalState:
    """Team lead for finance department"""
    task = state.get("task", "")
    
    team_lead_result = f"💰 Finance Team Lead: Coordinated '{task}'\n"
    team_lead_result += "- Analyzed financial implications\n"
    team_lead_result += "- Prepared budget estimates\n"
    team_lead_result += "- Ensured compliance\n"
    team_lead_result += "- Provided cost-benefit analysis"
    
    print(f"💰 Finance Team Lead: {team_lead_result}")
    
    return {
        "team_lead_result": team_lead_result
    }

def specialist_agent_node(state: HierarchicalState) -> HierarchicalState:
    """Specialist agent that executes the actual work"""
    task = state.get("task", "")
    department = state.get("department", "")
    
    specialist_result = f"🎯 {department.title()} Specialist: Executed '{task}'\n"
    specialist_result += "- Applied specialized knowledge\n"
    specialist_result += "- Delivered high-quality work\n"
    specialist_result += "- Met all requirements\n"
    specialist_result += "- Provided detailed results"
    
    print(f"🎯 {department.title()} Specialist: {specialist_result}")
    
    return {
        "specialist_result": specialist_result
    }

def final_approval_node(state: HierarchicalState) -> HierarchicalState:
    """Final approval from top management"""
    manager_decision = state.get("manager_decision", "")
    team_lead_result = state.get("team_lead_result", "")
    specialist_result = state.get("specialist_result", "")
    
    final_approval = f"✅ Final Approval: Task completed successfully\n"
    final_approval += "- All quality standards met\n"
    final_approval += "- Delivered on time\n"
    final_approval += "- Exceeded expectations\n"
    final_approval += "- Ready for implementation"
    
    hierarchical_output = f"🏢 Hierarchical Multi-Agent System Result:\n\n"
    hierarchical_output += f"Management Decision:\n{manager_decision}\n\n"
    hierarchical_output += f"Team Lead Coordination:\n{team_lead_result}\n\n"
    hierarchical_output += f"Specialist Execution:\n{specialist_result}\n\n"
    hierarchical_output += f"Final Approval:\n{final_approval}\n\n"
    hierarchical_output += "🎯 Hierarchical structure ensured quality and accountability!"
    
    print(f"✅ Final Approval: {final_approval}")
    print(f"🏢 Hierarchical Output: {hierarchical_output}")
    
    return {
        "final_approval": final_approval,
        "hierarchical_output": hierarchical_output
    }

# Build hierarchical multi-agent workflow
hierarchical_workflow = StateGraph(HierarchicalState)

# Add nodes
hierarchical_workflow.add_node("department_manager", department_manager_node)
hierarchical_workflow.add_node("engineering_lead", engineering_team_lead_node)
hierarchical_workflow.add_node("marketing_lead", marketing_team_lead_node)
hierarchical_workflow.add_node("finance_lead", finance_team_lead_node)
hierarchical_workflow.add_node("specialist", specialist_agent_node)
hierarchical_workflow.add_node("final_approval", final_approval_node)

# Add edges
hierarchical_workflow.add_edge(START, "department_manager")

# Conditional routing based on department
def route_to_department(state: HierarchicalState) -> str:
    department = state.get("department", "")
    return f"{department}_lead"

hierarchical_workflow.add_conditional_edges(
    "department_manager",
    route_to_department,
    {
        "engineering_lead": "engineering_lead",
        "marketing_lead": "marketing_lead",
        "finance_lead": "finance_lead"
    }
)

# All team leads go to specialist
hierarchical_workflow.add_edge("engineering_lead", "specialist")
hierarchical_workflow.add_edge("marketing_lead", "specialist")
hierarchical_workflow.add_edge("finance_lead", "specialist")

# Specialist goes to final approval
hierarchical_workflow.add_edge("specialist", "final_approval")
hierarchical_workflow.add_edge("final_approval", END)

hierarchical_app = hierarchical_workflow.compile()

print("✅ Hierarchical multi-agent workflow created!")
print("🏢 This workflow demonstrates a hierarchical organizational structure")


In [ ]:
# Test hierarchical multi-agent system
print("🏢 Testing Hierarchical Multi-Agent System:")
print("="*60)

# Test 1: Engineering task
print("Test 1: Engineering task")
result1 = hierarchical_app.invoke(
    {
        "messages": [HumanMessage(content="Develop a new technical solution")],
        "task": "Develop a new technical solution",
        "department": "",
        "team_lead_result": "",
        "specialist_result": "",
        "manager_decision": "",
        "final_approval": "",
        "hierarchical_output": ""
    }
)
print(f"Result: {result1['hierarchical_output']}\n")

# Test 2: Marketing task
print("Test 2: Marketing task")
result2 = hierarchical_app.invoke(
    {
        "messages": [HumanMessage(content="Launch a new marketing campaign")],
        "task": "Launch a new marketing campaign",
        "department": "",
        "team_lead_result": "",
        "specialist_result": "",
        "manager_decision": "",
        "final_approval": "",
        "hierarchical_output": ""
    }
)
print(f"Result: {result2['hierarchical_output']}\n")

# Test 3: Finance task
print("Test 3: Finance task")
result3 = hierarchical_app.invoke(
    {
        "messages": [HumanMessage(content="Analyze budget allocation for Q4")],
        "task": "Analyze budget allocation for Q4",
        "department": "",
        "team_lead_result": "",
        "specialist_result": "",
        "manager_decision": "",
        "final_approval": "",
        "hierarchical_output": ""
    }
)
print(f"Result: {result3['hierarchical_output']}\n")

print("="*60)
print("✅ Hierarchical multi-agent system demonstrated!")
print("🏢 Hierarchical structure ensures proper oversight and quality control")


## Key Takeaways - Multi-Agent

✅ **When to use:**
- Complex tasks requiring multiple areas of expertise
- Workflows that benefit from specialization
- Systems requiring coordination and oversight
- Hierarchical organizational structures

💡 **Key Features:**
- **Agent Specialization**: Different agents with specific expertise
- **Supervisor Pattern**: A coordinator agent that delegates tasks
- **Agent Collaboration**: Agents working together on shared goals
- **Hierarchical Structures**: Multi-level agent organizations

⚠️ **Common Pitfalls:**
- Design clear communication protocols between agents
- Avoid over-complexity in agent coordination
- Consider performance implications of multi-agent systems
- Plan for agent failure and recovery scenarios

---


# 1️⃣2️⃣ MCP Integration

<a id="mcp-integration"></a>

## Overview

**MCP Integration** in LangGraph enables connecting to external systems and resources through Model Context Protocol (MCP) servers, enabling:
- **External Resource Access**: Connect to databases, APIs, and external services
- **Tool Integration**: Use MCP tools within graph workflows
- **Dynamic Resource Discovery**: Discover and use available MCP resources
- **Protocol Compliance**: Follow MCP standards for interoperability

## Key Features

- **MCP Server Connection**: Connect to external MCP servers
- **Resource Discovery**: Discover available tools and resources
- **Tool Execution**: Execute MCP tools within graph nodes
- **Protocol Compliance**: Follow MCP communication protocols

---

## Example 1: Basic MCP Server Connection


In [ ]:
# Note: MCP integration requires additional setup and MCP servers
# This example demonstrates the concept with simulated MCP functionality

class MCPState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    mcp_server_url: str
    available_tools: list[str]
    selected_tool: str
    tool_result: str
    mcp_response: str

def mcp_connection_node(state: MCPState) -> MCPState:
    """Node that simulates connecting to an MCP server"""
    mcp_server_url = state.get("mcp_server_url", "")
    
    # Simulate MCP server connection
    print(f"🔌 Connecting to MCP server: {mcp_server_url}")
    
    # Simulate discovering available tools
    available_tools = [
        "database_query",
        "file_operations", 
        "api_calls",
        "data_processing",
        "external_search"
    ]
    
    print(f"🔍 Discovered {len(available_tools)} available tools: {available_tools}")
    
    return {
        "available_tools": available_tools,
        "mcp_response": f"Successfully connected to MCP server at {mcp_server_url}"
    }

def tool_discovery_node(state: MCPState) -> MCPState:
    """Node that discovers and categorizes MCP tools"""
    available_tools = state.get("available_tools", [])
    
    # Categorize tools by type
    tool_categories = {
        "data": ["database_query", "data_processing"],
        "file": ["file_operations"],
        "network": ["api_calls", "external_search"],
        "utility": []
    }
    
    discovery_result = f"📋 MCP Tool Discovery Results:\n"
    discovery_result += f"- Total tools: {len(available_tools)}\n"
    discovery_result += f"- Data tools: {tool_categories['data']}\n"
    discovery_result += f"- File tools: {tool_categories['file']}\n"
    discovery_result += f"- Network tools: {tool_categories['network']}\n"
    discovery_result += f"- Utility tools: {tool_categories['utility']}"
    
    print(f"📋 Tool Discovery: {discovery_result}")
    
    return {
        "mcp_response": discovery_result
    }

def tool_execution_node(state: MCPState) -> MCPState:
    """Node that executes selected MCP tool"""
    selected_tool = state.get("selected_tool", "")
    available_tools = state.get("available_tools", [])
    
    if selected_tool not in available_tools:
        tool_result = f"❌ Tool '{selected_tool}' not available"
        print(f"❌ Tool Execution Failed: {tool_result}")
        return {"tool_result": tool_result}
    
    # Simulate tool execution based on tool type
    if selected_tool == "database_query":
        tool_result = f"🗄️ Database Query Result:\n- Executed SQL query\n- Retrieved 150 records\n- Processing time: 0.3s"
    elif selected_tool == "file_operations":
        tool_result = f"📁 File Operations Result:\n- Created backup file\n- Updated configuration\n- Verified file integrity"
    elif selected_tool == "api_calls":
        tool_result = f"🌐 API Call Result:\n- Connected to external API\n- Retrieved data successfully\n- Response time: 0.8s"
    elif selected_tool == "data_processing":
        tool_result = f"⚙️ Data Processing Result:\n- Processed 1000 records\n- Applied transformations\n- Generated reports"
    elif selected_tool == "external_search":
        tool_result = f"🔍 External Search Result:\n- Searched external sources\n- Found 25 relevant results\n- Ranked by relevance"
    else:
        tool_result = f"🔧 Tool '{selected_tool}' executed successfully"
    
    print(f"🔧 Tool Execution: {tool_result}")
    
    return {
        "tool_result": tool_result
    }

def mcp_response_node(state: MCPState) -> MCPState:
    """Node that formats MCP integration response"""
    mcp_response = state.get("mcp_response", "")
    tool_result = state.get("tool_result", "")
    
    final_response = f"🎯 MCP Integration Result:\n\n"
    final_response += f"Connection Status:\n{mcp_response}\n\n"
    final_response += f"Tool Execution:\n{tool_result}\n\n"
    final_response += "✅ MCP integration completed successfully!"
    
    print(f"🎯 MCP Response: {final_response}")
    
    return {
        "mcp_response": final_response
    }

# Build MCP integration workflow
mcp_workflow = StateGraph(MCPState)

# Add nodes
mcp_workflow.add_node("mcp_connection", mcp_connection_node)
mcp_workflow.add_node("tool_discovery", tool_discovery_node)
mcp_workflow.add_node("tool_execution", tool_execution_node)
mcp_workflow.add_node("mcp_response", mcp_response_node)

# Add edges
mcp_workflow.add_edge(START, "mcp_connection")
mcp_workflow.add_edge("mcp_connection", "tool_discovery")
mcp_workflow.add_edge("tool_discovery", "tool_execution")
mcp_workflow.add_edge("tool_execution", "mcp_response")
mcp_workflow.add_edge("mcp_response", END)

mcp_app = mcp_workflow.compile()

print("✅ MCP integration workflow created!")
print("🔌 This workflow demonstrates connecting to external MCP servers")


In [ ]:
# Test MCP integration workflow
print("🔌 Testing MCP Integration:")
print("="*60)

result = mcp_app.invoke(
    {
        "messages": [HumanMessage(content="Connect to MCP server and execute database query")],
        "mcp_server_url": "https://mcp-server.example.com",
        "available_tools": [],
        "selected_tool": "database_query",
        "tool_result": "",
        "mcp_response": ""
    }
)

print(f"\nFinal Response: {result['mcp_response']}")

print("="*60)
print("✅ MCP integration demonstrated!")
print("🔌 Successfully connected to MCP server and executed tools")


## Example 2: Dynamic MCP Resource Discovery


In [ ]:
# Create a more sophisticated MCP integration system
class DynamicMCPState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    mcp_servers: list[str]
    discovered_resources: dict
    resource_categories: dict
    selected_resource: str
    execution_result: str
    dynamic_response: str

def mcp_server_scanner_node(state: DynamicMCPState) -> DynamicMCPState:
    """Node that scans multiple MCP servers"""
    mcp_servers = state.get("mcp_servers", [])
    
    print(f"🔍 Scanning {len(mcp_servers)} MCP servers...")
    
    # Simulate discovering resources from different servers
    discovered_resources = {
        "server1.example.com": {
            "tools": ["database_query", "data_export", "user_management"],
            "capabilities": ["sql", "json", "authentication"],
            "status": "online"
        },
        "server2.example.com": {
            "tools": ["file_operations", "backup", "sync"],
            "capabilities": ["filesystem", "compression", "encryption"],
            "status": "online"
        },
        "server3.example.com": {
            "tools": ["api_gateway", "webhook", "monitoring"],
            "capabilities": ["http", "websocket", "metrics"],
            "status": "online"
        }
    }
    
    print(f"📡 Discovered resources from {len(discovered_resources)} servers")
    
    return {
        "discovered_resources": discovered_resources
    }

def resource_categorizer_node(state: DynamicMCPState) -> DynamicMCPState:
    """Node that categorizes discovered MCP resources"""
    discovered_resources = state.get("discovered_resources", {})
    
    # Categorize resources by type and capability
    resource_categories = {
        "data_management": {
            "tools": ["database_query", "data_export"],
            "servers": ["server1.example.com"],
            "description": "Database and data processing tools"
        },
        "file_operations": {
            "tools": ["file_operations", "backup", "sync"],
            "servers": ["server2.example.com"],
            "description": "File system and storage operations"
        },
        "api_services": {
            "tools": ["api_gateway", "webhook", "monitoring"],
            "servers": ["server3.example.com"],
            "description": "API and web service tools"
        },
        "user_management": {
            "tools": ["user_management"],
            "servers": ["server1.example.com"],
            "description": "User authentication and management"
        }
    }
    
    categorization_result = f"📋 Resource Categorization:\n"
    for category, info in resource_categories.items():
        categorization_result += f"- {category}: {info['description']}\n"
        categorization_result += f"  Tools: {info['tools']}\n"
        categorization_result += f"  Servers: {info['servers']}\n"
    
    print(f"📋 Resource Categorization: {categorization_result}")
    
    return {
        "resource_categories": resource_categories
    }

def resource_selector_node(state: DynamicMCPState) -> DynamicMCPState:
    """Node that selects appropriate resource based on task"""
    resource_categories = state.get("resource_categories", {})
    messages = state["messages"]
    
    # Analyze the task to determine which resource to use
    task_content = messages[0].content.lower() if messages else ""
    
    if any(word in task_content for word in ["database", "query", "data"]):
        selected_resource = "data_management"
    elif any(word in task_content for word in ["file", "backup", "storage"]):
        selected_resource = "file_operations"
    elif any(word in task_content for word in ["api", "webhook", "monitor"]):
        selected_resource = "api_services"
    elif any(word in task_content for word in ["user", "auth", "login"]):
        selected_resource = "user_management"
    else:
        selected_resource = "data_management"  # Default
    
    print(f"🎯 Selected resource category: {selected_resource}")
    
    return {
        "selected_resource": selected_resource
    }

def resource_executor_node(state: DynamicMCPState) -> DynamicMCPState:
    """Node that executes the selected MCP resource"""
    selected_resource = state.get("selected_resource", "")
    resource_categories = state.get("resource_categories", {})
    
    if selected_resource not in resource_categories:
        execution_result = f"❌ Resource category '{selected_resource}' not found"
        print(f"❌ Resource Execution Failed: {execution_result}")
        return {"execution_result": execution_result}
    
    resource_info = resource_categories[selected_resource]
    tools = resource_info["tools"]
    servers = resource_info["servers"]
    
    # Simulate executing tools from the selected category
    execution_result = f"⚙️ Executing {selected_resource} resources:\n"
    execution_result += f"- Available tools: {tools}\n"
    execution_result += f"- Target servers: {servers}\n"
    execution_result += f"- Description: {resource_info['description']}\n"
    execution_result += f"- Status: Successfully executed {len(tools)} tools"
    
    print(f"⚙️ Resource Execution: {execution_result}")
    
    return {
        "execution_result": execution_result
    }

def dynamic_response_node(state: DynamicMCPState) -> DynamicMCPState:
    """Node that formats dynamic MCP response"""
    discovered_resources = state.get("discovered_resources", {})
    resource_categories = state.get("resource_categories", {})
    execution_result = state.get("execution_result", "")
    
    dynamic_response = f"🎯 Dynamic MCP Integration Result:\n\n"
    dynamic_response += f"Server Discovery:\n"
    for server, info in discovered_resources.items():
        dynamic_response += f"- {server}: {len(info['tools'])} tools, {info['status']}\n"
    
    dynamic_response += f"\nResource Categories:\n"
    for category, info in resource_categories.items():
        dynamic_response += f"- {category}: {len(info['tools'])} tools\n"
    
    dynamic_response += f"\nExecution Result:\n{execution_result}\n\n"
    dynamic_response += "✅ Dynamic MCP resource discovery and execution completed!"
    
    print(f"🎯 Dynamic Response: {dynamic_response}")
    
    return {
        "dynamic_response": dynamic_response
    }

# Build dynamic MCP integration workflow
dynamic_mcp_workflow = StateGraph(DynamicMCPState)

# Add nodes
dynamic_mcp_workflow.add_node("server_scanner", mcp_server_scanner_node)
dynamic_mcp_workflow.add_node("resource_categorizer", resource_categorizer_node)
dynamic_mcp_workflow.add_node("resource_selector", resource_selector_node)
dynamic_mcp_workflow.add_node("resource_executor", resource_executor_node)
dynamic_mcp_workflow.add_node("dynamic_response", dynamic_response_node)

# Add edges
dynamic_mcp_workflow.add_edge(START, "server_scanner")
dynamic_mcp_workflow.add_edge("server_scanner", "resource_categorizer")
dynamic_mcp_workflow.add_edge("resource_categorizer", "resource_selector")
dynamic_mcp_workflow.add_edge("resource_selector", "resource_executor")
dynamic_mcp_workflow.add_edge("resource_executor", "dynamic_response")
dynamic_mcp_workflow.add_edge("dynamic_response", END)

dynamic_mcp_app = dynamic_mcp_workflow.compile()

print("✅ Dynamic MCP integration workflow created!")
print("🔍 This workflow demonstrates dynamic resource discovery and categorization")


In [ ]:
# Test dynamic MCP integration workflow
print("🔍 Testing Dynamic MCP Integration:")
print("="*60)

# Test 1: Database task
print("Test 1: Database task")
result1 = dynamic_mcp_app.invoke(
    {
        "messages": [HumanMessage(content="Query the database for user information")],
        "mcp_servers": ["server1.example.com", "server2.example.com", "server3.example.com"],
        "discovered_resources": {},
        "resource_categories": {},
        "selected_resource": "",
        "execution_result": "",
        "dynamic_response": ""
    }
)
print(f"Result: {result1['dynamic_response']}\n")

# Test 2: File operations task
print("Test 2: File operations task")
result2 = dynamic_mcp_app.invoke(
    {
        "messages": [HumanMessage(content="Backup important files to secure storage")],
        "mcp_servers": ["server1.example.com", "server2.example.com", "server3.example.com"],
        "discovered_resources": {},
        "resource_categories": {},
        "selected_resource": "",
        "execution_result": "",
        "dynamic_response": ""
    }
)
print(f"Result: {result2['dynamic_response']}\n")

# Test 3: API monitoring task
print("Test 3: API monitoring task")
result3 = dynamic_mcp_app.invoke(
    {
        "messages": [HumanMessage(content="Monitor API endpoints and webhook status")],
        "mcp_servers": ["server1.example.com", "server2.example.com", "server3.example.com"],
        "discovered_resources": {},
        "resource_categories": {},
        "selected_resource": "",
        "execution_result": "",
        "dynamic_response": ""
    }
)
print(f"Result: {result3['dynamic_response']}\n")

print("="*60)
print("✅ Dynamic MCP integration demonstrated!")
print("🔍 Successfully discovered and categorized resources from multiple servers")


## Example 3: MCP Tool Integration with Error Handling


In [ ]:
# Create MCP integration with robust error handling
class RobustMCPState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    mcp_config: dict
    tool_requests: list[str]
    execution_results: list[dict]
    error_log: list[str]
    success_count: int
    failure_count: int
    final_summary: str

def mcp_configuration_node(state: RobustMCPState) -> RobustMCPState:
    """Node that configures MCP connections with error handling"""
    mcp_config = {
        "servers": [
            {"url": "https://mcp-server1.com", "timeout": 30, "retries": 3},
            {"url": "https://mcp-server2.com", "timeout": 45, "retries": 2},
            {"url": "https://mcp-server3.com", "timeout": 60, "retries": 1}
        ],
        "global_timeout": 120,
        "max_concurrent": 5,
        "error_threshold": 0.3
    }
    
    print(f"⚙️ MCP Configuration: {len(mcp_config['servers'])} servers configured")
    
    return {
        "mcp_config": mcp_config
    }

def tool_request_processor_node(state: RobustMCPState) -> RobustMCPState:
    """Node that processes tool requests with validation"""
    messages = state["messages"]
    
    # Extract tool requests from messages
    tool_requests = []
    if messages:
        content = messages[0].content.lower()
        if "database" in content:
            tool_requests.append("database_query")
        if "file" in content:
            tool_requests.append("file_operations")
        if "api" in content:
            tool_requests.append("api_calls")
        if "search" in content:
            tool_requests.append("external_search")
    
    # Add some default requests if none found
    if not tool_requests:
        tool_requests = ["database_query", "file_operations"]
    
    print(f"📋 Tool Requests: {tool_requests}")
    
    return {
        "tool_requests": tool_requests
    }

def mcp_tool_executor_node(state: RobustMCPState) -> RobustMCPState:
    """Node that executes MCP tools with error handling"""
    tool_requests = state.get("tool_requests", [])
    mcp_config = state.get("mcp_config", {})
    
    execution_results = []
    error_log = []
    success_count = 0
    failure_count = 0
    
    print(f"🔧 Executing {len(tool_requests)} MCP tools...")
    
    for i, tool in enumerate(tool_requests):
        try:
            # Simulate tool execution with potential failures
            import random
            success_probability = 0.8  # 80% success rate
            
            if random.random() < success_probability:
                # Successful execution
                result = {
                    "tool": tool,
                    "status": "success",
                    "execution_time": random.uniform(0.1, 2.0),
                    "result": f"✅ {tool} executed successfully",
                    "server": f"mcp-server{(i % 3) + 1}.com"
                }
                execution_results.append(result)
                success_count += 1
                print(f"✅ {tool}: Success")
            else:
                # Simulated failure
                error_msg = f"❌ {tool} failed: Connection timeout"
                error_log.append(error_msg)
                failure_count += 1
                print(f"❌ {tool}: Failed")
                
        except Exception as e:
            error_msg = f"❌ {tool} error: {str(e)}"
            error_log.append(error_msg)
            failure_count += 1
            print(f"❌ {tool}: Exception")
    
    print(f"📊 Execution Summary: {success_count} success, {failure_count} failures")
    
    return {
        "execution_results": execution_results,
        "error_log": error_log,
        "success_count": success_count,
        "failure_count": failure_count
    }

def error_recovery_node(state: RobustMCPState) -> RobustMCPState:
    """Node that handles error recovery and retry logic"""
    error_log = state.get("error_log", [])
    failure_count = state.get("failure_count", 0)
    success_count = state.get("success_count", 0)
    
    if failure_count > 0:
        print(f"🔄 Error Recovery: {failure_count} failures detected")
        
        # Simulate retry logic for failed tools
        retry_results = []
        for error in error_log:
            if "timeout" in error.lower():
                retry_results.append(f"🔄 Retrying failed tool with increased timeout")
            elif "connection" in error.lower():
                retry_results.append(f"🔄 Retrying failed tool with different server")
            else:
                retry_results.append(f"🔄 Retrying failed tool with fallback method")
        
        # Simulate successful retries
        recovered_count = min(failure_count, 2)  # Recover up to 2 failures
        success_count += recovered_count
        failure_count -= recovered_count
        
        print(f"🔄 Recovery: {recovered_count} tools recovered")
        
        # Update error log with recovery information
        error_log.extend(retry_results)
    
    return {
        "error_log": error_log,
        "success_count": success_count,
        "failure_count": failure_count
    }

def mcp_summary_node(state: RobustMCPState) -> RobustMCPState:
    """Node that generates comprehensive MCP execution summary"""
    execution_results = state.get("execution_results", [])
    error_log = state.get("error_log", [])
    success_count = state.get("success_count", 0)
    failure_count = state.get("failure_count", 0)
    mcp_config = state.get("mcp_config", {})
    
    final_summary = f"🎯 MCP Integration Summary:\n\n"
    final_summary += f"Configuration:\n"
    final_summary += f"- Servers: {len(mcp_config.get('servers', []))}\n"
    final_summary += f"- Global timeout: {mcp_config.get('global_timeout', 0)}s\n"
    final_summary += f"- Max concurrent: {mcp_config.get('max_concurrent', 0)}\n\n"
    
    final_summary += f"Execution Results:\n"
    final_summary += f"- Successful: {success_count}\n"
    final_summary += f"- Failed: {failure_count}\n"
    final_summary += f"- Success rate: {(success_count / (success_count + failure_count) * 100):.1f}%\n\n"
    
    if execution_results:
        final_summary += f"Successful Tools:\n"
        for result in execution_results:
            final_summary += f"- {result['tool']}: {result['execution_time']:.2f}s\n"
    
    if error_log:
        final_summary += f"\nError Log:\n"
        for error in error_log[-3:]:  # Show last 3 errors
            final_summary += f"- {error}\n"
    
    final_summary += f"\n✅ MCP integration completed with robust error handling!"
    
    print(f"🎯 MCP Summary: {final_summary}")
    
    return {
        "final_summary": final_summary
    }

# Build robust MCP integration workflow
robust_mcp_workflow = StateGraph(RobustMCPState)

# Add nodes
robust_mcp_workflow.add_node("mcp_configuration", mcp_configuration_node)
robust_mcp_workflow.add_node("tool_request_processor", tool_request_processor_node)
robust_mcp_workflow.add_node("mcp_tool_executor", mcp_tool_executor_node)
robust_mcp_workflow.add_node("error_recovery", error_recovery_node)
robust_mcp_workflow.add_node("mcp_summary", mcp_summary_node)

# Add edges
robust_mcp_workflow.add_edge(START, "mcp_configuration")
robust_mcp_workflow.add_edge("mcp_configuration", "tool_request_processor")
robust_mcp_workflow.add_edge("tool_request_processor", "mcp_tool_executor")
robust_mcp_workflow.add_edge("mcp_tool_executor", "error_recovery")
robust_mcp_workflow.add_edge("error_recovery", "mcp_summary")
robust_mcp_workflow.add_edge("mcp_summary", END)

robust_mcp_app = robust_mcp_workflow.compile()

print("✅ Robust MCP integration workflow created!")
print("🛡️ This workflow demonstrates error handling and recovery mechanisms")


In [ ]:
# Test robust MCP integration workflow
print("🛡️ Testing Robust MCP Integration:")
print("="*60)

result = robust_mcp_app.invoke(
    {
        "messages": [HumanMessage(content="Execute database queries and file operations with API monitoring")],
        "mcp_config": {},
        "tool_requests": [],
        "execution_results": [],
        "error_log": [],
        "success_count": 0,
        "failure_count": 0,
        "final_summary": ""
    }
)

print(f"\nFinal Summary: {result['final_summary']}")

print("="*60)
print("✅ Robust MCP integration demonstrated!")
print("🛡️ Successfully handled errors and recovery with comprehensive reporting")


## Key Takeaways - MCP Integration

✅ **When to use:**
- Connecting to external systems and services
- Integrating with databases, APIs, and file systems
- Building interoperable systems using MCP protocol
- Accessing distributed resources and tools

💡 **Key Features:**
- **MCP Server Connection**: Connect to external MCP servers
- **Resource Discovery**: Discover available tools and resources
- **Tool Execution**: Execute MCP tools within graph nodes
- **Protocol Compliance**: Follow MCP communication protocols

⚠️ **Common Pitfalls:**
- Ensure MCP servers are properly configured and accessible
- Handle connection timeouts and network failures gracefully
- Validate tool availability before execution
- Implement proper error handling and recovery mechanisms

---


# 1️⃣3️⃣ Evaluation

<a id="evaluation"></a>

## Overview

**Evaluation** in LangGraph enables comprehensive assessment of graph performance, quality, and behavior through integration with LangSmith and custom metrics, enabling:
- **Performance Monitoring**: Track execution time, resource usage, and throughput
- **Quality Assessment**: Evaluate response quality, accuracy, and relevance
- **Debugging Support**: Trace execution paths and identify bottlenecks
- **Continuous Improvement**: Use metrics to optimize graph performance

## Key Features

- **LangSmith Integration**: Connect to LangSmith for tracing and evaluation
- **Custom Metrics**: Define and track custom performance metrics
- **Quality Metrics**: Assess response quality and accuracy
- **Performance Analysis**: Monitor execution performance and bottlenecks

---

## Example 1: Basic LangSmith Integration


In [ ]:
# Note: LangSmith integration requires API key and proper setup
# This example demonstrates the concept with simulated LangSmith functionality

import time
import json
from datetime import datetime

class EvaluationState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    execution_start_time: float
    execution_end_time: float
    node_execution_times: dict
    langsmith_trace_id: str
    evaluation_metrics: dict
    performance_summary: str

def langsmith_tracer_node(state: EvaluationState) -> EvaluationState:
    """Node that simulates LangSmith tracing"""
    execution_start_time = time.time()
    
    # Simulate LangSmith trace creation
    trace_id = f"trace_{int(time.time())}_{hash(str(state)) % 10000}"
    
    print(f"🔍 LangSmith Trace Started: {trace_id}")
    
    return {
        "execution_start_time": execution_start_time,
        "langsmith_trace_id": trace_id
    }

def performance_monitor_node(state: EvaluationState) -> EvaluationState:
    """Node that monitors performance metrics"""
    execution_start_time = state.get("execution_start_time", time.time())
    langsmith_trace_id = state.get("langsmith_trace_id", "")
    
    # Simulate node execution time tracking
    node_execution_times = {
        "langsmith_tracer": 0.05,
        "performance_monitor": 0.03,
        "quality_evaluator": 0.12,
        "metrics_aggregator": 0.08
    }
    
    # Simulate performance metrics
    performance_metrics = {
        "total_execution_time": sum(node_execution_times.values()),
        "node_count": len(node_execution_times),
        "average_node_time": sum(node_execution_times.values()) / len(node_execution_times),
        "memory_usage": "45.2 MB",
        "cpu_usage": "12.3%"
    }
    
    print(f"📊 Performance Monitor: {performance_metrics}")
    
    return {
        "node_execution_times": node_execution_times,
        "evaluation_metrics": performance_metrics
    }

def quality_evaluator_node(state: EvaluationState) -> EvaluationState:
    """Node that evaluates response quality"""
    messages = state["messages"]
    langsmith_trace_id = state.get("langsmith_trace_id", "")
    
    # Simulate quality evaluation metrics
    quality_metrics = {
        "response_relevance": 0.92,
        "response_accuracy": 0.88,
        "response_completeness": 0.95,
        "response_clarity": 0.90,
        "overall_quality_score": 0.91
    }
    
    # Simulate LangSmith quality evaluation
    print(f"🎯 Quality Evaluator: Overall score {quality_metrics['overall_quality_score']:.2f}")
    
    # Update evaluation metrics
    current_metrics = state.get("evaluation_metrics", {})
    current_metrics.update(quality_metrics)
    
    return {
        "evaluation_metrics": current_metrics
    }

def metrics_aggregator_node(state: EvaluationState) -> EvaluationState:
    """Node that aggregates all evaluation metrics"""
    execution_start_time = state.get("execution_start_time", time.time())
    execution_end_time = time.time()
    node_execution_times = state.get("node_execution_times", {})
    evaluation_metrics = state.get("evaluation_metrics", {})
    langsmith_trace_id = state.get("langsmith_trace_id", "")
    
    # Calculate total execution time
    total_execution_time = execution_end_time - execution_start_time
    
    # Generate comprehensive performance summary
    performance_summary = f"🎯 LangGraph Evaluation Summary:\n\n"
    performance_summary += f"LangSmith Trace ID: {langsmith_trace_id}\n"
    performance_summary += f"Total Execution Time: {total_execution_time:.3f}s\n\n"
    
    performance_summary += f"Node Performance:\n"
    for node, exec_time in node_execution_times.items():
        performance_summary += f"- {node}: {exec_time:.3f}s\n"
    
    performance_summary += f"\nQuality Metrics:\n"
    if "overall_quality_score" in evaluation_metrics:
        performance_summary += f"- Overall Quality: {evaluation_metrics['overall_quality_score']:.2f}\n"
        performance_summary += f"- Relevance: {evaluation_metrics.get('response_relevance', 0):.2f}\n"
        performance_summary += f"- Accuracy: {evaluation_metrics.get('response_accuracy', 0):.2f}\n"
        performance_summary += f"- Completeness: {evaluation_metrics.get('response_completeness', 0):.2f}\n"
        performance_summary += f"- Clarity: {evaluation_metrics.get('response_clarity', 0):.2f}\n"
    
    performance_summary += f"\nSystem Metrics:\n"
    performance_summary += f"- Memory Usage: {evaluation_metrics.get('memory_usage', 'N/A')}\n"
    performance_summary += f"- CPU Usage: {evaluation_metrics.get('cpu_usage', 'N/A')}\n"
    
    performance_summary += f"\n✅ Evaluation completed successfully!"
    
    print(f"📈 Metrics Aggregator: {performance_summary}")
    
    return {
        "execution_end_time": execution_end_time,
        "performance_summary": performance_summary
    }

# Build evaluation workflow
evaluation_workflow = StateGraph(EvaluationState)

# Add nodes
evaluation_workflow.add_node("langsmith_tracer", langsmith_tracer_node)
evaluation_workflow.add_node("performance_monitor", performance_monitor_node)
evaluation_workflow.add_node("quality_evaluator", quality_evaluator_node)
evaluation_workflow.add_node("metrics_aggregator", metrics_aggregator_node)

# Add edges
evaluation_workflow.add_edge(START, "langsmith_tracer")
evaluation_workflow.add_edge("langsmith_tracer", "performance_monitor")
evaluation_workflow.add_edge("performance_monitor", "quality_evaluator")
evaluation_workflow.add_edge("quality_evaluator", "metrics_aggregator")
evaluation_workflow.add_edge("metrics_aggregator", END)

evaluation_app = evaluation_workflow.compile()

print("✅ Evaluation workflow created!")
print("🔍 This workflow demonstrates LangSmith integration and performance monitoring")


In [ ]:
# Test evaluation workflow
print("🔍 Testing Evaluation Workflow:")
print("="*60)

result = evaluation_app.invoke(
    {
        "messages": [HumanMessage(content="Evaluate the performance of this LangGraph workflow")],
        "execution_start_time": 0.0,
        "execution_end_time": 0.0,
        "node_execution_times": {},
        "langsmith_trace_id": "",
        "evaluation_metrics": {},
        "performance_summary": ""
    }
)

print(f"\nFinal Summary: {result['performance_summary']}")

print("="*60)
print("✅ Evaluation workflow demonstrated!")
print("🔍 Successfully integrated LangSmith tracing and performance monitoring")


## Example 2: Custom Metrics and Quality Assessment


In [ ]:
# Create a comprehensive evaluation system with custom metrics
class CustomEvaluationState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    input_data: str
    custom_metrics: dict
    quality_scores: dict
    performance_data: dict
    evaluation_report: str
    recommendations: list[str]

def input_analyzer_node(state: CustomEvaluationState) -> CustomEvaluationState:
    """Node that analyzes input data for evaluation"""
    messages = state["messages"]
    input_data = messages[0].content if messages else ""
    
    # Analyze input characteristics
    input_analysis = {
        "length": len(input_data),
        "word_count": len(input_data.split()),
        "complexity": "high" if len(input_data.split()) > 20 else "medium" if len(input_data.split()) > 10 else "low",
        "contains_questions": "?" in input_data,
        "contains_technical_terms": any(term in input_data.lower() for term in ["api", "database", "algorithm", "function"]),
        "sentiment": "positive" if any(word in input_data.lower() for word in ["good", "great", "excellent"]) else "neutral"
    }
    
    print(f"📊 Input Analysis: {input_analysis}")
    
    return {
        "input_data": input_data,
        "custom_metrics": input_analysis
    }

def quality_assessor_node(state: CustomEvaluationState) -> CustomEvaluationState:
    """Node that assesses response quality using custom criteria"""
    input_data = state.get("input_data", "")
    custom_metrics = state.get("custom_metrics", {})
    
    # Simulate quality assessment based on input characteristics
    quality_scores = {
        "relevance": 0.85,
        "accuracy": 0.92,
        "completeness": 0.88,
        "clarity": 0.90,
        "technical_accuracy": 0.95 if custom_metrics.get("contains_technical_terms", False) else 0.80,
        "response_time": 0.95,
        "user_satisfaction": 0.87
    }
    
    # Adjust scores based on input complexity
    if custom_metrics.get("complexity") == "high":
        quality_scores["completeness"] += 0.05
        quality_scores["clarity"] -= 0.03
    
    # Ensure scores are within valid range
    for key in quality_scores:
        quality_scores[key] = max(0.0, min(1.0, quality_scores[key]))
    
    print(f"🎯 Quality Assessment: {quality_scores}")
    
    return {
        "quality_scores": quality_scores
    }

def performance_analyzer_node(state: CustomEvaluationState) -> CustomEvaluationState:
    """Node that analyzes performance metrics"""
    custom_metrics = state.get("custom_metrics", {})
    quality_scores = state.get("quality_scores", {})
    
    # Calculate performance metrics
    performance_data = {
        "throughput": 150.0,  # requests per minute
        "latency": 0.45,  # seconds
        "error_rate": 0.02,  # 2%
        "resource_utilization": {
            "cpu": 35.2,
            "memory": 128.5,
            "disk": 45.8
        },
        "scalability_score": 0.88,
        "reliability_score": 0.94
    }
    
    # Adjust performance based on input complexity
    if custom_metrics.get("complexity") == "high":
        performance_data["latency"] += 0.1
        performance_data["resource_utilization"]["cpu"] += 10.0
    
    print(f"⚡ Performance Analysis: {performance_data}")
    
    return {
        "performance_data": performance_data
    }

def evaluation_reporter_node(state: CustomEvaluationState) -> CustomEvaluationState:
    """Node that generates comprehensive evaluation report"""
    custom_metrics = state.get("custom_metrics", {})
    quality_scores = state.get("quality_scores", {})
    performance_data = state.get("performance_data", {})
    
    # Generate evaluation report
    evaluation_report = f"📋 Comprehensive Evaluation Report:\n\n"
    
    evaluation_report += f"Input Analysis:\n"
    evaluation_report += f"- Length: {custom_metrics.get('length', 0)} characters\n"
    evaluation_report += f"- Word Count: {custom_metrics.get('word_count', 0)} words\n"
    evaluation_report += f"- Complexity: {custom_metrics.get('complexity', 'unknown')}\n"
    evaluation_report += f"- Technical Content: {custom_metrics.get('contains_technical_terms', False)}\n\n"
    
    evaluation_report += f"Quality Scores:\n"
    for metric, score in quality_scores.items():
        evaluation_report += f"- {metric.replace('_', ' ').title()}: {score:.2f}\n"
    
    evaluation_report += f"\nPerformance Metrics:\n"
    evaluation_report += f"- Throughput: {performance_data.get('throughput', 0):.1f} req/min\n"
    evaluation_report += f"- Latency: {performance_data.get('latency', 0):.3f}s\n"
    evaluation_report += f"- Error Rate: {performance_data.get('error_rate', 0):.1%}\n"
    evaluation_report += f"- CPU Usage: {performance_data.get('resource_utilization', {}).get('cpu', 0):.1f}%\n"
    evaluation_report += f"- Memory Usage: {performance_data.get('resource_utilization', {}).get('memory', 0):.1f} MB\n"
    
    evaluation_report += f"\nOverall Assessment:\n"
    overall_score = sum(quality_scores.values()) / len(quality_scores)
    evaluation_report += f"- Overall Quality Score: {overall_score:.2f}\n"
    evaluation_report += f"- Performance Grade: {'A' if overall_score > 0.9 else 'B' if overall_score > 0.8 else 'C'}\n"
    
    print(f"📋 Evaluation Report: {evaluation_report}")
    
    return {
        "evaluation_report": evaluation_report
    }

def recommendation_engine_node(state: CustomEvaluationState) -> CustomEvaluationState:
    """Node that generates improvement recommendations"""
    quality_scores = state.get("quality_scores", {})
    performance_data = state.get("performance_data", {})
    custom_metrics = state.get("custom_metrics", {})
    
    recommendations = []
    
    # Generate recommendations based on scores
    if quality_scores.get("clarity", 0) < 0.85:
        recommendations.append("Improve response clarity and readability")
    
    if quality_scores.get("completeness", 0) < 0.90:
        recommendations.append("Enhance response completeness and detail")
    
    if performance_data.get("latency", 0) > 0.5:
        recommendations.append("Optimize response time and reduce latency")
    
    if performance_data.get("error_rate", 0) > 0.05:
        recommendations.append("Implement better error handling and validation")
    
    if custom_metrics.get("complexity") == "high" and performance_data.get("resource_utilization", {}).get("cpu", 0) > 50:
        recommendations.append("Consider load balancing for high-complexity requests")
    
    # Add general recommendations
    if not recommendations:
        recommendations.append("System performance is optimal - maintain current configuration")
    else:
        recommendations.append("Monitor system performance regularly")
        recommendations.append("Consider A/B testing for optimization")
    
    print(f"💡 Recommendations: {recommendations}")
    
    return {
        "recommendations": recommendations
    }

# Build custom evaluation workflow
custom_evaluation_workflow = StateGraph(CustomEvaluationState)

# Add nodes
custom_evaluation_workflow.add_node("input_analyzer", input_analyzer_node)
custom_evaluation_workflow.add_node("quality_assessor", quality_assessor_node)
custom_evaluation_workflow.add_node("performance_analyzer", performance_analyzer_node)
custom_evaluation_workflow.add_node("evaluation_reporter", evaluation_reporter_node)
custom_evaluation_workflow.add_node("recommendation_engine", recommendation_engine_node)

# Add edges
custom_evaluation_workflow.add_edge(START, "input_analyzer")
custom_evaluation_workflow.add_edge("input_analyzer", "quality_assessor")
custom_evaluation_workflow.add_edge("quality_assessor", "performance_analyzer")
custom_evaluation_workflow.add_edge("performance_analyzer", "evaluation_reporter")
custom_evaluation_workflow.add_edge("evaluation_reporter", "recommendation_engine")
custom_evaluation_workflow.add_edge("recommendation_engine", END)

custom_evaluation_app = custom_evaluation_workflow.compile()

print("✅ Custom evaluation workflow created!")
print("📊 This workflow demonstrates custom metrics and quality assessment")


In [ ]:
# Test custom evaluation workflow
print("📊 Testing Custom Evaluation Workflow:")
print("="*60)

# Test 1: Simple input
print("Test 1: Simple input")
result1 = custom_evaluation_app.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "input_data": "",
        "custom_metrics": {},
        "quality_scores": {},
        "performance_data": {},
        "evaluation_report": "",
        "recommendations": []
    }
)
print(f"Report: {result1['evaluation_report']}")
print(f"Recommendations: {result1['recommendations']}\n")

# Test 2: Complex technical input
print("Test 2: Complex technical input")
result2 = custom_evaluation_app.invoke(
    {
        "messages": [HumanMessage(content="Can you help me optimize this database query performance and implement a new API endpoint with proper error handling?")],
        "input_data": "",
        "custom_metrics": {},
        "quality_scores": {},
        "performance_data": {},
        "evaluation_report": "",
        "recommendations": []
    }
)
print(f"Report: {result2['evaluation_report']}")
print(f"Recommendations: {result2['recommendations']}\n")

print("="*60)
print("✅ Custom evaluation workflow demonstrated!")
print("📊 Successfully assessed quality and performance with custom metrics")


## Example 3: Continuous Monitoring and Alerting


In [ ]:
# Create a continuous monitoring and alerting system
class MonitoringState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    monitoring_config: dict
    current_metrics: dict
    historical_data: list[dict]
    alerts: list[str]
    system_status: str
    monitoring_report: str

def monitoring_config_node(state: MonitoringState) -> MonitoringState:
    """Node that configures monitoring parameters"""
    monitoring_config = {
        "quality_thresholds": {
            "min_relevance": 0.80,
            "min_accuracy": 0.85,
            "min_completeness": 0.80,
            "min_clarity": 0.75
        },
        "performance_thresholds": {
            "max_latency": 1.0,  # seconds
            "max_error_rate": 0.05,  # 5%
            "max_cpu_usage": 80.0,  # 80%
            "max_memory_usage": 512.0  # 512 MB
        },
        "alert_settings": {
            "enable_alerts": True,
            "alert_frequency": "immediate",
            "escalation_levels": ["warning", "critical", "emergency"]
        }
    }
    
    print(f"⚙️ Monitoring Configuration: {monitoring_config}")
    
    return {
        "monitoring_config": monitoring_config
    }

def metrics_collector_node(state: MonitoringState) -> MonitoringState:
    """Node that collects current system metrics"""
    monitoring_config = state.get("monitoring_config", {})
    
    # Simulate collecting current metrics
    current_metrics = {
        "quality": {
            "relevance": 0.92,
            "accuracy": 0.88,
            "completeness": 0.95,
            "clarity": 0.90
        },
        "performance": {
            "latency": 0.65,
            "error_rate": 0.03,
            "cpu_usage": 45.2,
            "memory_usage": 256.8,
            "throughput": 120.0
        },
        "timestamp": time.time(),
        "request_count": 1250
    }
    
    print(f"📊 Current Metrics: {current_metrics}")
    
    return {
        "current_metrics": current_metrics
    }

def threshold_checker_node(state: MonitoringState) -> MonitoringState:
    """Node that checks metrics against thresholds"""
    current_metrics = state.get("current_metrics", {})
    monitoring_config = state.get("monitoring_config", {})
    
    alerts = []
    quality_thresholds = monitoring_config.get("quality_thresholds", {})
    performance_thresholds = monitoring_config.get("performance_thresholds", {})
    
    # Check quality thresholds
    quality_metrics = current_metrics.get("quality", {})
    for metric, threshold in quality_thresholds.items():
        current_value = quality_metrics.get(metric, 0)
        if current_value < threshold:
            alerts.append(f"⚠️ Quality Alert: {metric} ({current_value:.2f}) below threshold ({threshold:.2f})")
    
    # Check performance thresholds
    performance_metrics = current_metrics.get("performance", {})
    for metric, threshold in performance_thresholds.items():
        current_value = performance_metrics.get(metric, 0)
        if current_value > threshold:
            alerts.append(f"🚨 Performance Alert: {metric} ({current_value:.2f}) above threshold ({threshold:.2f})")
    
    # Determine system status
    if len(alerts) == 0:
        system_status = "healthy"
    elif len(alerts) <= 2:
        system_status = "warning"
    else:
        system_status = "critical"
    
    print(f"🔍 Threshold Check: {len(alerts)} alerts, status: {system_status}")
    
    return {
        "alerts": alerts,
        "system_status": system_status
    }

def historical_analyzer_node(state: MonitoringState) -> MonitoringState:
    """Node that analyzes historical data trends"""
    current_metrics = state.get("current_metrics", {})
    
    # Simulate historical data analysis
    historical_data = [
        {"timestamp": time.time() - 3600, "latency": 0.45, "error_rate": 0.02, "cpu_usage": 35.0},
        {"timestamp": time.time() - 1800, "latency": 0.52, "error_rate": 0.025, "cpu_usage": 38.5},
        {"timestamp": time.time() - 900, "latency": 0.58, "error_rate": 0.028, "cpu_usage": 42.1},
        {"timestamp": time.time() - 300, "latency": 0.62, "error_rate": 0.031, "cpu_usage": 44.8},
        {"timestamp": time.time(), "latency": 0.65, "error_rate": 0.03, "cpu_usage": 45.2}
    ]
    
    # Analyze trends
    trend_analysis = {
        "latency_trend": "increasing",
        "error_rate_trend": "increasing", 
        "cpu_usage_trend": "increasing",
        "performance_degradation": True
    }
    
    # Add trend-based alerts
    alerts = state.get("alerts", [])
    if trend_analysis["performance_degradation"]:
        alerts.append("📈 Trend Alert: Performance degradation detected over last hour")
    
    print(f"📈 Historical Analysis: {trend_analysis}")
    
    return {
        "historical_data": historical_data,
        "alerts": alerts
    }

def alert_manager_node(state: MonitoringState) -> MonitoringState:
    """Node that manages alerts and notifications"""
    alerts = state.get("alerts", [])
    system_status = state.get("system_status", "unknown")
    monitoring_config = state.get("monitoring_config", {})
    
    # Simulate alert management
    alert_actions = []
    
    if system_status == "critical":
        alert_actions.append("🚨 CRITICAL: Immediate attention required")
        alert_actions.append("📞 Notifying on-call engineer")
        alert_actions.append("🔄 Initiating automatic scaling")
    elif system_status == "warning":
        alert_actions.append("⚠️ WARNING: Monitor closely")
        alert_actions.append("📧 Sending email notification")
    else:
        alert_actions.append("✅ System healthy - no action required")
    
    # Add specific alert actions
    for alert in alerts:
        if "latency" in alert.lower():
            alert_actions.append("🔧 Optimizing response time")
        elif "error_rate" in alert.lower():
            alert_actions.append("🛠️ Investigating error sources")
        elif "cpu_usage" in alert.lower():
            alert_actions.append("⚡ Scaling resources")
    
    print(f"🚨 Alert Manager: {alert_actions}")
    
    return {
        "alerts": alerts,
        "system_status": system_status
    }

def monitoring_reporter_node(state: MonitoringState) -> MonitoringState:
    """Node that generates comprehensive monitoring report"""
    current_metrics = state.get("current_metrics", {})
    alerts = state.get("alerts", [])
    system_status = state.get("system_status", "unknown")
    historical_data = state.get("historical_data", [])
    
    monitoring_report = f"📊 Continuous Monitoring Report:\n\n"
    
    monitoring_report += f"System Status: {system_status.upper()}\n"
    monitoring_report += f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    
    monitoring_report += f"Current Metrics:\n"
    quality_metrics = current_metrics.get("quality", {})
    performance_metrics = current_metrics.get("performance", {})
    
    monitoring_report += f"Quality Scores:\n"
    for metric, value in quality_metrics.items():
        monitoring_report += f"- {metric.replace('_', ' ').title()}: {value:.2f}\n"
    
    monitoring_report += f"\nPerformance Metrics:\n"
    for metric, value in performance_metrics.items():
        monitoring_report += f"- {metric.replace('_', ' ').title()}: {value:.2f}\n"
    
    if alerts:
        monitoring_report += f"\nActive Alerts ({len(alerts)}):\n"
        for alert in alerts:
            monitoring_report += f"- {alert}\n"
    else:
        monitoring_report += f"\nNo active alerts - system operating normally\n"
    
    monitoring_report += f"\nHistorical Trends:\n"
    monitoring_report += f"- Data points analyzed: {len(historical_data)}\n"
    monitoring_report += f"- Performance trend: {'Degrading' if len(alerts) > 0 else 'Stable'}\n"
    
    monitoring_report += f"\n✅ Monitoring report generated successfully!"
    
    print(f"📊 Monitoring Report: {monitoring_report}")
    
    return {
        "monitoring_report": monitoring_report
    }

# Build continuous monitoring workflow
monitoring_workflow = StateGraph(MonitoringState)

# Add nodes
monitoring_workflow.add_node("monitoring_config", monitoring_config_node)
monitoring_workflow.add_node("metrics_collector", metrics_collector_node)
monitoring_workflow.add_node("threshold_checker", threshold_checker_node)
monitoring_workflow.add_node("historical_analyzer", historical_analyzer_node)
monitoring_workflow.add_node("alert_manager", alert_manager_node)
monitoring_workflow.add_node("monitoring_reporter", monitoring_reporter_node)

# Add edges
monitoring_workflow.add_edge(START, "monitoring_config")
monitoring_workflow.add_edge("monitoring_config", "metrics_collector")
monitoring_workflow.add_edge("metrics_collector", "threshold_checker")
monitoring_workflow.add_edge("threshold_checker", "historical_analyzer")
monitoring_workflow.add_edge("historical_analyzer", "alert_manager")
monitoring_workflow.add_edge("alert_manager", "monitoring_reporter")
monitoring_workflow.add_edge("monitoring_reporter", END)

monitoring_app = monitoring_workflow.compile()

print("✅ Continuous monitoring workflow created!")
print("📊 This workflow demonstrates real-time monitoring and alerting")


In [ ]:
# Test continuous monitoring workflow
print("📊 Testing Continuous Monitoring Workflow:")
print("="*60)

result = monitoring_app.invoke(
    {
        "messages": [HumanMessage(content="Monitor system performance and generate alerts")],
        "monitoring_config": {},
        "current_metrics": {},
        "historical_data": [],
        "alerts": [],
        "system_status": "",
        "monitoring_report": ""
    }
)

print(f"\nMonitoring Report: {result['monitoring_report']}")

print("="*60)
print("✅ Continuous monitoring workflow demonstrated!")
print("📊 Successfully implemented real-time monitoring with alerting")


## Key Takeaways - Evaluation

✅ **When to use:**
- Monitoring system performance and quality
- Debugging and optimizing graph workflows
- Ensuring consistent response quality
- Implementing continuous improvement processes

💡 **Key Features:**
- **LangSmith Integration**: Connect to LangSmith for tracing and evaluation
- **Custom Metrics**: Define and track custom performance metrics
- **Quality Metrics**: Assess response quality and accuracy
- **Performance Analysis**: Monitor execution performance and bottlenecks

⚠️ **Common Pitfalls:**
- Set appropriate thresholds for alerts and monitoring
- Balance monitoring overhead with system performance
- Ensure evaluation metrics align with business objectives
- Implement proper alert escalation and response procedures

---


# 🎯 Complete Working Examples

<a id="complete-examples"></a>

This section demonstrates comprehensive LangGraph workflows that combine multiple capabilities to solve real-world problems.

---

## Example 1: Intelligent Chatbot with Memory and Tools


In [ ]:
# Example 1: Intelligent Chatbot with Memory, Tools, and Streaming
# This example combines: Memory, Tools, Streaming, Context, and Models

class IntelligentChatbotState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    conversation_history: list[dict]
    user_preferences: dict
    tool_results: list[str]
    memory_context: dict
    response_stream: list[str]
    final_response: str

# Define tools for the chatbot
@tool
def get_weather_tool(city: str) -> str:
    """Get current weather for a city"""
    # Simulate weather API call
    weather_data = {
        "New York": "Sunny, 72°F",
        "London": "Cloudy, 65°F", 
        "Tokyo": "Rainy, 68°F",
        "Paris": "Partly cloudy, 70°F"
    }
    return weather_data.get(city, f"Weather data not available for {city}")

@tool
def calculate_tool(expression: str) -> str:
    """Calculate mathematical expressions safely"""
    try:
        # Simple calculator for basic operations
        allowed_chars = set('0123456789+-*/.() ')
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        else:
            return "Error: Invalid characters in expression"
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search_knowledge_tool(query: str) -> str:
    """Search knowledge base for information"""
    # Simulate knowledge base search
    knowledge_base = {
        "langgraph": "LangGraph is a framework for building stateful, multi-actor applications with LLMs",
        "python": "Python is a high-level programming language known for its simplicity and readability",
        "ai": "Artificial Intelligence refers to the simulation of human intelligence in machines"
    }
    
    query_lower = query.lower()
    for key, value in knowledge_base.items():
        if key in query_lower:
            return f"Found: {value}"
    
    return f"No information found for: {query}"

def memory_manager_node(state: IntelligentChatbotState) -> IntelligentChatbotState:
    """Node that manages conversation memory and user context"""
    messages = state["messages"]
    user_id = state.get("user_id", "default_user")
    conversation_history = state.get("conversation_history", [])
    
    # Extract user preferences from conversation history
    user_preferences = {
        "language": "English",
        "response_style": "friendly",
        "topics_of_interest": [],
        "previous_queries": []
    }
    
    # Update preferences based on conversation history
    if conversation_history:
        recent_topics = [msg.get("topic", "") for msg in conversation_history[-5:]]
        user_preferences["topics_of_interest"] = list(set(recent_topics))
        user_preferences["previous_queries"] = [msg.get("content", "") for msg in conversation_history[-3:]]
    
    # Create memory context
    memory_context = {
        "user_id": user_id,
        "conversation_count": len(conversation_history),
        "last_interaction": conversation_history[-1].get("timestamp", "unknown") if conversation_history else "first_interaction",
        "user_preferences": user_preferences
    }
    
    print(f"🧠 Memory Manager: Context for user {user_id}")
    
    return {
        "user_preferences": user_preferences,
        "memory_context": memory_context
    }

def tool_selector_node(state: IntelligentChatbotState) -> IntelligentChatbotState:
    """Node that determines which tools to use based on user input"""
    messages = state["messages"]
    current_message = messages[-1].content if messages else ""
    
    tool_results = []
    
    # Analyze message content to determine tool usage
    if any(word in current_message.lower() for word in ["weather", "temperature", "forecast"]):
        # Extract city name (simplified)
        words = current_message.lower().split()
        city = None
        for word in words:
            if word in ["new york", "london", "tokyo", "paris"]:
                city = word.title()
                break
        
        if city:
            result = get_weather_tool(city)
            tool_results.append(f"Weather: {result}")
    
    if any(word in current_message.lower() for word in ["calculate", "compute", "math", "+", "-", "*", "/"]):
        # Extract mathematical expression
        import re
        math_pattern = r'[\d\+\-\*\/\(\)\.\s]+'
        matches = re.findall(math_pattern, current_message)
        if matches:
            expression = matches[0].strip()
            if len(expression) > 2:  # Basic validation
                result = calculate_tool(expression)
                tool_results.append(f"Calculation: {result}")
    
    if any(word in current_message.lower() for word in ["what is", "explain", "tell me about", "search"]):
        # Extract search query
        query_words = current_message.lower().split()
        if "what is" in current_message.lower():
            query = " ".join(query_words[query_words.index("what") + 2:])
        elif "tell me about" in current_message.lower():
            query = " ".join(query_words[query_words.index("about") + 1:])
        else:
            query = current_message
        
        result = search_knowledge_tool(query)
        tool_results.append(f"Knowledge: {result}")
    
    print(f"🔧 Tool Selector: {len(tool_results)} tools executed")
    
    return {
        "tool_results": tool_results
    }

def response_generator_node(state: IntelligentChatbotState) -> IntelligentChatbotState:
    """Node that generates streaming response based on context and tools"""
    messages = state["messages"]
    tool_results = state.get("tool_results", [])
    user_preferences = state.get("user_preferences", {})
    memory_context = state.get("memory_context", {})
    
    current_message = messages[-1].content if messages else ""
    
    # Generate response based on context
    response_parts = []
    
    # Greeting based on conversation history
    if memory_context.get("conversation_count", 0) == 0:
        response_parts.append("Hello! I'm your intelligent assistant. How can I help you today?")
    else:
        response_parts.append("I'm here to help! What would you like to know?")
    
    # Add tool results
    if tool_results:
        response_parts.append("Here's what I found:")
        for result in tool_results:
            response_parts.append(f"• {result}")
    
    # Add contextual response
    if "hello" in current_message.lower() or "hi" in current_message.lower():
        response_parts.append("Nice to meet you! I can help with weather, calculations, and general knowledge.")
    elif "thank" in current_message.lower():
        response_parts.append("You're welcome! Is there anything else I can help you with?")
    elif not tool_results:
        response_parts.append("I can help you with weather information, calculations, or general knowledge questions.")
    
    # Simulate streaming response
    response_stream = []
    full_response = " ".join(response_parts)
    
    # Split response into chunks for streaming
    words = full_response.split()
    chunk_size = 3
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        response_stream.append(chunk)
    
    print(f"💬 Response Generator: {len(response_stream)} chunks generated")
    
    return {
        "response_stream": response_stream,
        "final_response": full_response
    }

def response_streamer_node(state: IntelligentChatbotState) -> IntelligentChatbotState:
    """Node that simulates streaming the response"""
    response_stream = state.get("response_stream", [])
    
    print("📡 Streaming response:")
    for i, chunk in enumerate(response_stream, 1):
        print(f"  Chunk {i}: {chunk}")
        # Simulate streaming delay
        import time
        time.sleep(0.1)
    
    print("✅ Streaming complete")
    
    return {}

# Build intelligent chatbot workflow
intelligent_chatbot_workflow = StateGraph(IntelligentChatbotState)

# Add nodes
intelligent_chatbot_workflow.add_node("memory_manager", memory_manager_node)
intelligent_chatbot_workflow.add_node("tool_selector", tool_selector_node)
intelligent_chatbot_workflow.add_node("response_generator", response_generator_node)
intelligent_chatbot_workflow.add_node("response_streamer", response_streamer_node)

# Add edges
intelligent_chatbot_workflow.add_edge(START, "memory_manager")
intelligent_chatbot_workflow.add_edge("memory_manager", "tool_selector")
intelligent_chatbot_workflow.add_edge("tool_selector", "response_generator")
intelligent_chatbot_workflow.add_edge("response_generator", "response_streamer")
intelligent_chatbot_workflow.add_edge("response_streamer", END)

intelligent_chatbot_app = intelligent_chatbot_workflow.compile()

print("✅ Intelligent chatbot workflow created!")
print("🤖 Combines Memory, Tools, Streaming, Context, and Models capabilities")


In [ ]:
# Test intelligent chatbot workflow
print("🤖 Testing Intelligent Chatbot:")
print("="*60)

# Test 1: Weather query
print("Test 1: Weather query")
result1 = intelligent_chatbot_app.invoke(
    {
        "messages": [HumanMessage(content="What's the weather like in New York?")],
        "user_id": "user_123",
        "conversation_history": [],
        "user_preferences": {},
        "tool_results": [],
        "memory_context": {},
        "response_stream": [],
        "final_response": ""
    }
)
print(f"Final Response: {result1['final_response']}\n")

# Test 2: Calculation request
print("Test 2: Calculation request")
result2 = intelligent_chatbot_app.invoke(
    {
        "messages": [HumanMessage(content="Calculate 25 * 4 + 10")],
        "user_id": "user_123",
        "conversation_history": [{"content": "What's the weather like in New York?", "timestamp": "2024-01-01", "topic": "weather"}],
        "user_preferences": {},
        "tool_results": [],
        "memory_context": {},
        "response_stream": [],
        "final_response": ""
    }
)
print(f"Final Response: {result2['final_response']}\n")

# Test 3: Knowledge query
print("Test 3: Knowledge query")
result3 = intelligent_chatbot_app.invoke(
    {
        "messages": [HumanMessage(content="What is LangGraph?")],
        "user_id": "user_123",
        "conversation_history": [
            {"content": "What's the weather like in New York?", "timestamp": "2024-01-01", "topic": "weather"},
            {"content": "Calculate 25 * 4 + 10", "timestamp": "2024-01-01", "topic": "calculation"}
        ],
        "user_preferences": {},
        "tool_results": [],
        "memory_context": {},
        "response_stream": [],
        "final_response": ""
    }
)
print(f"Final Response: {result3['final_response']}\n")

print("="*60)
print("✅ Intelligent chatbot demonstrated!")
print("🤖 Successfully combined Memory, Tools, Streaming, Context, and Models")


## Example 2: RAG System with Tools and Human-in-the-Loop


In [ ]:
# Example 2: RAG System with Tools, Human-in-the-Loop, and Evaluation
# This example combines: Tools, Human-in-the-Loop, Evaluation, Context, and Models

class RAGSystemState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    query: str
    retrieved_documents: list[dict]
    generated_answer: str
    confidence_score: float
    human_feedback: str
    quality_metrics: dict
    final_answer: str
    evaluation_report: str

# Define tools for the RAG system
@tool
def document_retrieval_tool(query: str) -> str:
    """Retrieve relevant documents from knowledge base"""
    # Simulate document retrieval
    documents = [
        {
            "id": "doc1",
            "title": "Machine Learning Basics",
            "content": "Machine learning is a subset of artificial intelligence that focuses on algorithms and statistical models.",
            "relevance_score": 0.95
        },
        {
            "id": "doc2", 
            "title": "Deep Learning Fundamentals",
            "content": "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
            "relevance_score": 0.87
        },
        {
            "id": "doc3",
            "title": "Natural Language Processing",
            "content": "NLP is a field of AI that focuses on the interaction between computers and humans through natural language.",
            "relevance_score": 0.82
        }
    ]
    
    # Filter documents based on query relevance
    relevant_docs = []
    query_lower = query.lower()
    
    for doc in documents:
        if any(term in doc["content"].lower() for term in query_lower.split()):
            relevant_docs.append(doc)
    
    return f"Retrieved {len(relevant_docs)} relevant documents"

@tool
def answer_generation_tool(query: str, documents: list[dict]) -> str:
    """Generate answer based on query and retrieved documents"""
    # Simulate answer generation
    if not documents:
        return "I couldn't find relevant information to answer your question."
    
    # Simple answer generation based on document content
    answer_parts = []
    for doc in documents[:2]:  # Use top 2 documents
        answer_parts.append(doc["content"])
    
    return " ".join(answer_parts)

@tool
def confidence_scorer_tool(query: str, answer: str, documents: list[dict]) -> str:
    """Calculate confidence score for the generated answer"""
    # Simulate confidence scoring
    base_confidence = 0.7
    
    # Adjust based on document relevance
    if documents:
        avg_relevance = sum(doc.get("relevance_score", 0.5) for doc in documents) / len(documents)
        base_confidence += (avg_relevance - 0.5) * 0.3
    
    # Adjust based on answer length and completeness
    if len(answer.split()) > 20:
        base_confidence += 0.1
    
    # Ensure confidence is between 0 and 1
    confidence = max(0.0, min(1.0, base_confidence))
    
    return f"Confidence score: {confidence:.2f}"

def document_retriever_node(state: RAGSystemState) -> RAGSystemState:
    """Node that retrieves relevant documents"""
    messages = state["messages"]
    query = messages[-1].content if messages else ""
    
    # Simulate document retrieval
    retrieved_documents = [
        {
            "id": "doc1",
            "title": "Machine Learning Basics",
            "content": "Machine learning is a subset of artificial intelligence that focuses on algorithms and statistical models.",
            "relevance_score": 0.95
        },
        {
            "id": "doc2", 
            "title": "Deep Learning Fundamentals",
            "content": "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
            "relevance_score": 0.87
        }
    ]
    
    print(f"📚 Document Retriever: Retrieved {len(retrieved_documents)} documents")
    
    return {
        "query": query,
        "retrieved_documents": retrieved_documents
    }

def answer_generator_node(state: RAGSystemState) -> RAGSystemState:
    """Node that generates answer from retrieved documents"""
    query = state.get("query", "")
    retrieved_documents = state.get("retrieved_documents", [])
    
    # Generate answer based on documents
    if retrieved_documents:
        answer_parts = []
        for doc in retrieved_documents:
            answer_parts.append(doc["content"])
        generated_answer = " ".join(answer_parts)
    else:
        generated_answer = "I couldn't find relevant information to answer your question."
    
    print(f"💬 Answer Generator: Generated answer ({len(generated_answer)} characters)")
    
    return {
        "generated_answer": generated_answer
    }

def confidence_evaluator_node(state: RAGSystemState) -> RAGSystemState:
    """Node that evaluates confidence in the generated answer"""
    query = state.get("query", "")
    generated_answer = state.get("generated_answer", "")
    retrieved_documents = state.get("retrieved_documents", [])
    
    # Calculate confidence score
    base_confidence = 0.7
    
    if retrieved_documents:
        avg_relevance = sum(doc.get("relevance_score", 0.5) for doc in retrieved_documents) / len(retrieved_documents)
        base_confidence += (avg_relevance - 0.5) * 0.3
    
    if len(generated_answer.split()) > 20:
        base_confidence += 0.1
    
    confidence_score = max(0.0, min(1.0, base_confidence))
    
    print(f"📊 Confidence Evaluator: Score {confidence_score:.2f}")
    
    return {
        "confidence_score": confidence_score
    }

def human_review_node(state: RAGSystemState) -> RAGSystemState:
    """Node that simulates human review of the generated answer"""
    generated_answer = state.get("generated_answer", "")
    confidence_score = state.get("confidence_score", 0.0)
    
    # Simulate human feedback based on confidence score
    if confidence_score > 0.8:
        human_feedback = "approved"
        print("👤 Human Review: Answer approved - high confidence")
    elif confidence_score > 0.6:
        human_feedback = "needs_review"
        print("👤 Human Review: Answer needs review - medium confidence")
    else:
        human_feedback = "rejected"
        print("👤 Human Review: Answer rejected - low confidence")
    
    return {
        "human_feedback": human_feedback
    }

def quality_evaluator_node(state: RAGSystemState) -> RAGSystemState:
    """Node that evaluates the quality of the RAG system"""
    generated_answer = state.get("generated_answer", "")
    confidence_score = state.get("confidence_score", 0.0)
    human_feedback = state.get("human_feedback", "")
    retrieved_documents = state.get("retrieved_documents", [])
    
    # Calculate quality metrics
    quality_metrics = {
        "answer_length": len(generated_answer),
        "document_count": len(retrieved_documents),
        "confidence_score": confidence_score,
        "human_approval": human_feedback == "approved",
        "relevance_score": sum(doc.get("relevance_score", 0) for doc in retrieved_documents) / len(retrieved_documents) if retrieved_documents else 0,
        "overall_quality": 0.0
    }
    
    # Calculate overall quality score
    quality_metrics["overall_quality"] = (
        quality_metrics["confidence_score"] * 0.4 +
        (1.0 if quality_metrics["human_approval"] else 0.0) * 0.3 +
        quality_metrics["relevance_score"] * 0.3
    )
    
    print(f"📈 Quality Evaluator: Overall quality {quality_metrics['overall_quality']:.2f}")
    
    return {
        "quality_metrics": quality_metrics
    }

def final_answer_node(state: RAGSystemState) -> RAGSystemState:
    """Node that generates the final answer based on human feedback"""
    generated_answer = state.get("generated_answer", "")
    human_feedback = state.get("human_feedback", "")
    quality_metrics = state.get("quality_metrics", {})
    
    if human_feedback == "approved":
        final_answer = generated_answer
    elif human_feedback == "needs_review":
        final_answer = f"[REVIEW NEEDED] {generated_answer}"
    else:
        final_answer = "I apologize, but I couldn't provide a reliable answer to your question. Please try rephrasing or providing more context."
    
    # Generate evaluation report
    evaluation_report = f"📋 RAG System Evaluation Report:\n\n"
    evaluation_report += f"Query: {state.get('query', 'N/A')}\n"
    evaluation_report += f"Documents Retrieved: {quality_metrics.get('document_count', 0)}\n"
    evaluation_report += f"Confidence Score: {quality_metrics.get('confidence_score', 0):.2f}\n"
    evaluation_report += f"Human Feedback: {human_feedback}\n"
    evaluation_report += f"Overall Quality: {quality_metrics.get('overall_quality', 0):.2f}\n"
    evaluation_report += f"Final Answer Length: {len(final_answer)} characters\n\n"
    evaluation_report += "✅ RAG system evaluation completed!"
    
    print(f"🎯 Final Answer: {final_answer}")
    
    return {
        "final_answer": final_answer,
        "evaluation_report": evaluation_report
    }

# Build RAG system workflow
rag_system_workflow = StateGraph(RAGSystemState)

# Add nodes
rag_system_workflow.add_node("document_retriever", document_retriever_node)
rag_system_workflow.add_node("answer_generator", answer_generator_node)
rag_system_workflow.add_node("confidence_evaluator", confidence_evaluator_node)
rag_system_workflow.add_node("human_review", human_review_node)
rag_system_workflow.add_node("quality_evaluator", quality_evaluator_node)
rag_system_workflow.add_node("final_answer", final_answer_node)

# Add edges
rag_system_workflow.add_edge(START, "document_retriever")
rag_system_workflow.add_edge("document_retriever", "answer_generator")
rag_system_workflow.add_edge("answer_generator", "confidence_evaluator")
rag_system_workflow.add_edge("confidence_evaluator", "human_review")
rag_system_workflow.add_edge("human_review", "quality_evaluator")
rag_system_workflow.add_edge("quality_evaluator", "final_answer")
rag_system_workflow.add_edge("final_answer", END)

rag_system_app = rag_system_workflow.compile()

print("✅ RAG system workflow created!")
print("📚 Combines Tools, Human-in-the-Loop, Evaluation, Context, and Models capabilities")


In [ ]:
# Test RAG system workflow
print("📚 Testing RAG System:")
print("="*60)

# Test 1: High confidence query
print("Test 1: High confidence query")
result1 = rag_system_app.invoke(
    {
        "messages": [HumanMessage(content="What is machine learning?")],
        "query": "",
        "retrieved_documents": [],
        "generated_answer": "",
        "confidence_score": 0.0,
        "human_feedback": "",
        "quality_metrics": {},
        "final_answer": "",
        "evaluation_report": ""
    }
)
print(f"Final Answer: {result1['final_answer']}")
print(f"Evaluation Report: {result1['evaluation_report']}\n")

# Test 2: Medium confidence query
print("Test 2: Medium confidence query")
result2 = rag_system_app.invoke(
    {
        "messages": [HumanMessage(content="How does artificial intelligence work?")],
        "query": "",
        "retrieved_documents": [],
        "generated_answer": "",
        "confidence_score": 0.0,
        "human_feedback": "",
        "quality_metrics": {},
        "final_answer": "",
        "evaluation_report": ""
    }
)
print(f"Final Answer: {result2['final_answer']}")
print(f"Evaluation Report: {result2['evaluation_report']}\n")

# Test 3: Low confidence query
print("Test 3: Low confidence query")
result3 = rag_system_app.invoke(
    {
        "messages": [HumanMessage(content="What is the meaning of life?")],
        "query": "",
        "retrieved_documents": [],
        "generated_answer": "",
        "confidence_score": 0.0,
        "human_feedback": "",
        "quality_metrics": {},
        "final_answer": "",
        "evaluation_report": ""
    }
)
print(f"Final Answer: {result3['final_answer']}")
print(f"Evaluation Report: {result3['evaluation_report']}\n")

print("="*60)
print("✅ RAG system demonstrated!")
print("📚 Successfully combined Tools, Human-in-the-Loop, Evaluation, Context, and Models")


## Example 3: Multi-Agent Research Assistant with Subgraphs


In [ ]:
# Example 3: Multi-Agent Research Assistant with Subgraphs
# This example combines: Multi-Agent, Subgraphs, Tools, Memory, and Evaluation

class ResearchAssistantState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    research_topic: str
    research_plan: dict
    data_collection_results: dict
    analysis_results: dict
    report_generation_results: dict
    quality_assessment: dict
    final_report: str
    collaboration_summary: str

# Define tools for research
@tool
def web_search_tool(query: str) -> str:
    """Search the web for information"""
    # Simulate web search results
    search_results = {
        "machine learning": "Machine learning is a subset of AI that enables computers to learn without explicit programming.",
        "artificial intelligence": "AI refers to the simulation of human intelligence in machines.",
        "data science": "Data science combines statistics, programming, and domain expertise to extract insights from data."
    }
    
    query_lower = query.lower()
    for key, value in search_results.items():
        if key in query_lower:
            return f"Web search result: {value}"
    
    return f"Web search: Found general information about {query}"

@tool
def data_analysis_tool(data: str) -> str:
    """Analyze data and extract insights"""
    # Simulate data analysis
    analysis_result = f"Data Analysis Results:\n"
    analysis_result += f"- Data points analyzed: {len(data.split())}\n"
    analysis_result += f"- Key insights: 3 main patterns identified\n"
    analysis_result += f"- Statistical significance: 95% confidence\n"
    analysis_result += f"- Recommendations: 2 actionable insights"
    
    return analysis_result

@tool
def report_generator_tool(content: str, format_type: str) -> str:
    """Generate formatted report"""
    # Simulate report generation
    report = f"Research Report ({format_type.upper()}):\n\n"
    report += f"Executive Summary:\n{content[:100]}...\n\n"
    report += f"Detailed Analysis:\n{content}\n\n"
    report += f"Conclusions:\nBased on the research, key findings include...\n\n"
    report += f"Recommendations:\n1. Further research needed\n2. Implementation considerations"
    
    return report

# Subgraph 1: Data Collection Agent
class DataCollectionState(TypedDict):
    topic: str
    search_results: list[str]
    data_sources: list[str]
    collection_summary: str

def web_researcher_node(state: DataCollectionState) -> DataCollectionState:
    """Node that searches the web for information"""
    topic = state.get("topic", "")
    
    # Simulate web research
    search_results = [
        f"Academic paper: Recent advances in {topic}",
        f"Industry report: {topic} market trends",
        f"News article: Latest developments in {topic}"
    ]
    
    print(f"🔍 Web Researcher: Found {len(search_results)} sources")
    
    return {
        "search_results": search_results
    }

def data_validator_node(state: DataCollectionState) -> DataCollectionState:
    """Node that validates collected data"""
    search_results = state.get("search_results", [])
    
    # Simulate data validation
    validated_sources = []
    for result in search_results:
        if len(result) > 20:  # Basic validation
            validated_sources.append(result)
    
    collection_summary = f"Data Collection Summary:\n"
    collection_summary += f"- Total sources found: {len(search_results)}\n"
    collection_summary += f"- Validated sources: {len(validated_sources)}\n"
    collection_summary += f"- Quality score: 0.85\n"
    collection_summary += f"- Coverage: Comprehensive"
    
    print(f"✅ Data Validator: {len(validated_sources)} sources validated")
    
    return {
        "data_sources": validated_sources,
        "collection_summary": collection_summary
    }

# Build data collection subgraph
data_collection_subgraph = StateGraph(DataCollectionState)
data_collection_subgraph.add_node("web_researcher", web_researcher_node)
data_collection_subgraph.add_node("data_validator", data_validator_node)
data_collection_subgraph.add_edge(START, "web_researcher")
data_collection_subgraph.add_edge("web_researcher", "data_validator")
data_collection_subgraph.add_edge("data_validator", END)

compiled_data_collection = data_collection_subgraph.compile()

# Subgraph 2: Analysis Agent
class AnalysisState(TypedDict):
    data_sources: list[str]
    analysis_methods: list[str]
    insights: list[str]
    analysis_summary: str

def data_analyzer_node(state: AnalysisState) -> AnalysisState:
    """Node that analyzes collected data"""
    data_sources = state.get("data_sources", [])
    
    # Simulate data analysis
    analysis_methods = ["Statistical analysis", "Trend analysis", "Comparative analysis"]
    insights = [
        "Key trend identified in the data",
        "Statistical correlation found",
        "Comparative analysis reveals patterns"
    ]
    
    print(f"📊 Data Analyzer: {len(insights)} insights generated")
    
    return {
        "analysis_methods": analysis_methods,
        "insights": insights
    }

def insight_synthesizer_node(state: AnalysisState) -> AnalysisState:
    """Node that synthesizes insights"""
    insights = state.get("insights", [])
    analysis_methods = state.get("analysis_methods", [])
    
    analysis_summary = f"Analysis Summary:\n"
    analysis_summary += f"- Methods used: {', '.join(analysis_methods)}\n"
    analysis_summary += f"- Key insights: {len(insights)}\n"
    analysis_summary += f"- Confidence level: High\n"
    analysis_summary += f"- Recommendations: 3 actionable items"
    
    print(f"🧠 Insight Synthesizer: Analysis completed")
    
    return {
        "analysis_summary": analysis_summary
    }

# Build analysis subgraph
analysis_subgraph = StateGraph(AnalysisState)
analysis_subgraph.add_node("data_analyzer", data_analyzer_node)
analysis_subgraph.add_node("insight_synthesizer", insight_synthesizer_node)
analysis_subgraph.add_edge(START, "data_analyzer")
analysis_subgraph.add_edge("data_analyzer", "insight_synthesizer")
analysis_subgraph.add_edge("insight_synthesizer", END)

compiled_analysis = analysis_subgraph.compile()

# Subgraph 3: Report Generation Agent
class ReportGenerationState(TypedDict):
    insights: list[str]
    analysis_summary: str
    report_format: str
    final_report: str

def report_writer_node(state: ReportGenerationState) -> ReportGenerationState:
    """Node that writes the research report"""
    insights = state.get("insights", [])
    analysis_summary = state.get("analysis_summary", "")
    
    # Generate report content
    report_content = f"Research Report\n\n"
    report_content += f"Analysis Summary:\n{analysis_summary}\n\n"
    report_content += f"Key Insights:\n"
    for i, insight in enumerate(insights, 1):
        report_content += f"{i}. {insight}\n"
    
    print(f"📝 Report Writer: Report generated ({len(report_content)} characters)")
    
    return {
        "final_report": report_content
    }

# Build report generation subgraph
report_generation_subgraph = StateGraph(ReportGenerationState)
report_generation_subgraph.add_node("report_writer", report_writer_node)
report_generation_subgraph.add_edge(START, "report_writer")
report_generation_subgraph.add_edge("report_writer", END)

compiled_report_generation = report_generation_subgraph.compile()

# Main Research Assistant Workflow
def research_planner_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that plans the research approach"""
    messages = state["messages"]
    research_topic = messages[-1].content if messages else ""
    
    research_plan = {
        "topic": research_topic,
        "approach": "Multi-agent collaborative research",
        "agents": ["Data Collection", "Analysis", "Report Generation"],
        "timeline": "3 phases",
        "quality_checks": ["Data validation", "Analysis review", "Report quality"]
    }
    
    print(f"📋 Research Planner: Plan created for '{research_topic}'")
    
    return {
        "research_topic": research_topic,
        "research_plan": research_plan
    }

def data_collection_coordinator_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that coordinates data collection using subgraph"""
    research_topic = state.get("research_topic", "")
    
    # Invoke data collection subgraph
    data_collection_result = compiled_data_collection.invoke({
        "topic": research_topic,
        "search_results": [],
        "data_sources": [],
        "collection_summary": ""
    })
    
    print(f"🔍 Data Collection Coordinator: Subgraph completed")
    
    return {
        "data_collection_results": data_collection_result
    }

def analysis_coordinator_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that coordinates analysis using subgraph"""
    data_collection_results = state.get("data_collection_results", {})
    data_sources = data_collection_results.get("data_sources", [])
    
    # Invoke analysis subgraph
    analysis_result = compiled_analysis.invoke({
        "data_sources": data_sources,
        "analysis_methods": [],
        "insights": [],
        "analysis_summary": ""
    })
    
    print(f"📊 Analysis Coordinator: Subgraph completed")
    
    return {
        "analysis_results": analysis_result
    }

def report_coordinator_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that coordinates report generation using subgraph"""
    analysis_results = state.get("analysis_results", {})
    insights = analysis_results.get("insights", [])
    analysis_summary = analysis_results.get("analysis_summary", "")
    
    # Invoke report generation subgraph
    report_result = compiled_report_generation.invoke({
        "insights": insights,
        "analysis_summary": analysis_summary,
        "report_format": "comprehensive",
        "final_report": ""
    })
    
    print(f"📝 Report Coordinator: Subgraph completed")
    
    return {
        "report_generation_results": report_result
    }

def quality_assessor_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that assesses the quality of the research process"""
    data_collection_results = state.get("data_collection_results", {})
    analysis_results = state.get("analysis_results", {})
    report_generation_results = state.get("report_generation_results", {})
    
    # Assess quality of each phase
    quality_assessment = {
        "data_collection_quality": 0.85,
        "analysis_quality": 0.90,
        "report_quality": 0.88,
        "overall_quality": 0.88,
        "collaboration_score": 0.92
    }
    
    print(f"📈 Quality Assessor: Overall quality {quality_assessment['overall_quality']:.2f}")
    
    return {
        "quality_assessment": quality_assessment
    }

def collaboration_summarizer_node(state: ResearchAssistantState) -> ResearchAssistantState:
    """Node that summarizes the multi-agent collaboration"""
    research_plan = state.get("research_plan", {})
    quality_assessment = state.get("quality_assessment", {})
    report_generation_results = state.get("report_generation_results", {})
    
    final_report = report_generation_results.get("final_report", "")
    
    collaboration_summary = f"🤝 Multi-Agent Research Collaboration Summary:\n\n"
    collaboration_summary += f"Research Topic: {research_plan.get('topic', 'N/A')}\n"
    collaboration_summary += f"Agents Involved: {', '.join(research_plan.get('agents', []))}\n"
    collaboration_summary += f"Collaboration Score: {quality_assessment.get('collaboration_score', 0):.2f}\n"
    collaboration_summary += f"Overall Quality: {quality_assessment.get('overall_quality', 0):.2f}\n\n"
    collaboration_summary += f"Final Report:\n{final_report}\n\n"
    collaboration_summary += "✅ Multi-agent research collaboration completed successfully!"
    
    print(f"🤝 Collaboration Summarizer: Summary generated")
    
    return {
        "final_report": final_report,
        "collaboration_summary": collaboration_summary
    }

# Build main research assistant workflow
research_assistant_workflow = StateGraph(ResearchAssistantState)

# Add nodes
research_assistant_workflow.add_node("research_planner", research_planner_node)
research_assistant_workflow.add_node("data_collection_coordinator", data_collection_coordinator_node)
research_assistant_workflow.add_node("analysis_coordinator", analysis_coordinator_node)
research_assistant_workflow.add_node("report_coordinator", report_coordinator_node)
research_assistant_workflow.add_node("quality_assessor", quality_assessor_node)
research_assistant_workflow.add_node("collaboration_summarizer", collaboration_summarizer_node)

# Add edges
research_assistant_workflow.add_edge(START, "research_planner")
research_assistant_workflow.add_edge("research_planner", "data_collection_coordinator")
research_assistant_workflow.add_edge("data_collection_coordinator", "analysis_coordinator")
research_assistant_workflow.add_edge("analysis_coordinator", "report_coordinator")
research_assistant_workflow.add_edge("report_coordinator", "quality_assessor")
research_assistant_workflow.add_edge("quality_assessor", "collaboration_summarizer")
research_assistant_workflow.add_edge("collaboration_summarizer", END)

research_assistant_app = research_assistant_workflow.compile()

print("✅ Multi-agent research assistant workflow created!")
print("🤝 Combines Multi-Agent, Subgraphs, Tools, Memory, and Evaluation capabilities")


In [ ]:
# Test multi-agent research assistant workflow
print("🤝 Testing Multi-Agent Research Assistant:")
print("="*60)

# Test 1: Machine learning research
print("Test 1: Machine learning research")
result1 = research_assistant_app.invoke(
    {
        "messages": [HumanMessage(content="Research the latest trends in machine learning")],
        "research_topic": "",
        "research_plan": {},
        "data_collection_results": {},
        "analysis_results": {},
        "report_generation_results": {},
        "quality_assessment": {},
        "final_report": "",
        "collaboration_summary": ""
    }
)
print(f"Collaboration Summary: {result1['collaboration_summary']}\n")

# Test 2: AI ethics research
print("Test 2: AI ethics research")
result2 = research_assistant_app.invoke(
    {
        "messages": [HumanMessage(content="Investigate ethical considerations in artificial intelligence")],
        "research_topic": "",
        "research_plan": {},
        "data_collection_results": {},
        "analysis_results": {},
        "report_generation_results": {},
        "quality_assessment": {},
        "final_report": "",
        "collaboration_summary": ""
    }
)
print(f"Collaboration Summary: {result2['collaboration_summary']}\n")

print("="*60)
print("✅ Multi-agent research assistant demonstrated!")
print("🤝 Successfully combined Multi-Agent, Subgraphs, Tools, Memory, and Evaluation")


## Example 4: Human-in-the-Loop Content Moderator with Time Travel


In [ ]:
# Example 4: Human-in-the-Loop Content Moderator with Time Travel
# This example combines: Human-in-the-Loop, Time Travel, Evaluation, Memory, and Context

class ContentModeratorState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    content: str
    moderation_score: float
    human_decision: str
    moderation_history: list[dict]
    checkpoint_states: list[dict]
    time_travel_point: int
    final_decision: str
    moderation_report: str

def content_analyzer_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that analyzes content for moderation"""
    messages = state["messages"]
    content = messages[-1].content if messages else ""
    
    # Simulate content analysis
    moderation_score = 0.3  # Default score
    
    # Check for potentially problematic content
    problematic_keywords = ["spam", "inappropriate", "offensive", "harmful"]
    for keyword in problematic_keywords:
        if keyword in content.lower():
            moderation_score += 0.2
    
    # Check content length and complexity
    if len(content) > 1000:
        moderation_score += 0.1
    
    # Ensure score is between 0 and 1
    moderation_score = max(0.0, min(1.0, moderation_score))
    
    print(f"🔍 Content Analyzer: Score {moderation_score:.2f}")
    
    return {
        "content": content,
        "moderation_score": moderation_score
    }

def human_review_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that simulates human review"""
    moderation_score = state.get("moderation_score", 0.0)
    content = state.get("content", "")
    
    # Simulate human decision based on score
    if moderation_score > 0.7:
        human_decision = "reject"
        print("👤 Human Review: Content rejected - high risk score")
    elif moderation_score > 0.4:
        human_decision = "review"
        print("👤 Human Review: Content needs review - medium risk score")
    else:
        human_decision = "approve"
        print("👤 Human Review: Content approved - low risk score")
    
    return {
        "human_decision": human_decision
    }

def moderation_history_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that maintains moderation history"""
    content = state.get("content", "")
    moderation_score = state.get("moderation_score", 0.0)
    human_decision = state.get("human_decision", "")
    
    # Create moderation record
    moderation_record = {
        "timestamp": time.time(),
        "content": content,
        "score": moderation_score,
        "decision": human_decision,
        "reviewer": "human_moderator"
    }
    
    # Get existing history
    moderation_history = state.get("moderation_history", [])
    moderation_history.append(moderation_record)
    
    print(f"📚 Moderation History: {len(moderation_history)} records")
    
    return {
        "moderation_history": moderation_history
    }

def checkpoint_creator_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that creates checkpoints for time travel"""
    moderation_history = state.get("moderation_history", [])
    
    # Create checkpoint of current state
    checkpoint = {
        "timestamp": time.time(),
        "state_snapshot": {
            "content": state.get("content", ""),
            "moderation_score": state.get("moderation_score", 0.0),
            "human_decision": state.get("human_decision", ""),
            "history_length": len(moderation_history)
        }
    }
    
    # Get existing checkpoints
    checkpoint_states = state.get("checkpoint_states", [])
    checkpoint_states.append(checkpoint)
    
    print(f"💾 Checkpoint Creator: Checkpoint {len(checkpoint_states)} created")
    
    return {
        "checkpoint_states": checkpoint_states
    }

def time_travel_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that simulates time travel functionality"""
    checkpoint_states = state.get("checkpoint_states", [])
    human_decision = state.get("human_decision", "")
    
    # Simulate time travel based on decision
    if human_decision == "review":
        # Travel back to modify the decision
        time_travel_point = len(checkpoint_states) - 1
        print(f"⏰ Time Travel: Traveling back to checkpoint {time_travel_point}")
        
        # Simulate modifying the decision
        modified_decision = "approve"  # Human changes mind
        print(f"⏰ Time Travel: Decision modified from 'review' to '{modified_decision}'")
        
        return {
            "time_travel_point": time_travel_point,
            "human_decision": modified_decision
        }
    else:
        print("⏰ Time Travel: No time travel needed")
        return {
            "time_travel_point": -1
        }

def decision_finalizer_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that finalizes the moderation decision"""
    human_decision = state.get("human_decision", "")
    content = state.get("content", "")
    moderation_score = state.get("moderation_score", 0.0)
    
    # Finalize decision
    if human_decision == "approve":
        final_decision = "APPROVED: Content meets moderation standards"
    elif human_decision == "reject":
        final_decision = "REJECTED: Content violates moderation guidelines"
    else:
        final_decision = "PENDING: Content requires additional review"
    
    print(f"✅ Decision Finalizer: {final_decision}")
    
    return {
        "final_decision": final_decision
    }

def moderation_reporter_node(state: ContentModeratorState) -> ContentModeratorState:
    """Node that generates moderation report"""
    content = state.get("content", "")
    moderation_score = state.get("moderation_score", 0.0)
    human_decision = state.get("human_decision", "")
    final_decision = state.get("final_decision", "")
    moderation_history = state.get("moderation_history", [])
    checkpoint_states = state.get("checkpoint_states", [])
    time_travel_point = state.get("time_travel_point", -1)
    
    moderation_report = f"📋 Content Moderation Report:\n\n"
    moderation_report += f"Content: {content[:100]}...\n"
    moderation_report += f"Moderation Score: {moderation_score:.2f}\n"
    moderation_report += f"Human Decision: {human_decision}\n"
    moderation_report += f"Final Decision: {final_decision}\n"
    moderation_report += f"Time Travel Used: {'Yes' if time_travel_point >= 0 else 'No'}\n"
    moderation_report += f"Checkpoints Created: {len(checkpoint_states)}\n"
    moderation_report += f"History Records: {len(moderation_history)}\n\n"
    
    if time_travel_point >= 0:
        moderation_report += f"Time Travel Details:\n"
        moderation_report += f"- Traveled to checkpoint {time_travel_point}\n"
        moderation_report += f"- Decision modified during review\n"
        moderation_report += f"- Process completed successfully\n\n"
    
    moderation_report += "✅ Content moderation process completed!"
    
    print(f"📋 Moderation Reporter: Report generated")
    
    return {
        "moderation_report": moderation_report
    }

# Build content moderator workflow
content_moderator_workflow = StateGraph(ContentModeratorState)

# Add nodes
content_moderator_workflow.add_node("content_analyzer", content_analyzer_node)
content_moderator_workflow.add_node("human_review", human_review_node)
content_moderator_workflow.add_node("moderation_history", moderation_history_node)
content_moderator_workflow.add_node("checkpoint_creator", checkpoint_creator_node)
content_moderator_workflow.add_node("time_travel", time_travel_node)
content_moderator_workflow.add_node("decision_finalizer", decision_finalizer_node)
content_moderator_workflow.add_node("moderation_reporter", moderation_reporter_node)

# Add edges
content_moderator_workflow.add_edge(START, "content_analyzer")
content_moderator_workflow.add_edge("content_analyzer", "human_review")
content_moderator_workflow.add_edge("human_review", "moderation_history")
content_moderator_workflow.add_edge("moderation_history", "checkpoint_creator")
content_moderator_workflow.add_edge("checkpoint_creator", "time_travel")
content_moderator_workflow.add_edge("time_travel", "decision_finalizer")
content_moderator_workflow.add_edge("decision_finalizer", "moderation_reporter")
content_moderator_workflow.add_edge("moderation_reporter", END)

content_moderator_app = content_moderator_workflow.compile()

print("✅ Content moderator workflow created!")
print("👤 Combines Human-in-the-Loop, Time Travel, Evaluation, Memory, and Context capabilities")


In [ ]:
# Test content moderator workflow
print("👤 Testing Content Moderator:")
print("="*60)

# Test 1: Low risk content
print("Test 1: Low risk content")
result1 = content_moderator_app.invoke(
    {
        "messages": [HumanMessage(content="Hello, this is a friendly message about machine learning")],
        "content": "",
        "moderation_score": 0.0,
        "human_decision": "",
        "moderation_history": [],
        "checkpoint_states": [],
        "time_travel_point": -1,
        "final_decision": "",
        "moderation_report": ""
    }
)
print(f"Final Decision: {result1['final_decision']}")
print(f"Moderation Report: {result1['moderation_report']}\n")

# Test 2: Medium risk content (triggers time travel)
print("Test 2: Medium risk content (triggers time travel)")
result2 = content_moderator_app.invoke(
    {
        "messages": [HumanMessage(content="This is a long message with potentially inappropriate content that needs review")],
        "content": "",
        "moderation_score": 0.0,
        "human_decision": "",
        "moderation_history": [],
        "checkpoint_states": [],
        "time_travel_point": -1,
        "final_decision": "",
        "moderation_report": ""
    }
)
print(f"Final Decision: {result2['final_decision']}")
print(f"Moderation Report: {result2['moderation_report']}\n")

# Test 3: High risk content
print("Test 3: High risk content")
result3 = content_moderator_app.invoke(
    {
        "messages": [HumanMessage(content="This message contains offensive and harmful content that should be rejected")],
        "content": "",
        "moderation_score": 0.0,
        "human_decision": "",
        "moderation_history": [],
        "checkpoint_states": [],
        "time_travel_point": -1,
        "final_decision": "",
        "moderation_report": ""
    }
)
print(f"Final Decision: {result3['final_decision']}")
print(f"Moderation Report: {result3['moderation_report']}\n")

print("="*60)
print("✅ Content moderator demonstrated!")
print("👤 Successfully combined Human-in-the-Loop, Time Travel, Evaluation, Memory, and Context")


## Key Takeaways - Complete Working Examples

✅ **Example 1: Intelligent Chatbot**
- **Capabilities**: Memory, Tools, Streaming, Context, Models
- **Use Case**: Conversational AI with persistent memory and tool integration
- **Key Features**: User context management, tool selection, streaming responses

✅ **Example 2: RAG System**
- **Capabilities**: Tools, Human-in-the-Loop, Evaluation, Context, Models
- **Use Case**: Knowledge retrieval with human oversight and quality assessment
- **Key Features**: Document retrieval, confidence scoring, human review, quality metrics

✅ **Example 3: Multi-Agent Research Assistant**
- **Capabilities**: Multi-Agent, Subgraphs, Tools, Memory, Evaluation
- **Use Case**: Collaborative research with specialized agents and modular workflows
- **Key Features**: Agent coordination, subgraph composition, quality assessment

✅ **Example 4: Content Moderator**
- **Capabilities**: Human-in-the-Loop, Time Travel, Evaluation, Memory, Context
- **Use Case**: Content moderation with human oversight and decision tracking
- **Key Features**: Human review, checkpoint creation, time travel, moderation history

💡 **Common Patterns:**
- **State Management**: Comprehensive state schemas for complex workflows
- **Error Handling**: Robust error handling and recovery mechanisms
- **Quality Assurance**: Built-in evaluation and quality assessment
- **Human Oversight**: Human-in-the-loop for critical decisions
- **Modularity**: Subgraphs for reusable components

⚠️ **Best Practices:**
- Design clear state schemas for complex workflows
- Implement proper error handling and recovery
- Include evaluation and quality assessment
- Plan for human oversight where needed
- Use subgraphs for modularity and reusability

---


# 🚀 Best Practices & Tips

<a id="best-practices"></a>

This section provides essential best practices, common patterns, and deployment considerations for building robust LangGraph applications.

---

## 🎯 When to Use Each Capability


### Capability Selection Guide

| Capability | Best Use Cases | When to Avoid |
|------------|----------------|---------------|
| **Streaming** | Real-time applications, user-facing interfaces, long-running processes | Simple batch processing, one-time operations |
| **Persistence** | Multi-step workflows, conversation systems, stateful applications | Stateless operations, simple transformations |
| **Durable Execution** | Long-running workflows, critical business processes, error-prone operations | Simple, fast operations, development/testing |
| **Memory** | Conversational AI, user personalization, context-aware systems | Stateless APIs, one-off requests |
| **Context** | Multi-tenant systems, user-specific configurations, dynamic behavior | Static workflows, simple use cases |
| **Models** | LLM integration, AI-powered applications, natural language processing | Rule-based systems, simple logic |
| **Tools** | External system integration, function calling, API interactions | Pure data transformations, internal logic only |
| **Human-in-the-Loop** | Content moderation, approval workflows, quality control | Automated systems, high-volume processing |
| **Time Travel** | Debugging, experimentation, complex state management | Simple workflows, production systems |
| **Subgraphs** | Modular systems, reusable components, complex workflows | Simple, linear processes |
| **Multi-Agent** | Complex tasks, specialized expertise, collaborative systems | Simple, single-purpose workflows |
| **MCP Integration** | External system connectivity, protocol compliance, interoperability | Internal-only systems, simple integrations |
| **Evaluation** | Quality assurance, performance monitoring, continuous improvement | Prototypes, simple demos |

---

## 🏗️ Common Patterns


### 1. State Management Patterns

#### **Comprehensive State Schema**
```python
class ComprehensiveState(TypedDict):
    # Core data
    messages: Annotated[list[BaseMessage], operator.add]
    user_id: str
    session_id: str
    
    # Workflow state
    current_step: int
    total_steps: int
    status: str
    
    # Results and outputs
    results: list[dict]
    final_output: str
    
    # Metadata
    timestamp: float
    metadata: dict
```

#### **State Validation Pattern**
```python
def validate_state(state: ComprehensiveState) -> ComprehensiveState:
    """Validate state before processing"""
    if not state.get("user_id"):
        raise ValueError("user_id is required")
    
    if state.get("current_step", 0) < 0:
        raise ValueError("current_step must be non-negative")
    
    return state
```

### 2. Error Handling Patterns

#### **Graceful Error Recovery**
```python
def robust_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node with comprehensive error handling"""
    try:
        # Main processing logic
        result = process_data(state)
        return {"results": [result], "status": "success"}
    
    except ValueError as e:
        # Handle validation errors
        return {"status": "validation_error", "error": str(e)}
    
    except Exception as e:
        # Handle unexpected errors
        return {"status": "error", "error": str(e)}
```

#### **Retry Pattern**
```python
def retry_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node with retry logic"""
    max_retries = 3
    retry_count = state.get("retry_count", 0)
    
    try:
        result = risky_operation(state)
        return {"results": [result], "retry_count": 0}
    
    except Exception as e:
        if retry_count < max_retries:
            return {"retry_count": retry_count + 1, "error": str(e)}
        else:
            return {"status": "failed", "error": str(e)}
```

### 3. Performance Optimization Patterns

#### **Lazy Loading**
```python
def lazy_loading_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node that loads data only when needed"""
    if state.get("data_loaded", False):
        return {}  # Skip if already loaded
    
    # Load expensive data
    expensive_data = load_expensive_data(state)
    
    return {
        "data": expensive_data,
        "data_loaded": True
    }
```

#### **Caching Pattern**
```python
from functools import lru_cache

@lru_cache(maxsize=128)
def cached_operation(key: str) -> str:
    """Cached expensive operation"""
    return expensive_operation(key)

def caching_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node that uses caching"""
    key = state.get("cache_key", "")
    result = cached_operation(key)
    
    return {"cached_result": result}
```

---

## ⚠️ Anti-Patterns to Avoid


### 1. State Management Anti-Patterns

#### ❌ **Mutating State Directly**
```python
# BAD: Direct mutation
def bad_node(state: ComprehensiveState) -> ComprehensiveState:
    state["results"].append("new_result")  # Mutates original state
    return state

# GOOD: Return new state
def good_node(state: ComprehensiveState) -> ComprehensiveState:
    new_results = state.get("results", []) + ["new_result"]
    return {"results": new_results}
```

#### ❌ **Overly Complex State**
```python
# BAD: Too many fields in state
class BadState(TypedDict):
    field1: str
    field2: str
    field3: str
    field4: str
    field5: str
    field6: str
    field7: str
    field8: str
    field9: str
    field10: str
    # ... 50 more fields

# GOOD: Group related fields
class GoodState(TypedDict):
    user_data: dict
    workflow_data: dict
    results: dict
    metadata: dict
```

### 2. Node Design Anti-Patterns

#### ❌ **God Nodes (Too Much Responsibility)**
```python
# BAD: Node that does everything
def god_node(state: ComprehensiveState) -> ComprehensiveState:
    # Validate input
    validate_input(state)
    
    # Process data
    processed_data = process_data(state)
    
    # Call external API
    api_result = call_external_api(processed_data)
    
    # Transform result
    transformed_result = transform_result(api_result)
    
    # Log everything
    log_result(transformed_result)
    
    # Send notification
    send_notification(transformed_result)
    
    return {"result": transformed_result}

# GOOD: Break into smaller nodes
def validation_node(state: ComprehensiveState) -> ComprehensiveState:
    validate_input(state)
    return {}

def processing_node(state: ComprehensiveState) -> ComprehensiveState:
    processed_data = process_data(state)
    return {"processed_data": processed_data}

def api_node(state: ComprehensiveState) -> ComprehensiveState:
    api_result = call_external_api(state["processed_data"])
    return {"api_result": api_result}
```

#### ❌ **Tight Coupling**
```python
# BAD: Nodes depend on specific field names
def tightly_coupled_node(state: ComprehensiveState) -> ComprehensiveState:
    specific_field = state["very_specific_field_name"]
    return {"result": process(specific_field)}

# GOOD: Use flexible field access
def loosely_coupled_node(state: ComprehensiveState) -> ComprehensiveState:
    data = state.get("data", {})
    result = process(data)
    return {"result": result}
```

### 3. Workflow Design Anti-Patterns

#### ❌ **Circular Dependencies**
```python
# BAD: Circular workflow
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", "node_c")
workflow.add_edge("node_c", "node_a")  # Creates cycle

# GOOD: Linear or tree-like structure
workflow.add_edge("node_a", "node_b")
workflow.add_edge("node_b", "node_c")
workflow.add_edge("node_c", "node_d")
```

#### ❌ **Overly Complex Conditional Logic**
```python
# BAD: Complex conditional routing
def complex_router(state: ComprehensiveState) -> str:
    if state.get("type") == "A" and state.get("status") == "active":
        if state.get("priority") > 5:
            if state.get("user_type") == "premium":
                return "premium_handler"
            else:
                return "standard_handler"
        else:
            return "low_priority_handler"
    elif state.get("type") == "B":
        # ... more complex logic
        pass

# GOOD: Simple, clear routing
def simple_router(state: ComprehensiveState) -> str:
    if state.get("priority") == "high":
        return "high_priority_handler"
    elif state.get("priority") == "low":
        return "low_priority_handler"
    else:
        return "default_handler"
```

---

## 🚀 Deployment Considerations


### 1. Production Readiness

#### **Environment Configuration**
```python
import os
from typing import Optional

class ProductionConfig:
    """Production configuration management"""
    
    def __init__(self):
        self.api_key = os.getenv("API_KEY")
        self.database_url = os.getenv("DATABASE_URL")
        self.redis_url = os.getenv("REDIS_URL")
        self.log_level = os.getenv("LOG_LEVEL", "INFO")
        
        # Validate required config
        if not self.api_key:
            raise ValueError("API_KEY environment variable is required")
    
    def get_checkpointer(self):
        """Get appropriate checkpointer for environment"""
        if self.redis_url:
            from langgraph.checkpoint.redis import RedisSaver
            return RedisSaver.from_conn_string(self.redis_url)
        else:
            from langgraph.checkpoint.memory import MemorySaver
            return MemorySaver()
```

#### **Logging and Monitoring**
```python
import logging
import time
from functools import wraps

def setup_logging():
    """Setup production logging"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler('langgraph.log'),
            logging.StreamHandler()
        ]
    )

def monitor_performance(func):
    """Decorator to monitor node performance"""
    @wraps(func)
    def wrapper(state):
        start_time = time.time()
        try:
            result = func(state)
            execution_time = time.time() - start_time
            logging.info(f"Node {func.__name__} completed in {execution_time:.2f}s")
            return result
        except Exception as e:
            execution_time = time.time() - start_time
            logging.error(f"Node {func.__name__} failed after {execution_time:.2f}s: {e}")
            raise
    return wrapper

@monitor_performance
def monitored_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node with performance monitoring"""
    # Node logic here
    return {"result": "success"}
```

### 2. Scalability Patterns

#### **Horizontal Scaling**
```python
class ScalableWorkflow:
    """Workflow designed for horizontal scaling"""
    
    def __init__(self, config: ProductionConfig):
        self.config = config
        self.checkpointer = config.get_checkpointer()
        self.workflow = self._build_workflow()
    
    def _build_workflow(self):
        """Build workflow with scalability considerations"""
        workflow = StateGraph(ComprehensiveState)
        
        # Add nodes with proper error handling
        workflow.add_node("input_processor", self._input_processor)
        workflow.add_node("data_processor", self._data_processor)
        workflow.add_node("output_generator", self._output_generator)
        
        # Add edges
        workflow.add_edge(START, "input_processor")
        workflow.add_edge("input_processor", "data_processor")
        workflow.add_edge("data_processor", "output_generator")
        workflow.add_edge("output_generator", END)
        
        return workflow.compile(checkpointer=self.checkpointer)
    
    def process_request(self, request_data: dict) -> dict:
        """Process request with proper error handling"""
        try:
            result = self.workflow.invoke(
                request_data,
                config={"configurable": {"thread_id": request_data.get("session_id")}}
            )
            return {"status": "success", "result": result}
        except Exception as e:
            logging.error(f"Workflow failed: {e}")
            return {"status": "error", "error": str(e)}
```

#### **Resource Management**
```python
import asyncio
from contextlib import asynccontextmanager

class ResourceManager:
    """Manage resources for LangGraph workflows"""
    
    def __init__(self, max_concurrent: int = 10):
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.active_workflows = set()
    
    @asynccontextmanager
    async def workflow_context(self, workflow_id: str):
        """Context manager for workflow execution"""
        async with self.semaphore:
            self.active_workflows.add(workflow_id)
            try:
                yield
            finally:
                self.active_workflows.discard(workflow_id)
    
    def get_status(self) -> dict:
        """Get current resource status"""
        return {
            "active_workflows": len(self.active_workflows),
            "max_concurrent": self.semaphore._value,
            "available_slots": self.semaphore._value
        }
```

### 3. Security Considerations

#### **Input Validation**
```python
import re
from typing import Any

class SecurityValidator:
    """Security validation for LangGraph workflows"""
    
    @staticmethod
    def validate_input(data: Any) -> bool:
        """Validate input data for security"""
        if isinstance(data, str):
            # Check for SQL injection patterns
            sql_patterns = [r"(\b(SELECT|INSERT|UPDATE|DELETE)\b)", r"(\b(DROP|CREATE|ALTER)\b)"]
            for pattern in sql_patterns:
                if re.search(pattern, data, re.IGNORECASE):
                    return False
            
            # Check for script injection
            script_patterns = [r"<script.*?>", r"javascript:", r"vbscript:"]
            for pattern in script_patterns:
                if re.search(pattern, data, re.IGNORECASE):
                    return False
        
        return True
    
    @staticmethod
    def sanitize_output(data: Any) -> Any:
        """Sanitize output data"""
        if isinstance(data, str):
            # Remove potentially dangerous characters
            return re.sub(r'[<>"\']', '', data)
        return data

def secure_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node with security validation"""
    validator = SecurityValidator()
    
    # Validate input
    if not validator.validate_input(state.get("user_input", "")):
        raise ValueError("Invalid input detected")
    
    # Process data
    result = process_data(state)
    
    # Sanitize output
    sanitized_result = validator.sanitize_output(result)
    
    return {"result": sanitized_result}
```

#### **Access Control**
```python
class AccessController:
    """Access control for LangGraph workflows"""
    
    def __init__(self):
        self.permissions = {
            "admin": ["read", "write", "delete", "execute"],
            "user": ["read", "execute"],
            "guest": ["read"]
        }
    
    def check_permission(self, user_role: str, action: str) -> bool:
        """Check if user has permission for action"""
        user_permissions = self.permissions.get(user_role, [])
        return action in user_permissions
    
    def authorize_node(self, user_role: str, node_name: str) -> bool:
        """Authorize access to specific node"""
        # Define node permissions
        node_permissions = {
            "admin_node": "admin",
            "user_node": "user",
            "public_node": "guest"
        }
        
        required_role = node_permissions.get(node_name, "guest")
        role_hierarchy = {"admin": 3, "user": 2, "guest": 1}
        
        return role_hierarchy.get(user_role, 0) >= role_hierarchy.get(required_role, 0)

def authorized_node(state: ComprehensiveState) -> ComprehensiveState:
    """Node with access control"""
    controller = AccessController()
    user_role = state.get("user_role", "guest")
    
    if not controller.authorize_node(user_role, "authorized_node"):
        raise PermissionError("Insufficient permissions")
    
    return {"result": "authorized"}
```

---

## 📊 Performance Optimization


### 1. Memory Optimization

#### **Efficient State Management**
```python
class EfficientState(TypedDict):
    """Optimized state schema"""
    # Use specific types instead of generic dict
    user_id: str
    session_id: str
    current_step: int
    status: str
    # Group related data
    workflow_data: dict
    results: list[str]
    # Use optional fields for large data
    large_data: Optional[dict] = None

def memory_efficient_node(state: EfficientState) -> EfficientState:
    """Node optimized for memory usage"""
    # Process only necessary data
    workflow_data = state.get("workflow_data", {})
    
    # Use generators for large datasets
    def process_large_dataset(data):
        for item in data:
            yield process_item(item)
    
    # Clear unused data
    if "large_data" in state:
        processed_data = list(process_large_dataset(state["large_data"]))
        return {
            "workflow_data": {"processed": processed_data},
            "large_data": None  # Clear after processing
        }
    
    return {"workflow_data": workflow_data}
```

#### **Lazy Evaluation**
```python
class LazyEvaluator:
    """Lazy evaluation for expensive operations"""
    
    def __init__(self):
        self._cache = {}
    
    def get_or_compute(self, key: str, compute_func):
        """Get cached result or compute if not available"""
        if key not in self._cache:
            self._cache[key] = compute_func()
        return self._cache[key]

def lazy_node(state: EfficientState) -> EfficientState:
    """Node with lazy evaluation"""
    evaluator = LazyEvaluator()
    
    # Only compute expensive operation when needed
    expensive_result = evaluator.get_or_compute(
        f"expensive_{state.get('user_id')}",
        lambda: expensive_operation(state)
    )
    
    return {"result": expensive_result}
```

### 2. Execution Optimization

#### **Parallel Processing**
```python
import asyncio
from concurrent.futures import ThreadPoolExecutor

class ParallelProcessor:
    """Parallel processing for LangGraph workflows"""
    
    def __init__(self, max_workers: int = 4):
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
    
    async def process_parallel(self, tasks: list) -> list:
        """Process multiple tasks in parallel"""
        loop = asyncio.get_event_loop()
        futures = [
            loop.run_in_executor(self.executor, task)
            for task in tasks
        ]
        return await asyncio.gather(*futures)

def parallel_node(state: EfficientState) -> EfficientState:
    """Node that processes data in parallel"""
    processor = ParallelProcessor()
    
    # Define parallel tasks
    tasks = [
        lambda: process_data_chunk(state, "chunk1"),
        lambda: process_data_chunk(state, "chunk2"),
        lambda: process_data_chunk(state, "chunk3")
    ]
    
    # Process in parallel
    results = asyncio.run(processor.process_parallel(tasks))
    
    return {"results": results}
```

#### **Batch Processing**
```python
class BatchProcessor:
    """Batch processing for efficient data handling"""
    
    def __init__(self, batch_size: int = 100):
        self.batch_size = batch_size
    
    def process_batch(self, data: list) -> list:
        """Process data in batches"""
        results = []
        
        for i in range(0, len(data), self.batch_size):
            batch = data[i:i + self.batch_size]
            batch_result = self._process_single_batch(batch)
            results.extend(batch_result)
        
        return results
    
    def _process_single_batch(self, batch: list) -> list:
        """Process a single batch"""
        # Batch processing logic
        return [process_item(item) for item in batch]

def batch_node(state: EfficientState) -> EfficientState:
    """Node that processes data in batches"""
    processor = BatchProcessor(batch_size=50)
    
    data = state.get("workflow_data", {}).get("items", [])
    results = processor.process_batch(data)
    
    return {"results": results}
```

### 3. Caching Strategies

#### **Multi-Level Caching**
```python
import redis
from functools import lru_cache

class MultiLevelCache:
    """Multi-level caching system"""
    
    def __init__(self, redis_url: str = None):
        self.memory_cache = {}
        self.redis_client = redis.Redis.from_url(redis_url) if redis_url else None
    
    def get(self, key: str) -> Any:
        """Get value from cache"""
        # Level 1: Memory cache
        if key in self.memory_cache:
            return self.memory_cache[key]
        
        # Level 2: Redis cache
        if self.redis_client:
            value = self.redis_client.get(key)
            if value:
                self.memory_cache[key] = value
                return value
        
        return None
    
    def set(self, key: str, value: Any, ttl: int = 3600):
        """Set value in cache"""
        # Level 1: Memory cache
        self.memory_cache[key] = value
        
        # Level 2: Redis cache
        if self.redis_client:
            self.redis_client.setex(key, ttl, value)

def cached_node(state: EfficientState) -> EfficientState:
    """Node with multi-level caching"""
    cache = MultiLevelCache()
    
    cache_key = f"result_{state.get('user_id')}_{state.get('current_step')}"
    
    # Try to get from cache
    cached_result = cache.get(cache_key)
    if cached_result:
        return {"result": cached_result}
    
    # Compute result
    result = expensive_computation(state)
    
    # Cache result
    cache.set(cache_key, result, ttl=1800)
    
    return {"result": result}
```

---

## 🔧 Development Workflow


### 1. Testing Strategies

#### **Unit Testing**
```python
import unittest
from unittest.mock import Mock, patch

class TestLangGraphWorkflow(unittest.TestCase):
    """Test suite for LangGraph workflows"""
    
    def setUp(self):
        """Setup test environment"""
        self.workflow = build_test_workflow()
        self.test_state = {
            "user_id": "test_user",
            "session_id": "test_session",
            "current_step": 0,
            "status": "active",
            "workflow_data": {},
            "results": []
        }
    
    def test_node_execution(self):
        """Test individual node execution"""
        result = self.workflow.invoke(self.test_state)
        
        self.assertIn("result", result)
        self.assertEqual(result["status"], "completed")
    
    def test_error_handling(self):
        """Test error handling"""
        invalid_state = {"invalid": "data"}
        
        with self.assertRaises(ValueError):
            self.workflow.invoke(invalid_state)
    
    def test_state_validation(self):
        """Test state validation"""
        # Test with missing required fields
        incomplete_state = {"user_id": "test"}
        
        with self.assertRaises(ValueError):
            self.workflow.invoke(incomplete_state)
    
    @patch('external_api.call')
    def test_external_api_integration(self, mock_api):
        """Test external API integration"""
        mock_api.return_value = {"status": "success"}
        
        result = self.workflow.invoke(self.test_state)
        
        mock_api.assert_called_once()
        self.assertEqual(result["api_result"]["status"], "success")
```

#### **Integration Testing**
```python
class TestWorkflowIntegration(unittest.TestCase):
    """Integration tests for complete workflows"""
    
    def test_end_to_end_workflow(self):
        """Test complete workflow execution"""
        # Setup test data
        test_data = {
            "user_id": "integration_test_user",
            "session_id": "integration_test_session",
            "workflow_data": {"input": "test_input"},
            "results": []
        }
        
        # Execute workflow
        result = self.workflow.invoke(test_data)
        
        # Verify results
        self.assertIn("final_output", result)
        self.assertEqual(result["status"], "completed")
        self.assertGreater(len(result["results"]), 0)
    
    def test_workflow_with_checkpointing(self):
        """Test workflow with checkpointing"""
        checkpointer = MemorySaver()
        workflow_with_checkpoint = self.workflow.compile(checkpointer=checkpointer)
        
        config = {"configurable": {"thread_id": "test_thread"}}
        
        # Execute workflow
        result = workflow_with_checkpoint.invoke(test_data, config=config)
        
        # Verify checkpointing
        state = workflow_with_checkpoint.get_state(config)
        self.assertIsNotNone(state)
```

### 2. Debugging Techniques

#### **State Inspection**
```python
def debug_node(state: EfficientState) -> EfficientState:
    """Node with debugging capabilities"""
    import logging
    
    # Log state information
    logging.info(f"Node execution started")
    logging.info(f"State keys: {list(state.keys())}")
    logging.info(f"Current step: {state.get('current_step', 'N/A')}")
    
    # Validate state
    if not state.get("user_id"):
        logging.error("Missing user_id in state")
        raise ValueError("user_id is required")
    
    # Process data
    result = process_data(state)
    
    # Log result
    logging.info(f"Node execution completed")
    logging.info(f"Result keys: {list(result.keys())}")
    
    return result
```

#### **Execution Tracing**
```python
class ExecutionTracer:
    """Trace workflow execution for debugging"""
    
    def __init__(self):
        self.trace = []
    
    def trace_node(self, node_name: str):
        """Decorator to trace node execution"""
        def decorator(func):
            def wrapper(state):
                start_time = time.time()
                self.trace.append({
                    "node": node_name,
                    "start_time": start_time,
                    "state_keys": list(state.keys())
                })
                
                try:
                    result = func(state)
                    execution_time = time.time() - start_time
                    
                    self.trace[-1].update({
                        "status": "success",
                        "execution_time": execution_time,
                        "result_keys": list(result.keys())
                    })
                    
                    return result
                except Exception as e:
                    execution_time = time.time() - start_time
                    
                    self.trace[-1].update({
                        "status": "error",
                        "execution_time": execution_time,
                        "error": str(e)
                    })
                    
                    raise
            
            return wrapper
        return decorator
    
    def get_trace(self) -> list:
        """Get execution trace"""
        return self.trace
    
    def print_trace(self):
        """Print execution trace"""
        for entry in self.trace:
            print(f"Node: {entry['node']}")
            print(f"Status: {entry['status']}")
            print(f"Execution Time: {entry['execution_time']:.2f}s")
            if entry['status'] == 'error':
                print(f"Error: {entry['error']}")
            print("-" * 40)

# Usage
tracer = ExecutionTracer()

@tracer.trace_node("debug_node")
def debug_node(state: EfficientState) -> EfficientState:
    return {"result": "success"}
```

### 3. Development Environment Setup

#### **Local Development**
```python
# development.py
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

class DevelopmentConfig:
    """Development configuration"""
    
    def __init__(self):
        self.debug = True
        self.log_level = "DEBUG"
        self.api_key = os.getenv("DEV_API_KEY", "dev_key")
        self.database_url = os.getenv("DEV_DATABASE_URL", "sqlite:///dev.db")
        
    def get_checkpointer(self):
        """Get development checkpointer"""
        from langgraph.checkpoint.memory import MemorySaver
        return MemorySaver()
    
    def setup_logging(self):
        """Setup development logging"""
        import logging
        logging.basicConfig(
            level=logging.DEBUG,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
        )

# development_workflow.py
def build_development_workflow():
    """Build workflow for development"""
    config = DevelopmentConfig()
    config.setup_logging()
    
    workflow = StateGraph(EfficientState)
    
    # Add nodes with debugging
    workflow.add_node("debug_node", debug_node)
    
    # Compile with development checkpointer
    return workflow.compile(checkpointer=config.get_checkpointer())
```

#### **Testing Environment**
```python
# testing.py
class TestingConfig:
    """Testing configuration"""
    
    def __init__(self):
        self.test_mode = True
        self.mock_external_apis = True
        self.use_test_database = True
        
    def get_checkpointer(self):
        """Get testing checkpointer"""
        from langgraph.checkpoint.memory import MemorySaver
        return MemorySaver()
    
    def setup_mocks(self):
        """Setup mocks for testing"""
        from unittest.mock import patch
        
        # Mock external APIs
        self.api_patcher = patch('external_api.call')
        self.api_mock = self.api_patcher.start()
        
        # Mock database
        self.db_patcher = patch('database.connection')
        self.db_mock = self.db_patcher.start()
    
    def cleanup_mocks(self):
        """Cleanup mocks"""
        self.api_patcher.stop()
        self.db_patcher.stop()
```

---

## 📚 Additional Resources


### 1. Official Documentation

- **LangGraph Documentation**: https://langchain-ai.github.io/langgraph/
- **LangChain Documentation**: https://python.langchain.com/
- **LangSmith Documentation**: https://docs.smith.langchain.com/

### 2. Community Resources

- **LangGraph GitHub**: https://github.com/langchain-ai/langgraph
- **LangChain GitHub**: https://github.com/langchain-ai/langchain
- **Discord Community**: https://discord.gg/langchain
- **Stack Overflow**: Tag `langgraph` and `langchain`

### 3. Learning Materials

- **LangGraph Tutorials**: https://langchain-ai.github.io/langgraph/tutorials/
- **LangChain Tutorials**: https://python.langchain.com/docs/tutorials/
- **YouTube Channel**: LangChain YouTube Channel
- **Blog Posts**: LangChain Blog

### 4. Tools and Libraries

- **LangSmith**: https://smith.langchain.com/ (Evaluation and tracing)
- **LangGraph Studio**: https://studio.langchain.com/ (Visual graph editor)
- **LangChain CLI**: https://python.langchain.com/docs/langchain_cli/
- **LangGraph CLI**: https://langchain-ai.github.io/langgraph/cli/

### 5. Best Practices Guides

- **Production Deployment**: https://python.langchain.com/docs/deployment/
- **Security Guidelines**: https://python.langchain.com/docs/security/
- **Performance Optimization**: https://python.langchain.com/docs/performance/
- **Testing Strategies**: https://python.langchain.com/docs/testing/

---

## 🎯 Summary

This comprehensive guide has covered all 13 core capabilities of LangGraph:

1. **Streaming** - Real-time updates and token streaming
2. **Persistence** - State saving and recovery
3. **Durable Execution** - Long-running workflows with checkpoints
4. **Memory** - Conversation history and cross-thread memory
5. **Context** - External data and configuration passing
6. **Models** - LLM integration and multi-model workflows
7. **Tools** - Function calling and external system integration
8. **Human-in-the-Loop** - Human oversight and approval workflows
9. **Time Travel** - State inspection and execution rewinding
10. **Subgraphs** - Modular graph composition
11. **Multi-Agent** - Agent coordination and collaboration
12. **MCP Integration** - External resource connectivity
13. **Evaluation** - Performance monitoring and quality assessment

### Key Takeaways:

- **Start Simple**: Begin with basic workflows and gradually add complexity
- **Design for Scale**: Consider performance and scalability from the start
- **Test Thoroughly**: Implement comprehensive testing strategies
- **Monitor Performance**: Use evaluation and monitoring tools
- **Follow Best Practices**: Apply the patterns and avoid anti-patterns
- **Plan for Production**: Consider deployment and security requirements

### Next Steps:

1. **Practice**: Build small workflows using different capabilities
2. **Experiment**: Try combining multiple capabilities
3. **Deploy**: Start with simple deployments and scale up
4. **Monitor**: Use LangSmith and other tools for evaluation
5. **Contribute**: Share your experiences with the community

Remember: LangGraph is a powerful framework, but with great power comes great responsibility. Always consider the implications of your design decisions and prioritize reliability, security, and maintainability.

---

**Happy building with LangGraph! 🚀**
